# 07 — Computational Causal Validation of PIE-SAE Features

**Status:** Experimental analyses A–F completed. Static editorial and methodological audit performed; independent execution against the full local PIE-SAE artifact collection has not been performed as part of this review.

**Research objective:** Investigate whether controlled
interventions on selected SAE feature components alter
the predictions of the pretrained Perturbation Is
Everything (PIE) model.

This notebook extends the statistical analyses in
`06a_statistical_significance.ipynb` and
`06b_full_statistical_validation.ipynb`.

## Research questions

1. Do targeted SAE feature interventions change
   downstream PIE predictions?
2. Are the changes distinguishable from matched
   control interventions?
3. Are the effects reproducible across perturbation
   conditions and cell contexts?
4. Do geometrically matched cross-layer SAE features
   show consistent intervention effects?

## Initial candidate features

| PIE layer | SAE feature | Role |
|---|---:|---|
| decoder.0 | 189 | Primary candidate |
| decoder.0 | 0 | Secondary candidate |
| decoder.0 | 2120 | Contrasting unstable feature |

Candidate selection is exploratory and was informed
by previous statistical results. The 32 analyzed validation
conditions are a previously selected subset of the 290-condition
validation dataset; they are not independent biological replicates.
The intervention findings must not be interpreted as confirmatory
post-selection tests.

## Methodological boundaries

SAE features are learned directions in the hidden
representation space. Their activation and decoder
directions do not necessarily correspond to independent
causal variables.

An intervention on a reconstructed feature component
may affect other correlated representation components.

Therefore, causal effects must be interpreted with
respect to the specified intervention operator,
model architecture, and downstream output metric.

Computational causal effects are distinct from
biological causal validation.

## Safety and reproducibility

The original PIE checkpoint, SAE checkpoints,
activation datasets, and previous results are immutable.

Initial stages are read-only. No model training,
checkpoint replacement, or destructive file operations
are performed.

All subsequent interventions are applied in memory
and evaluated against an unmodified baseline. The notebook
contains no explicit code to overwrite research checkpoints
or activation HDF5 files. This does not independently verify
possible cache activity in imported libraries or disk integrity
via before-and-after hashes.

**Provenance:** Pretrained PIE loading and activation extraction
follow the earlier PIE notebooks. The trained SAE checkpoints,
feature catalogs, and dictionary matches are reused from the
existing PIE-SAE pipeline. Bootstrap procedures, matched-direction
controls, patching experiments, and cross-layer intervention tests
in this notebook are additional project-specific analyses; they
should not be attributed as pre-existing implementations from
upstream bio-sae.

## A.1. Environment and Artifact Inventory

This stage verifies the local environment and the
existing model artifacts required for computational
causal validation.

The audit checks:

- The pretrained PIE checkpoint.
- Independent SAE training manifests.
- Training and validation activation datasets.
- Existing analysis notebooks and source modules.
- PyTorch and CUDA availability.

No models are loaded or executed in this stage.
All operations are read-only.

In [1]:
from pathlib import Path
import sys
import platform
import json

import numpy as np
import torch

print("=== A.1. ENVIRONMENT AND ARTIFACT INVENTORY ===")

# --------------------------------------------------
# 1. Locate the project root
# --------------------------------------------------

ROOT = None

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (
        (candidate / "notebooks").is_dir()
        and (candidate / "src").is_dir()
    ):
        ROOT = candidate
        break

if ROOT is None:
    raise FileNotFoundError(
        "PIE-SAE project root was not found."
    )

print("\n=== ENVIRONMENT ===")

print("Project root:", ROOT)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

# --------------------------------------------------
# 2. Verify required research artifacts
# --------------------------------------------------

artifacts = {
    "PIE checkpoint":
        ROOT / "models/replogle_wdataset/k562/best_auprc.ckpt",

    "SAE training manifest":
        ROOT / "results/sae_training/independent_sae_manifest.json",

    "Extraction manifest":
        ROOT / "results/extraction_validation/extraction_manifest.json",

    "Training HDF5":
        ROOT / "data/activations/expanded_128_32/train_128.h5",

    "Validation HDF5":
        ROOT / "data/activations/expanded_128_32/val_32.h5",

    "Notebook 01":
        ROOT / "notebooks/01_pie_architecture.ipynb",

    "Notebook 03":
        ROOT / "notebooks/03_train_layerwise_sae.ipynb",

    "Notebook 05":
        ROOT / "notebooks/05_cross_layer_analysis.ipynb",

    "Notebook 06b":
        ROOT / "notebooks/06b_full_statistical_validation.ipynb",
}

print("\n=== RESEARCH ARTIFACTS ===")

for name, path in artifacts.items():

    exists = path.is_file()

    print(
        f"{name:25s}: "
        f"{'FOUND' if exists else 'MISSING'}"
    )

# --------------------------------------------------
# 3. Inspect manifest structure without loading models
# --------------------------------------------------

print("\n=== SAE MANIFEST ===")

manifest_path = artifacts["SAE training manifest"]

if manifest_path.is_file():

    sae_manifest = json.loads(
        manifest_path.read_text(
            encoding="utf-8"
        )
    )

    print("Root type:", type(sae_manifest).__name__)

    if isinstance(sae_manifest, dict):
        print("Top-level keys:", list(sae_manifest.keys()))

print("\n=== EXTRACTION MANIFEST ===")

extraction_path = artifacts["Extraction manifest"]

if extraction_path.is_file():

    extraction_manifest = json.loads(
        extraction_path.read_text(
            encoding="utf-8"
        )
    )

    if isinstance(extraction_manifest, dict):
        print(
            "Top-level keys:",
            list(extraction_manifest.keys())
        )

# --------------------------------------------------
# 4. List research source modules
# --------------------------------------------------

print("\n=== SOURCE MODULES ===")

source_files = sorted(
    (ROOT / "src").rglob("*.py")
)

print("Python source files:", len(source_files))

for path in source_files[:25]:
    print(" ", path.relative_to(ROOT))

print("\nRead-only environment audit completed.")

=== A.1. ENVIRONMENT AND ARTIFACT INVENTORY ===

=== ENVIRONMENT ===
Project root: d:\bio_paper\pie-sae
Python: 3.12.13
Platform: Windows-11-10.0.26300-SP0
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA GeForce GTX 1650

=== RESEARCH ARTIFACTS ===
PIE checkpoint           : FOUND
SAE training manifest    : FOUND
Extraction manifest      : FOUND
Training HDF5            : FOUND
Validation HDF5          : FOUND
Notebook 01              : FOUND
Notebook 03              : FOUND
Notebook 05              : FOUND
Notebook 06b             : FOUND

=== SAE MANIFEST ===
Root type: dict
Top-level keys: ['series', 'bio_sae_commit', 'training_module', 'training_config', 'input_extraction_manifest', 'checkpoints']

=== EXTRACTION MANIFEST ===
Top-level keys: ['experiment', 'created_at_utc', 'model', 'extraction', 'gene_reference', 'validation']

=== SOURCE MODULES ===
Python source files: 6
  src\prepare_string_reference.py
  src\sae_activation_stats.py
  src\sae_annotation_null.py
  src\sae

## A.2. PIE Forward Path and Intervention Points

Computational causal validation requires modifying an
internal PIE representation during model execution.

The previous notebooks provide two relevant sources:

- Notebook 01: pretrained PIE architecture, loading,
  and forward-pass inspection.
- Notebook 02: activation extraction and forward hooks.

This stage recovers the exact model-loading procedure,
forward API, and activation capture mechanism.

The intervention point must satisfy three requirements:

1. It corresponds to the representation used to train
   the selected SAE.
2. Its modification can affect downstream PIE computation.
3. An unmodified forward pass can be reproduced before
   introducing any intervention.

The analysis is limited to source-code inspection.
No checkpoint is loaded, no model is executed, and
no research artifacts are modified.

In [2]:
import json
import re
from pathlib import Path

print("=== A.2. PIE FORWARD AND HOOK PROVENANCE ===")

notebooks_a2 = {
    "Architecture (01)":
        ROOT / "notebooks/01_pie_architecture.ipynb",
    "Activation extraction (02)":
        ROOT / "notebooks/02_extract_activations.ipynb",
}

patterns_a2 = {
    "model_loading": re.compile(
        r"load_from_checkpoint|torch\.load|"
        r"load_state_dict|PieModel|PIEModel",
        re.IGNORECASE,
    ),
    "forward": re.compile(
        r"\.forward\(|\bforward\(|"
        r"model\(|predict_step|"
        r"validation_step",
        re.IGNORECASE,
    ),
    "hooks": re.compile(
        r"register_forward_hook|"
        r"register_forward_pre_hook|"
        r"named_modules|hook_fn|hook_handle|"
        r"capture_activations",
        re.IGNORECASE,
    ),
    "decoder": re.compile(
        r"decoder_blocks|decoder_layers|"
        r"decoder\.0|model\.decoder|"
        r"decoder\[0\]|hook_paths",
        re.IGNORECASE,
    ),
}

for notebook_label, notebook_path in notebooks_a2.items():

    print(f"\n=== {notebook_label.upper()} ===")
    print("Exists:", notebook_path.is_file())

    if not notebook_path.is_file():
        continue

    notebook = json.loads(
        notebook_path.read_text(encoding="utf-8")
    )

    candidates = []

    for cell_index, cell in enumerate(notebook["cells"]):

        if cell["cell_type"] != "code":
            continue

        lines = "".join(
            cell.get("source", [])
        ).splitlines()

        hits = []

        for i, line in enumerate(lines):
            categories = [
                name
                for name, pattern in patterns_a2.items()
                if pattern.search(line)
            ]

            if categories:
                hits.append((i, categories))

        if hits:
            candidates.append(
                (cell_index, lines, hits)
            )

    print("Relevant code cells:", len(candidates))

    # Prioritize cells containing hooks, loading,
    # or direct forward calls.
    candidates.sort(
        key=lambda item: (
            -sum(
                3 if "hooks" in categories else
                2 if "model_loading" in categories else 1
                for _, categories in item[2]
            ),
            item[0],
        )
    )

    for cell_index, lines, hits in candidates[:4]:

        print(f"\nCode cell index: {cell_index}")
        print("Source lines:", len(lines))

        selected = set()

        for line_index, _ in hits[:3]:
            selected.update(
                range(
                    max(0, line_index - 3),
                    min(len(lines), line_index + 5),
                )
            )

        for i in sorted(selected)[:24]:
            print(f"{i + 1:3d}: {lines[i][:160]}")

# --------------------------------------------------
# Inspect checkpoint and manifest references
# without loading model weights.
# --------------------------------------------------

print("\n=== TRAINED SAE CHECKPOINT PROVENANCE ===")

manifest_path_a2 = (
    ROOT
    / "results/sae_training/independent_sae_manifest.json"
)

manifest_a2 = json.loads(
    manifest_path_a2.read_text(encoding="utf-8")
)

checkpoints_a2 = manifest_a2.get("checkpoints")

print(
    "Checkpoint registry type:",
    type(checkpoints_a2).__name__
)

if isinstance(checkpoints_a2, dict):
    print(
        "First checkpoint keys:",
        list(checkpoints_a2.keys())[:8]
    )
elif isinstance(checkpoints_a2, list):
    print("Registered checkpoints:", len(checkpoints_a2))
    if checkpoints_a2:
        print(
            "First record keys:",
            list(checkpoints_a2[0].keys())
        )

print("\nRead-only forward-path audit completed.")

=== A.2. PIE FORWARD AND HOOK PROVENANCE ===

=== ARCHITECTURE (01) ===
Exists: True
Relevant code cells: 6

Code cell index: 5
Source lines: 21
  7: if str(PIE_SRC) not in sys.path:
  8:     sys.path.insert(0, str(PIE_SRC))
  9: 
 10: from pie.model.pie import PieModel
 11: 
 12: print("PIE model class:", PieModel)
 13: print("Source file:", inspect.getfile(PieModel))
 14: 
 15: print("\nConstructor:")
 16: print(inspect.signature(PieModel.__init__))
 17: 

Code cell index: 14
Source lines: 51
  1: # Reconstruct the pretrained PIE model
  2: 
  3: model = PieModel(
  4:     cfg=loaded.config.model,
  5:     stats=loaded.stats,
  6:     source_dims=source_dims,
  7:     gene_query_text=gene_query_text,
  8: )
  9: 
 10: # Restore pretrained parameters
 11: load_result = model.load_state_dict(
 12:     loaded.state_dict,
 13:     strict=True,
 14: )
 15: 
 28: print("\nModel architecture:")
 29: print("Encoder blocks:", len(model.encoder))
 30: print("Processor blocks:", len(model.proce

### A.2.1. Model Loading and Intervention Hook Verification

The previous audit located the pretrained PIE
reconstruction and activation extraction procedures.

This inspection identifies the exact checkpoint-loading
operations, decoder activation hooks, and forward calls.

The goal is to reproduce the original activation capture
path before introducing in-memory modifications.

No model is loaded or executed during this inspection.

In [3]:
import json
import re

print("=== A.2.1. MODEL LOADING AND HOOK VERIFICATION ===")

notebook_01 = json.loads(
    (ROOT / "notebooks/01_pie_architecture.ipynb")
    .read_text(encoding="utf-8")
)

notebook_02 = json.loads(
    (ROOT / "notebooks/02_extract_activations.ipynb")
    .read_text(encoding="utf-8")
)

def show_matching_lines(notebook, patterns, limit=18):
    matches = []

    for cell_idx, cell in enumerate(notebook["cells"]):
        if cell["cell_type"] != "code":
            continue

        lines = "".join(cell.get("source", [])).splitlines()

        for line_idx, line in enumerate(lines):
            if any(re.search(p, line, re.I) for p in patterns):
                matches.append(
                    (cell_idx, line_idx + 1, line.strip())
                )

    print("Total matching lines:", len(matches))

    for cell_idx, lineno, line in matches[:limit]:
        print(
            f"Cell {cell_idx}, line {lineno}: "
            f"{line[:180]}"
        )

print("\n=== PIE MODEL LOADING (NOTEBOOK 01) ===")

show_matching_lines(
    notebook_01,
    [
        r"PieModel\(",
        r"torch\.load",
        r"load_state_dict",
        r"load_from_checkpoint",
        r"\.eval\(",
        r"\.to\(",
    ],
    limit=16,
)

print("\n=== ACTIVATION HOOKS (NOTEBOOK 02) ===")

show_matching_lines(
    notebook_02,
    [
        r"register_forward_hook",
        r"register_forward_pre_hook",
        r"named_modules",
        r"hook_paths",
        r"hook_handles",
        r"forward_hook",
    ],
    limit=20,
)

print("\n=== PIE FORWARD CALLS (NOTEBOOK 02) ===")

show_matching_lines(
    notebook_02,
    [
        r"model\(",
        r"\.forward\(",
        r"predict_step",
        r"validation_step",
        r"forward_batch",
    ],
    limit=15,
)

print("\n=== EXTRACTION MODULE IMPORTS ===")

show_matching_lines(
    notebook_02,
    [
        r"^from .* import",
        r"^import .*activation",
        r"^import .*pie",
    ],
    limit=18,
)

print("\nRead-only intervention-path inspection completed.")

=== A.2.1. MODEL LOADING AND HOOK VERIFICATION ===

=== PIE MODEL LOADING (NOTEBOOK 01) ===
Total matching lines: 5
Cell 14, line 3: model = PieModel(
Cell 14, line 11: load_result = model.load_state_dict(
Cell 14, line 18: model.eval()
Cell 20, line 9: model = model.to(device)
Cell 20, line 10: model.eval()

=== ACTIVATION HOOKS (NOTEBOOK 02) ===
Total matching lines: 14
Cell 6, line 63: handle = block.register_forward_hook(
Cell 11, line 68: hook_handles = []
Cell 12, line 116: for handle in hook_handles:
Cell 12, line 119: hook_handles.clear()
Cell 12, line 155: handle = block.register_forward_hook(
Cell 12, line 159: hook_handles.append(handle)
Cell 12, line 176: for handle in hook_handles:
Cell 12, line 179: hook_handles.clear()
Cell 15, line 80: for handle in hook_handles:
Cell 15, line 83: hook_handles.clear()
Cell 15, line 132: handle = block.register_forward_hook(
Cell 15, line 136: hook_handles.append(handle)
Cell 15, line 153: for handle in hook_handles:
Cell 15, line 156: h

## A.3. Baseline Forward Reproduction

The pretrained PIE model and its existing activation
hooks were identified in the previous notebooks.

Before performing any computational intervention,
the exact checkpoint-loading and forward-call
procedures are recovered from the previously
executed implementation.

The first baseline run will use an unmodified
model, a fixed validation condition, and a
deterministic evaluation procedure.

Its outputs and decoder.0 activations will serve
as reference values for subsequent intervention
and identity-control experiments.

No intervention is performed in this stage.

In [4]:
import json
import re

print("=== A.3. BASELINE IMPLEMENTATION RECOVERY ===")

nb01 = json.loads(
    (ROOT / "notebooks/01_pie_architecture.ipynb")
    .read_text(encoding="utf-8")
)

nb02 = json.loads(
    (ROOT / "notebooks/02_extract_activations.ipynb")
    .read_text(encoding="utf-8")
)

# 1. Previously working checkpoint reconstruction.

print("\n=== MODEL RECONSTRUCTION: NOTEBOOK 01 ===")

loading_lines = "".join(
    nb01["cells"][14]["source"]
).splitlines()

for i, line in enumerate(loading_lines[:55], start=1):
    print(f"{i:3d}: {line}")

# 2. Find actual model execution and batch preparation.

print("\n=== FORWARD AND BATCH CALLS: NOTEBOOK 02 ===")

pattern = re.compile(
    r"\bmodel\s*\(|"
    r"\bmodel\.forward\s*\(|"
    r"\bforward_batch\s*\(|"
    r"\bmodel\.predict_step\s*\(|"
    r"\bnext\s*\(.*loader|"
    r"\bval_dataloader\s*\(",
    re.IGNORECASE
)

shown = 0

for cell_idx, cell in enumerate(nb02["cells"]):

    if cell["cell_type"] != "code":
        continue

    lines = "".join(cell["source"]).splitlines()

    for i, line in enumerate(lines):

        if not pattern.search(line):
            continue

        print(f"\nCell {cell_idx}, line {i + 1}:")

        for j in range(
            max(0, i - 3),
            min(len(lines), i + 5)
        ):
            print(f"{j + 1:3d}: {lines[j]}")

        shown += 1

        if shown >= 6:
            break

    if shown >= 6:
        break

print("\nForward-call excerpts found:", shown)

# 3. Identify decoder.0 hook targets.

print("\n=== DECODER HOOK DEFINITION ===")

for cell_idx, cell in enumerate(nb02["cells"]):

    if cell["cell_type"] != "code":
        continue

    lines = "".join(cell["source"]).splitlines()

    for i, line in enumerate(lines):

        if "register_forward_hook" not in line:
            continue

        print(f"\nCell {cell_idx}, line {i + 1}:")

        for j in range(
            max(0, i - 5),
            min(len(lines), i + 4)
        ):
            print(f"{j + 1:3d}: {lines[j]}")

print("\nRead-only baseline preparation completed.")

=== A.3. BASELINE IMPLEMENTATION RECOVERY ===

=== MODEL RECONSTRUCTION: NOTEBOOK 01 ===
  1: # Reconstruct the pretrained PIE model
  2: 
  3: model = PieModel(
  4:     cfg=loaded.config.model,
  5:     stats=loaded.stats,
  6:     source_dims=source_dims,
  7:     gene_query_text=gene_query_text,
  8: )
  9: 
 10: # Restore pretrained parameters
 11: load_result = model.load_state_dict(
 12:     loaded.state_dict,
 13:     strict=True,
 14: )
 15: 
 16: # Freeze the pretrained model and enable evaluation mode
 17: model.requires_grad_(False)
 18: model.eval()
 19: 
 20: print("Pretrained PIE model reconstructed successfully!")
 21: 
 22: # Verify checkpoint compatibility
 23: print("\nCheckpoint verification:")
 24: print("Missing keys:", load_result.missing_keys)
 25: print("Unexpected keys:", load_result.unexpected_keys)
 26: 
 27: # Inspect architecture
 28: print("\nModel architecture:")
 29: print("Encoder blocks:", len(model.encoder))
 30: print("Processor blocks:", len(model.

### A.3.1. Baseline Model Inputs and Forward API

The pretrained PIE reconstruction procedure has been
located in notebook 01.

The remaining preparation step identifies the exact
input construction and model invocation used in the
existing activation-extraction pipeline.

The baseline must use the original PIE model interface
and a valid validation condition.

No model inference or intervention is performed here.

In [5]:
import json
import re

print("=== A.3.1. BASELINE INPUT AND FORWARD API ===")

nb01 = json.loads(
    (ROOT / "notebooks/01_pie_architecture.ipynb")
    .read_text(encoding="utf-8")
)

nb02 = json.loads(
    (ROOT / "notebooks/02_extract_activations.ipynb")
    .read_text(encoding="utf-8")
)

# 1. Find how PIE reconstruction dependencies were created.

print("\n=== MODEL SETUP DEPENDENCIES ===")

setup_pattern = re.compile(
    r"^\s*(loaded|source_dims|gene_query_text|"
    r"load_state_dict)\s*="
)

count = 0

for cell_idx, cell in enumerate(nb01["cells"]):
    if cell["cell_type"] != "code":
        continue

    lines = "".join(cell["source"]).splitlines()

    for i, line in enumerate(lines):
        if setup_pattern.search(line):
            print(
                f"Cell {cell_idx}, line {i + 1}: "
                f"{line.strip()}"
            )
            count += 1

print("Dependency assignments found:", count)

# 2. Find actual model execution and input preparation.

print("\n=== FORWARD EXECUTION CANDIDATES ===")

forward_pattern = re.compile(
    r"\bmodel\s*\(|"
    r"\bmodel\.forward\s*\(|"
    r"\bforward_batch\s*\(|"
    r"val_dataloader|"
    r"validation_step|"
    r"predict_step|"
    r"PieDataModule|"
    r"extract.*activation",
    re.IGNORECASE
)

matches = []

for label, notebook in [
    ("01", nb01),
    ("02", nb02),
]:
    for cell_idx, cell in enumerate(notebook["cells"]):
        if cell["cell_type"] != "code":
            continue

        lines = "".join(cell["source"]).splitlines()

        for i, line in enumerate(lines):
            if forward_pattern.search(line):
                matches.append(
                    (label, cell_idx, i + 1, line.strip())
                )

print("Matching lines:", len(matches))

for label, cell_idx, line_no, text in matches[:25]:
    print(
        f"Notebook {label}, cell {cell_idx}, "
        f"line {line_no}: {text[:170]}"
    )

# 3. Inspect local extraction implementation names.

print("\n=== EXTRACTION SOURCE FILES ===")

for path in sorted((ROOT / "src").glob("*.py")):
    name = path.name.lower()

    if "extract" in name or "activation" in name:
        print(path.relative_to(ROOT))

print("\nRead-only baseline input audit completed.")

=== A.3.1. BASELINE INPUT AND FORWARD API ===

=== MODEL SETUP DEPENDENCIES ===
Cell 7, line 8: loaded = load_checkpoint(checkpoint_path)
Cell 12, line 5: source_dims = dict(dm.source_dims)
Cell 12, line 6: gene_query_text = dm.gene_query_text()
Cell 14, line 6: source_dims=source_dims,
Cell 14, line 7: gene_query_text=gene_query_text,
Dependency assignments found: 5

=== FORWARD EXECUTION CANDIDATES ===
Matching lines: 12
Notebook 01, cell 10, line 1: from pie.data.datamodule import PieDataModule
Notebook 01, cell 10, line 5: dm = PieDataModule(
Notebook 01, cell 20, line 30: outputs = model(batch_ready)
Notebook 01, cell 22, line 58: hooked_outputs = model(batch_ready)
Notebook 02, cell 4, line 2: from pie.data.datamodule import PieDataModule
Notebook 02, cell 4, line 19: dm = PieDataModule(
Notebook 02, cell 6, line 76: outputs = model(batch_ready)
Notebook 02, cell 6, line 84: # Verify extracted activations
Notebook 02, cell 11, line 25: EXTRACTION_DIR = DATA_ROOT / "activations" /

### A.3.2. Exact Baseline Setup Recovery

The existing notebooks confirm that PIE is executed
through the original PieDataModule batch interface.

This inspection recovers the previously working
checkpoint loading, data-module initialization,
model reconstruction, and batch preparation steps.

The extracted implementation will be used to create
a deterministic, unmodified validation baseline.

No model inference or intervention occurs here.

In [6]:
import json
import re

print("=== A.3.2. EXACT BASELINE SETUP ===")

path = ROOT / "notebooks/01_pie_architecture.ipynb"

notebook = json.loads(
    path.read_text(encoding="utf-8")
)

def show_cell_excerpt(cell_index, patterns, context=4):
    cell = notebook["cells"][cell_index]

    if cell["cell_type"] != "code":
        return

    lines = "".join(cell["source"]).splitlines()

    selected = set()

    for i, line in enumerate(lines):
        if any(
            re.search(pattern, line, re.IGNORECASE)
            for pattern in patterns
        ):
            selected.update(
                range(
                    max(0, i - context),
                    min(len(lines), i + context + 1)
                )
            )

    print(f"\n=== NOTEBOOK 01, CELL {cell_index} ===")

    previous = -2

    for i in sorted(selected):
        if i > previous + 1:
            print("    ...")

        print(f"{i + 1:3d}: {lines[i]}")
        previous = i


print("\n=== CHECKPOINT LOADING ===")

show_cell_excerpt(
    7,
    [r"load_checkpoint", r"checkpoint_path"],
    context=5
)

print("\n=== DATA MODULE INITIALIZATION ===")

show_cell_excerpt(
    11,
    [r"PieDataModule", r"\bdm\s*=", r"\.setup", r"setup_stats"],
    context=5
)

print("\n=== DATA MODULE METADATA ===")

show_cell_excerpt(
    12,
    [r"source_dims", r"gene_query_text"],
    context=3
)

print("\n=== MODEL CONSTRUCTION ===")

show_cell_excerpt(
    14,
    [r"PieModel", r"load_state_dict", r"model.eval"],
    context=5
)

print("\n=== VALIDATION BATCH AND FORWARD ===")

# Find where the original batch was constructed.
batch_pattern = re.compile(
    r"batch_ready|val_dataloader|"
    r"validation_dataset|"
    r"dataset\.rows|collate",
    re.IGNORECASE
)

matching_cells = []

for cell_index, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    if batch_pattern.search(source):
        matching_cells.append(cell_index)

print("Relevant cells:", matching_cells)

for cell_index in matching_cells[:8]:
    show_cell_excerpt(
        cell_index,
        [r"batch_ready", r"val_dataloader",
         r"validation_dataset", r"collate"],
        context=3
    )

print("\nRead-only baseline setup recovery completed.")

=== A.3.2. EXACT BASELINE SETUP ===

=== CHECKPOINT LOADING ===

=== NOTEBOOK 01, CELL 7 ===
    ...
  1: import os
  2: 
  3: os.environ["PIE_RUNS_ROOT"] = str(PROJECT_ROOT / "runs")
  4: from pie.predict import load_checkpoint
  5: 
  6: checkpoint_path = CHECKPOINT_DIR / "best_auprc.ckpt"
  7: 
  8: loaded = load_checkpoint(checkpoint_path)
  9: 
 10: print("Checkpoint loaded successfully!")
 11: print("Checkpoint:", checkpoint_path.name)
 12: 
 13: print("\nModel configuration:")
 14: print("Model width:", loaded.config.model.d_model)
 15: print("Latent slots:", loaded.config.model.n_latents)
 16: print("Encoder layers:", loaded.config.model.n_encoder_layers)

=== DATA MODULE INITIALIZATION ===

=== DATA MODULE METADATA ===

=== NOTEBOOK 01, CELL 12 ===
    ...
  2: dm.setup_stats(rank=0)
  3: 
  4: # Retrieve source dimensions and gene embeddings
  5: source_dims = dict(dm.source_dims)
  6: gene_query_text = dm.gene_query_text()
  7: 
  8: print("Data module setup completed.")
  9

### A.3.3. Pretrained PIE Validation Baseline

The pretrained PIE model is reconstructed using the
original checkpoint-loading and data-module procedures
from notebooks 01 and 02.

The first condition from the existing 32-condition
validation activation dataset is selected using its
recorded dataset index.

The official PIE collate function prepares the input
batch, and the model is evaluated without intervention.

A forward hook captures the output of decoder.0.
Because PIE processes output genes in chunks, the
captured tensors are concatenated in execution order.

The reconstructed activations and gene identifiers
are checked against the previously saved validation
HDF5 dataset.

This establishes a fixed baseline for subsequent
in-memory interventions.

No SAE or PIE weights are modified.

In [7]:
import os
import sys
import math
import h5py
import numpy as np
import torch

from pathlib import Path

print("=== A.3.3. PIE VALIDATION BASELINE ===")

# --------------------------------------------------
# 1. Reuse the original PIE environment
# --------------------------------------------------

PIE_ROOT_07 = ROOT / "external" / "pie"
PIE_SRC_07 = PIE_ROOT_07 / "src"

assert PIE_SRC_07.is_dir()

if str(PIE_SRC_07) not in sys.path:
    sys.path.insert(0, str(PIE_SRC_07))

os.environ["PIE_DATA_ROOT"] = str(ROOT / "data")
os.environ["PIE_CACHE_DIR"] = str(ROOT / "cache")
os.environ["PIE_RUNS_ROOT"] = str(ROOT / "runs")

from pie.predict import load_checkpoint, _batch_to
from pie.data.datamodule import PieDataModule
from pie.model.pie import PieModel

CHECKPOINT_07 = (
    ROOT
    / "models/replogle_wdataset/k562/best_auprc.ckpt"
)

VAL_H5_07 = (
    ROOT
    / "data/activations/expanded_128_32/val_32.h5"
)

assert CHECKPOINT_07.is_file()
assert VAL_H5_07.is_file()

# Match the original CPU extraction configuration.
DEVICE_07 = torch.device("cpu")

# --------------------------------------------------
# 2. Load the original pretrained PIE checkpoint
# --------------------------------------------------

loaded_07 = load_checkpoint(CHECKPOINT_07)

dm_07 = PieDataModule(
    cfg=loaded_07.config.data,
    run_dir=None,
    stats=loaded_07.stats,
    seed=loaded_07.config.seed,
    evidence_source=loaded_07.config.data,
)

dm_07.setup_stats(rank=0)
dm_07.setup(stage="fit")

source_dims_07 = dict(dm_07.source_dims)
gene_query_text_07 = dm_07.gene_query_text()

model_07 = PieModel(
    cfg=loaded_07.config.model,
    stats=loaded_07.stats,
    source_dims=source_dims_07,
    gene_query_text=gene_query_text_07,
)

load_result_07 = model_07.load_state_dict(
    loaded_07.state_dict,
    strict=True,
)

assert len(load_result_07.missing_keys) == 0
assert len(load_result_07.unexpected_keys) == 0

model_07.requires_grad_(False)
model_07 = model_07.to(DEVICE_07)
model_07.eval()

assert len(model_07.decoder) == 4

print("\n=== MODEL ===")
print("Checkpoint:", CHECKPOINT_07.name)
print("Device:", DEVICE_07)
print("Training mode:", model_07.training)
print("Decoder blocks:", len(model_07.decoder))
print("Validation dataset size:", len(dm_07.val_dataset))

# --------------------------------------------------
# 3. Recover the first saved validation condition
# --------------------------------------------------

with h5py.File(VAL_H5_07, "r") as h5:
    assert h5["condition_indices"].shape == (32,)
    val_index_07 = int(h5["condition_indices"][0])
    expected_gene_ids_07 = np.asarray(
        h5["gene_ids"][0],
        dtype=np.int64,
    )

assert 0 <= val_index_07 < len(dm_07.val_dataset)

sample_07 = dm_07.val_dataset[val_index_07]

# Reproduce the official extraction procedure.
collate_fn_07 = dm_07.train_dataloader().collate_fn

batch_07 = collate_fn_07([sample_07])

assert len(batch_07.groups) == 1

current_gene_ids_07 = (
    batch_07.groups[0]
    .gene_ids
    .detach()
    .cpu()
    .numpy()
)

assert np.array_equal(
    current_gene_ids_07,
    expected_gene_ids_07,
), "Validation gene IDs do not match the saved HDF5."

batch_ready_07 = _batch_to(
    batch_07,
    DEVICE_07,
    model_07.latents.dtype,
)

print("\n=== VALIDATION CONDITION ===")
print("Dataset index:", val_index_07)
print("Output genes:", len(current_gene_ids_07))
print("Gene-ID alignment: PASS")

# --------------------------------------------------
# 4. Capture decoder.0 during the real forward pass
# --------------------------------------------------

captured_decoder0_07 = []

def capture_decoder0_07(module, inputs, output):
    if not isinstance(output, torch.Tensor):
        raise TypeError("Expected decoder.0 tensor output.")

    captured_decoder0_07.append(
        output.detach().cpu().clone()
    )

handle_07 = model_07.decoder[0].register_forward_hook(
    capture_decoder0_07
)

try:
    with torch.inference_mode():
        outputs_07 = model_07(batch_ready_07)
finally:
    handle_07.remove()

assert len(outputs_07) == 1

expected_calls_07 = math.ceil(
    len(current_gene_ids_07)
    / int(model_07.cfg.inference_chunk_size)
)

assert len(captured_decoder0_07) == expected_calls_07

activation_decoder0_07 = torch.cat(
    captured_decoder0_07,
    dim=1,
).squeeze(0).numpy()

assert activation_decoder0_07.shape == (6642, 768)

# --------------------------------------------------
# 5. Validate predictions and stored activations
# --------------------------------------------------

out_07 = outputs_07[0]

baseline_predictions_07 = {
    name: getattr(out_07, name).detach().cpu().clone()
    for name in ["de_logits", "lfc", "dp_logits"]
}

for name, tensor in baseline_predictions_07.items():
    assert torch.isfinite(tensor).all()

with h5py.File(VAL_H5_07, "r") as h5:
    stored_decoder0_07 = np.asarray(
        h5["decoder"]["0"][0],
        dtype=np.float32,
    )

max_activation_error_07 = float(
    np.max(
        np.abs(
            activation_decoder0_07 - stored_decoder0_07
        )
    )
)

activation_match_07 = np.allclose(
    activation_decoder0_07,
    stored_decoder0_07,
    rtol=1e-4,
    atol=1e-4,
)

print("\n=== BASELINE OUTPUT ===")

for name, tensor in baseline_predictions_07.items():
    print(f"{name}: {tuple(tensor.shape)}")

print("\n=== DECODER.0 VERIFICATION ===")
print("Hook calls:", len(captured_decoder0_07))
print("Activation shape:", activation_decoder0_07.shape)
print("Maximum activation difference:", max_activation_error_07)
print("Saved HDF5 agreement:", activation_match_07)

if not activation_match_07:
    raise AssertionError(
        "Decoder.0 activations differ from the saved "
        "validation HDF5. Do not proceed to interventions."
    )

print("\nPretrained PIE validation baseline: PASS")

=== A.3.3. PIE VALIDATION BASELINE ===


d:\bio_paper\pie-sae\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



=== MODEL ===
Checkpoint: best_auprc.ckpt
Device: cpu
Training mode: False
Decoder blocks: 4
Validation dataset size: 290

=== VALIDATION CONDITION ===
Dataset index: 1
Output genes: 6642
Gene-ID alignment: PASS

=== BASELINE OUTPUT ===
de_logits: (1, 6642, 2)
lfc: (1, 6642)
dp_logits: (1, 6642, 17)

=== DECODER.0 VERIFICATION ===
Hook calls: 4
Activation shape: (6642, 768)
Maximum activation difference: 0.0
Saved HDF5 agreement: True

Pretrained PIE validation baseline: PASS


## A.4. Identity Intervention Control

Before applying feature ablation, the ability to replace
a decoder-block output is tested without changing its
numerical values.

An identity hook returns a cloned copy of each
decoder.0 output tensor during a PIE forward pass.

The hook must execute once per decoder gene chunk.

The resulting DE logits, log-fold changes, and delta-p
logits are compared against the unmodified baseline.

Agreement verifies the basic hook replacement mechanism,
but does not yet validate an SAE-based intervention.

The hook is removed immediately after inference.

In [8]:
import torch

print("=== A.4. IDENTITY INTERVENTION CONTROL ===")

assert model_07.training is False
assert len(baseline_predictions_07) == 3

identity_hook_calls_07 = 0

def identity_hook_07(module, inputs, output):
    global identity_hook_calls_07

    if not isinstance(output, torch.Tensor):
        raise TypeError(
            "Unexpected decoder.0 output type."
        )

    identity_hook_calls_07 += 1

    # Return a new tensor with identical values.
    return output.clone()

identity_handle_07 = (
    model_07.decoder[0].register_forward_hook(
        identity_hook_07
    )
)

try:
    with torch.inference_mode():
        identity_outputs_07 = model_07(batch_ready_07)
finally:
    identity_handle_07.remove()

assert len(identity_outputs_07) == 1
assert identity_hook_calls_07 == expected_calls_07

print("\n=== BASELINE VS IDENTITY ===")
print("Identity hook calls:", identity_hook_calls_07)

identity_pass_07 = True

for name, baseline in baseline_predictions_07.items():

    identity = (
        getattr(identity_outputs_07[0], name)
        .detach()
        .cpu()
    )

    max_difference = (
        (identity - baseline).abs().max().item()
    )

    matched = torch.allclose(
        identity,
        baseline,
        rtol=1e-6,
        atol=1e-6,
    )

    identity_pass_07 &= bool(matched)

    print(
        f"{name}: "
        f"max_difference={max_difference:.9e}, "
        f"match={matched}"
    )

if not identity_pass_07:
    raise AssertionError(
        "Identity intervention changed PIE predictions."
    )

print("\nIdentity intervention: PASS")
print("The pretrained model was not modified.")

=== A.4. IDENTITY INTERVENTION CONTROL ===

=== BASELINE VS IDENTITY ===
Identity hook calls: 4
de_logits: max_difference=0.000000000e+00, match=True
lfc: max_difference=0.000000000e+00, match=True
dp_logits: max_difference=0.000000000e+00, match=True

Identity intervention: PASS
The pretrained model was not modified.


## B. Single-Feature Computational Interventions

### B.1. Decoder.0 Feature 189 Ablation

The unmodified PIE baseline and the identity intervention
have both passed exact numerical checks.

This experiment performs the first non-identity
intervention on the pretrained PIE model.

The trained decoder.0 TopK SAE is used to encode each
hidden representation after subtracting the training
activation mean.

For feature 189, its decoded contribution is subtracted
from the original PIE hidden representation:

h_ablated = h - z_189 * w_dec_189

The complete hidden state is not replaced by an SAE
reconstruction. Here, "feature ablation" means subtracting
the decoded contribution `z_j * w_dec_j` from the original
PIE hidden state. Because the SAE encoder is nonlinear and
its learned directions are not necessarily orthogonal, this
operator does **not** guarantee that re-encoding the modified
hidden state produces `z_j = 0`. No post-intervention SAE
re-encoding zero-activation check is claimed.

The intervention is applied to all four decoder chunks
during a single validation-condition forward pass.

The experiment reports feature activation statistics,
intervention magnitude, and differences in PIE outputs.

This is a computational intervention on a learned
representation component. It does not independently
establish the biological function of the feature.

All pretrained weights and research artifacts remain
unchanged.

In [9]:
import json
import sys
import numpy as np
import torch

print("=== B.1. DECODER.0 FEATURE 189 ABLATION ===")

FEATURE_07 = 189
LAYER_07 = "decoder.0"

assert model_07.training is False
assert DEVICE_07.type == "cpu"
assert activation_match_07
assert identity_pass_07

# --------------------------------------------------
# 1. Load the previously trained decoder.0 SAE
# --------------------------------------------------

sae_source_07 = ROOT / "external/bio-sae/src"

if str(sae_source_07) not in sys.path:
    sys.path.insert(0, str(sae_source_07))

from sae_model import TopKSAE

manifest_07 = json.loads(
    (
        ROOT
        / "results/sae_training/independent_sae_manifest.json"
    ).read_text(encoding="utf-8")
)

registry_07 = {
    entry["layer"]: entry
    for entry in manifest_07["checkpoints"]
}

sae_path_07 = (
    ROOT / registry_07[LAYER_07]["checkpoint"]
)

checkpoint_sae_07 = torch.load(
    sae_path_07,
    map_location="cpu",
    weights_only=True,
)

sae_07 = TopKSAE(
    d_model=768,
    n_features=3072,
    k=32,
)

sae_07.load_state_dict(
    checkpoint_sae_07["model_state_dict"],
    strict=True,
)

sae_07.requires_grad_(False)
sae_07.eval()

mean_07 = torch.from_numpy(
    np.load(
        sae_path_07.parent / "activation_mean.npy"
    ).astype(np.float32)
)

assert mean_07.shape == (768,)

# The original bio-sae decoder dictionary
# contains one 768-dimensional row per feature.

# Recover decoder directions with one row per SAE feature.
W_dec_raw_07 = sae_07.W_dec.weight.detach()

print("Raw decoder weight shape:", tuple(W_dec_raw_07.shape))

if W_dec_raw_07.shape == (768, 3072):
    W_dec_07 = W_dec_raw_07.T.contiguous()
elif W_dec_raw_07.shape == (3072, 768):
    W_dec_07 = W_dec_raw_07
else:
    raise ValueError(
        f"Unexpected SAE decoder shape: {tuple(W_dec_raw_07.shape)}"
    )

assert W_dec_07.shape == (3072, 768)

direction_189_07 = W_dec_07[FEATURE_07]

assert direction_189_07.shape == (768,)

print("Feature-oriented dictionary shape:", tuple(W_dec_07.shape))
print("Feature 189 direction shape:", tuple(direction_189_07.shape))

print("\n=== SAE CHECKPOINT ===")
print("Layer:", LAYER_07)
print("Feature:", FEATURE_07)
print("SAE training mode:", sae_07.training)
print("Decoder weight shape:", tuple(W_dec_07.shape))
print(
    "Feature direction norm:",
    float(torch.linalg.vector_norm(direction_189_07))
)

# --------------------------------------------------
# 2. Verify the single-feature decoder contribution
# --------------------------------------------------

with torch.inference_mode():

    test_hidden = torch.from_numpy(
        activation_decoder0_07[:16].copy()
    )

    test_z, _ = sae_07.encode(
        test_hidden - mean_07
    )

    assert test_z.shape == (16, 3072)

    test_z_off = test_z.clone()
    test_z_off[:, FEATURE_07] = 0.0

    expected_delta = (
        test_z[:, FEATURE_07, None]
        * direction_189_07[None, :]
    )

    # If the SAE exposes a decode method,
    # verify the decoded difference directly.
    if callable(getattr(sae_07, "decode", None)):

        reconstruction = sae_07.decode(test_z)
        reconstruction_off = sae_07.decode(test_z_off)

        if isinstance(reconstruction, tuple):
            reconstruction = reconstruction[0]

        if isinstance(reconstruction_off, tuple):
            reconstruction_off = reconstruction_off[0]

        assert torch.allclose(
            reconstruction - reconstruction_off,
            expected_delta,
            rtol=1e-5,
            atol=1e-5,
        )

        print("SAE decoder contribution check: PASS")

    else:
        print(
            "Direct SAE decode method unavailable; "
            "using the checkpoint decoder direction."
        )

# --------------------------------------------------
# 3. Define the in-memory ablation hook
# --------------------------------------------------

ablation_records_07 = []

def ablate_feature_189_07(module, inputs, output):

    assert isinstance(output, torch.Tensor)
    assert output.shape[-1] == 768

    original_shape = output.shape

    hidden = output.reshape(-1, 768)

    with torch.no_grad():

        centered = hidden - mean_07

        z, _ = sae_07.encode(centered)

        feature_activation = z[:, FEATURE_07]

        contribution = (
            feature_activation[:, None]
            * direction_189_07[None, :]
        )

        modified_hidden = hidden - contribution

    assert modified_hidden.shape == hidden.shape
    assert torch.isfinite(modified_hidden).all()

    ablation_records_07.append({
        "positions": int(hidden.shape[0]),
        "active_positions": int(
            (feature_activation > 0).sum().item()
        ),
        "mean_activation": float(
            feature_activation.mean().item()
        ),
        "max_activation": float(
            feature_activation.max().item()
        ),
        "max_hidden_change": float(
            contribution.abs().max().item()
        ),
    })

    return modified_hidden.reshape(original_shape)

# --------------------------------------------------
# 4. Execute the intervened PIE forward pass
# --------------------------------------------------

handle_ablation_07 = (
    model_07.decoder[0].register_forward_hook(
        ablate_feature_189_07
    )
)

try:
    with torch.inference_mode():
        ablated_outputs_07 = model_07(batch_ready_07)
finally:
    handle_ablation_07.remove()

assert len(ablated_outputs_07) == 1
assert len(ablation_records_07) == expected_calls_07

# --------------------------------------------------
# 5. Report intervention activation statistics
# --------------------------------------------------

total_positions_07 = sum(
    item["positions"]
    for item in ablation_records_07
)

active_positions_07 = sum(
    item["active_positions"]
    for item in ablation_records_07
)

assert total_positions_07 == 6642

print("\n=== FEATURE ACTIVATION ===")

print("Decoder hook calls:", len(ablation_records_07))
print("Processed positions:", total_positions_07)
print("Active positions:", active_positions_07)

print(
    "Active position fraction:",
    f"{active_positions_07 / total_positions_07:.6f}"
)

print(
    "Maximum hidden-state modification:",
    max(
        item["max_hidden_change"]
        for item in ablation_records_07
    )
)

# --------------------------------------------------
# 6. Compare PIE predictions with the baseline
# --------------------------------------------------

effect_records_07 = []

print("\n=== BASELINE VS FEATURE ABLATION ===")

for name, baseline in baseline_predictions_07.items():

    modified = (
        getattr(ablated_outputs_07[0], name)
        .detach()
        .cpu()
    )

    assert modified.shape == baseline.shape
    assert torch.isfinite(modified).all()

    delta = modified - baseline

    result = {
        "output": name,
        "mean_abs_change": float(
            delta.abs().mean().item()
        ),
        "max_abs_change": float(
            delta.abs().max().item()
        ),
        "changed_elements_gt_1e-6": int(
            (delta.abs() > 1e-6).sum().item()
        ),
    }

    effect_records_07.append(result)

    print(f"\n{name}")
    print(
        "Mean absolute change:",
        f"{result['mean_abs_change']:.9e}"
    )
    print(
        "Maximum absolute change:",
        f"{result['max_abs_change']:.9e}"
    )
    print(
        "Elements changed > 1e-6:",
        result["changed_elements_gt_1e-6"]
    )

print("\n=== INTERVENTION STATUS ===")

if active_positions_07 == 0:
    print(
        "Feature 189 is inactive on this condition. "
        "No causal effect can be assessed here."
    )
else:
    print("Feature 189 had positive activations.")

print("Ablation forward completed.")
print("All intervention hooks were removed.")
print("Pretrained PIE and SAE weights remain unchanged.")

=== B.1. DECODER.0 FEATURE 189 ABLATION ===
Raw decoder weight shape: (768, 3072)
Feature-oriented dictionary shape: (3072, 768)
Feature 189 direction shape: (768,)

=== SAE CHECKPOINT ===
Layer: decoder.0
Feature: 189
SAE training mode: False
Decoder weight shape: (3072, 768)
Feature direction norm: 0.5036172270774841
SAE decoder contribution check: PASS

=== FEATURE ACTIVATION ===
Decoder hook calls: 4
Processed positions: 6642
Active positions: 68
Active position fraction: 0.010238
Maximum hidden-state modification: 1.1093850135803223

=== BASELINE VS FEATURE ABLATION ===

de_logits
Mean absolute change: 9.292525356e-04
Maximum absolute change: 3.980319500e-01
Elements changed > 1e-6: 136

lfc
Mean absolute change: 8.488043095e-04
Maximum absolute change: 3.751001954e-01
Elements changed > 1e-6: 68

dp_logits
Mean absolute change: 2.425878542e-03
Maximum absolute change: 1.220525742e+00
Elements changed > 1e-6: 1156

=== INTERVENTION STATUS ===
Feature 189 had positive activations.


### B.2. Dose–Response and Matched-Direction Controls

The initial decoder.0 feature 189 ablation produced
nonzero changes in the pretrained PIE predictions.

This experiment evaluates the effect of intervention
strength and compares the target intervention against
random directions of matched Euclidean norm.

The intervention is defined as:

h_modified = h - alpha * z_189 * direction

Three target-feature strengths are evaluated:
alpha = 0.5, 1.0, and 1.5.

Five random directions are also tested at alpha = 1.
They are orthogonalized against the target direction
and rescaled to match its norm.

The original feature-189 activation gate is retained
in the random-direction controls. Therefore, the
controls perturb the same positions with matched
hidden-state displacement magnitudes.

The experiment reports changes in DE logits,
log-fold-change predictions, and delta-p logits.

All comparisons use the same fixed validation input.
These pilot controls do not establish statistical
significance or biological specificity.

PIE and SAE weights remain unchanged.

In [10]:
import numpy as np
import pandas as pd
import torch

print("=== B.2. DOSE-RESPONSE AND DIRECTION CONTROLS ===")

assert identity_pass_07
assert len(ablation_records_07) == expected_calls_07
assert FEATURE_07 == 189

# Report the previously computed B.1 effects.
print("\n=== ORIGINAL B.1 OUTPUT EFFECTS ===")
print(pd.DataFrame(effect_records_07).to_string(index=False))

# --------------------------------------------------
# 1. Reusable in-memory intervention
# --------------------------------------------------

@torch.inference_mode()
def run_intervention_b2(direction, strength):

    direction = direction.detach().to(
        device=DEVICE_07,
        dtype=direction_189_07.dtype
    )

    assert direction.shape == (768,)
    assert torch.isfinite(direction).all()

    hook_counts = {"calls": 0, "active": 0}

    def intervention_hook(module, inputs, output):

        hidden = output.reshape(-1, 768)

        z, _ = sae_07.encode(
            hidden - mean_07
        )

        activation = z[:, FEATURE_07]

        hook_counts["calls"] += 1
        hook_counts["active"] += int(
            (activation > 0).sum().item()
        )

        delta = (
            strength
            * activation[:, None]
            * direction[None, :]
        )

        return (hidden - delta).reshape(output.shape)

    handle = model_07.decoder[0].register_forward_hook(
        intervention_hook
    )

    try:
        output = model_07(batch_ready_07)[0]
    finally:
        handle.remove()

    assert hook_counts["calls"] == expected_calls_07
    assert hook_counts["active"] == active_positions_07

    return {
        name: getattr(output, name).detach().cpu().clone()
        for name in baseline_predictions_07
    }


def compare_outputs_b2(output_dict, label):

    rows = []

    for name, baseline in baseline_predictions_07.items():

        modified = output_dict[name]

        assert modified.shape == baseline.shape
        assert torch.isfinite(modified).all()

        delta = (modified - baseline).abs()

        rows.append({
            "intervention": label,
            "output": name,
            "mean_abs_change": float(delta.mean()),
            "max_abs_change": float(delta.max()),
            "changed_gt_1e-6": int(
                (delta > 1e-6).sum()
            ),
        })

    return rows


# --------------------------------------------------
# 2. Feature-189 dose-response
# --------------------------------------------------

results_b2 = []

for strength in [0.5, 1.0, 1.5]:

    prediction = run_intervention_b2(
        direction_189_07,
        strength
    )

    # Reproduce the original B.1 ablation.
    if strength == 1.0:

        for name in baseline_predictions_07:

            original_b1 = getattr(
                ablated_outputs_07[0],
                name
            ).detach().cpu()

            assert torch.allclose(
                prediction[name],
                original_b1,
                rtol=1e-6,
                atol=1e-6
            )

        print("B.1 reproduction at alpha=1: PASS")

    results_b2.extend(
        compare_outputs_b2(
            prediction,
            f"feature189_alpha{strength}"
        )
    )

# --------------------------------------------------
# 3. Matched random orthogonal directions
# --------------------------------------------------

rng_b2 = torch.Generator(device="cpu")
rng_b2.manual_seed(20261114)

target_direction = direction_189_07.detach().cpu()

target_norm = torch.linalg.vector_norm(
    target_direction
)

assert target_norm > 0

for index in range(5):

    random_direction = torch.randn(
        768,
        generator=rng_b2,
        dtype=target_direction.dtype
    )

    # Remove the component aligned with feature 189.
    projection = (
        torch.dot(random_direction, target_direction)
        / torch.dot(target_direction, target_direction)
    )

    random_direction = (
        random_direction
        - projection * target_direction
    )

    random_norm = torch.linalg.vector_norm(
        random_direction
    )

    assert random_norm > 0

    random_direction = (
        random_direction
        / random_norm
        * target_norm
    )

    assert torch.allclose(
        torch.linalg.vector_norm(random_direction),
        target_norm,
        rtol=1e-5,
        atol=1e-6
    )

    assert abs(
        torch.dot(
            random_direction,
            target_direction
        ).item()
    ) < 1e-5

    prediction = run_intervention_b2(
        random_direction,
        1.0
    )

    results_b2.extend(
        compare_outputs_b2(
            prediction,
            f"random_{index + 1}"
        )
    )

# --------------------------------------------------
# 4. Summaries
# --------------------------------------------------

B2_results = pd.DataFrame(results_b2)

print("\n=== DOSE-RESPONSE RESULTS ===")

dose = B2_results[
    B2_results["intervention"].str.startswith("feature189")
]

print(
    dose[
        ["intervention", "output",
         "mean_abs_change", "max_abs_change"]
    ].to_string(index=False)
)

print("\n=== RANDOM-DIRECTION CONTROLS ===")

random_results = B2_results[
    B2_results["intervention"].str.startswith("random")
]

control_summary = (
    random_results
    .groupby("output")
    .agg(
        random_mean_change=("mean_abs_change", "mean"),
        random_max_change=("mean_abs_change", "max")
    )
)

target_alpha1 = (
    dose[dose["intervention"] == "feature189_alpha1.0"]
    .set_index("output")["mean_abs_change"]
)

control_summary["feature189_change"] = target_alpha1

print(control_summary.to_string())

print("\n=== INTEGRITY ===")
print("Intervention runs:", B2_results["intervention"].nunique())
print("Original B.1 reproduction: PASS")
print("All intervention hooks removed.")
print("No research files modified.")

=== B.2. DOSE-RESPONSE AND DIRECTION CONTROLS ===

=== ORIGINAL B.1 OUTPUT EFFECTS ===
   output  mean_abs_change  max_abs_change  changed_elements_gt_1e-6
de_logits         0.000929        0.398032                       136
      lfc         0.000849        0.375100                        68
dp_logits         0.002426        1.220526                      1156
B.1 reproduction at alpha=1: PASS

=== DOSE-RESPONSE RESULTS ===
       intervention    output  mean_abs_change  max_abs_change
feature189_alpha0.5 de_logits         0.000435        0.190662
feature189_alpha0.5       lfc         0.000413        0.224868
feature189_alpha0.5 dp_logits         0.001212        0.702113
feature189_alpha1.0 de_logits         0.000929        0.398032
feature189_alpha1.0       lfc         0.000849        0.375100
feature189_alpha1.0 dp_logits         0.002426        1.220526
feature189_alpha1.5 de_logits         0.001455        0.549300
feature189_alpha1.5       lfc         0.001286        0.528404
featu

### B.3. Cross-Context Intervention Pilot

The single-condition experiment established a
dose-dependent effect of feature 189 ablation.

However, its average effect on DE logits was only
slightly larger than the mean of five matched random
direction controls.

This experiment evaluates generalization across
cell contexts using two saved validation conditions
from each of HepG2, Jurkat, and RPE1.

For each condition, the analysis compares the
unmodified PIE baseline against feature 189 ablation
and five norm-matched orthogonal direction controls.

The random directions and intervention strength
are held fixed across conditions.

All runs use the original validation data pipeline,
the pretrained PIE model, and the trained decoder.0
SAE without updating model parameters.

This is a descriptive six-condition pilot, not an
independent biological replication or a confirmatory
statistical test.

In [11]:
import h5py
import numpy as np
import pandas as pd
import torch

print("=== B.3. CROSS-CONTEXT INTERVENTION PILOT ===")

assert identity_pass_07
assert FEATURE_07 == 189
assert model_07.training is False
assert sae_07.training is False

# --------------------------------------------------
# 1. Select two saved conditions per cell context
# --------------------------------------------------

with h5py.File(VAL_H5_07, "r") as h5:
    saved_indices = np.asarray(
        h5["condition_indices"][:],
        dtype=np.int64
    )

context_groups = {
    "hepg2": [],
    "jurkat": [],
    "rpe1": [],
}

for position, dataset_index in enumerate(saved_indices):
    row = dm_07.val_dataset.rows[int(dataset_index)]
    context = str(row.context).strip().lower()

    if context in context_groups:
        context_groups[context].append(
            (position, int(dataset_index))
        )

selected_conditions = []

for context, candidates in context_groups.items():
    assert len(candidates) >= 2, (
        f"Insufficient validation conditions: {context}"
    )

    for position, index in candidates[:2]:
        selected_conditions.append(
            (context, position, index)
        )

assert len(selected_conditions) == 6

print("\n=== SELECTED CONDITIONS ===")

for context, position, index in selected_conditions:
    print(
        f"{context}: HDF5 row={position}, "
        f"dataset index={index}"
    )

# --------------------------------------------------
# 2. Construct fixed matched control directions
# --------------------------------------------------

generator_b3 = torch.Generator(device="cpu")
generator_b3.manual_seed(20261114)

target_b3 = direction_189_07.detach().cpu()

target_norm_b3 = torch.linalg.vector_norm(target_b3)

directions_b3 = {
    "feature189": target_b3
}

for i in range(5):

    direction = torch.randn(
        768,
        generator=generator_b3,
        dtype=target_b3.dtype
    )

    direction -= (
        torch.dot(direction, target_b3)
        / torch.dot(target_b3, target_b3)
    ) * target_b3

    direction = (
        direction
        / torch.linalg.vector_norm(direction)
        * target_norm_b3
    )

    assert abs(
        torch.dot(direction, target_b3).item()
    ) < 1e-5

    directions_b3[f"random_{i + 1}"] = direction

# --------------------------------------------------
# 3. Define reusable intervention on any batch
# --------------------------------------------------

def run_b3_intervention(batch_ready, direction):

    hook_state = {
        "calls": 0,
        "active_positions": 0
    }

    def hook(module, inputs, output):

        hidden = output.reshape(-1, 768)

        z, _ = sae_07.encode(
            hidden - mean_07
        )

        activation = z[:, FEATURE_07]

        hook_state["calls"] += 1
        hook_state["active_positions"] += int(
            (activation > 0).sum().item()
        )

        delta = (
            activation[:, None]
            * direction[None, :]
        )

        return (hidden - delta).reshape(
            output.shape
        )

    handle = model_07.decoder[0].register_forward_hook(
        hook
    )

    try:
        with torch.inference_mode():
            result = model_07(batch_ready)[0]
    finally:
        handle.remove()

    assert hook_state["calls"] == expected_calls_07

    predictions = {
        name: getattr(result, name).detach().cpu().clone()
        for name in baseline_predictions_07
    }

    return predictions, hook_state

# --------------------------------------------------
# 4. Evaluate the six selected conditions
# --------------------------------------------------

records_b3 = []

for context, h5_position, dataset_index in selected_conditions:

    sample = dm_07.val_dataset[dataset_index]

    batch = collate_fn_07([sample])

    batch_ready = _batch_to(
        batch,
        DEVICE_07,
        model_07.latents.dtype
    )

    with torch.inference_mode():
        baseline_output = model_07(batch_ready)[0]

    baseline = {
        name: getattr(
            baseline_output, name
        ).detach().cpu().clone()
        for name in baseline_predictions_07
    }

    for label, direction in directions_b3.items():

        predictions, state = run_b3_intervention(
            batch_ready,
            direction
        )

        for name, original in baseline.items():

            delta = (
                predictions[name] - original
            ).abs()

            records_b3.append({
                "context": context,
                "dataset_index": dataset_index,
                "intervention": label,
                "output": name,
                "active_positions": state[
                    "active_positions"
                ],
                "mean_abs_change": float(
                    delta.mean()
                ),
                "max_abs_change": float(
                    delta.max()
                ),
            })

    print(
        f"Completed: {context}, "
        f"dataset index {dataset_index}"
    )

B3_results = pd.DataFrame(records_b3)

assert len(B3_results) == 6 * 6 * 3
assert np.isfinite(
    B3_results[
        ["mean_abs_change", "max_abs_change"]
    ].to_numpy()
).all()

# --------------------------------------------------
# 5. Compare feature 189 with random controls
# --------------------------------------------------

target_results = B3_results[
    B3_results["intervention"] == "feature189"
].copy()

random_results = B3_results[
    B3_results["intervention"].str.startswith(
        "random_"
    )
]

random_means = (
    random_results
    .groupby(
        ["context", "dataset_index", "output"]
    )["mean_abs_change"]
    .mean()
    .rename("random_mean_change")
    .reset_index()
)

comparison_b3 = target_results.merge(
    random_means,
    on=["context", "dataset_index", "output"],
    validate="one_to_one"
)

comparison_b3["target_to_random_ratio"] = (
    comparison_b3["mean_abs_change"]
    / comparison_b3["random_mean_change"].replace(
        0, np.nan
    )
)

print("\n=== CROSS-CONTEXT RESULTS ===")

print(
    comparison_b3[
        [
            "context",
            "dataset_index",
            "output",
            "active_positions",
            "mean_abs_change",
            "random_mean_change",
            "target_to_random_ratio",
        ]
    ].round(5).to_string(index=False)
)

print("\n=== SUMMARY BY OUTPUT ===")

summary_b3 = comparison_b3.groupby("output").agg(
    mean_target_change=("mean_abs_change", "mean"),
    mean_random_change=("random_mean_change", "mean"),
    mean_active_positions=("active_positions", "mean"),
    conditions_target_above_random=(
        "target_to_random_ratio",
        lambda x: int((x > 1).sum())
    ),
)

summary_b3["ratio_of_mean_changes"] = (
    summary_b3["mean_target_change"]
    / summary_b3["mean_random_change"]
)

print(summary_b3.round(5).to_string())

print("\nRead-only cross-context pilot completed.")
print("All intervention hooks removed.")

=== B.3. CROSS-CONTEXT INTERVENTION PILOT ===

=== SELECTED CONDITIONS ===
hepg2: HDF5 row=0, dataset index=1
hepg2: HDF5 row=1, dataset index=5
jurkat: HDF5 row=10, dataset index=103
jurkat: HDF5 row=11, dataset index=104
rpe1: HDF5 row=23, dataset index=204
rpe1: HDF5 row=24, dataset index=221
Completed: hepg2, dataset index 1
Completed: hepg2, dataset index 5
Completed: jurkat, dataset index 103
Completed: jurkat, dataset index 104
Completed: rpe1, dataset index 204
Completed: rpe1, dataset index 221

=== CROSS-CONTEXT RESULTS ===
context  dataset_index    output  active_positions  mean_abs_change  random_mean_change  target_to_random_ratio
  hepg2              1 de_logits                68          0.00093             0.00084                 1.10687
  hepg2              1       lfc                68          0.00085             0.00051                 1.66991
  hepg2              1 dp_logits                68          0.00243             0.00150                 1.61916
  hepg2     

### B.4. Gene-Level Localization of Feature 189 Effects

The cross-context pilot showed that feature 189 ablation
can alter PIE predictions, although its effects vary
across conditions and are not consistently larger than
matched random-direction controls.

This experiment examines the spatial localization of the
initial intervention on the first validation condition.

The original decoder.0 activations are re-encoded with
the trained SAE to identify positions where feature 189
is active.

For each PIE output, absolute prediction changes are
summarized separately for active and inactive positions.

The analysis also identifies the gene positions with
the largest changes in DE logits.

Because the intervention is explicitly gated by feature
activation, localization to active positions is an
implementation property rather than independent evidence
of biological specificity.

The analysis uses previously computed predictions and
performs no additional PIE forward passes.

In [12]:
import numpy as np
import pandas as pd
import torch

print("=== B.4. GENE-LEVEL EFFECT LOCALIZATION ===")

assert FEATURE_07 == 189
assert activation_decoder0_07.shape == (6642, 768)
assert len(current_gene_ids_07) == 6642

# --------------------------------------------------
# 1. Identify feature-active gene positions
# --------------------------------------------------

with torch.inference_mode():

    hidden_b4 = torch.from_numpy(
        activation_decoder0_07.copy()
    )

    z_b4, _ = sae_07.encode(
        hidden_b4 - mean_07
    )

    feature_activation_b4 = (
        z_b4[:, FEATURE_07]
        .detach()
        .cpu()
        .numpy()
    )

active_mask_b4 = feature_activation_b4 > 0

assert int(active_mask_b4.sum()) == active_positions_07

print("\n=== FEATURE ACTIVITY ===")

print("Total gene positions:", len(active_mask_b4))
print("Active positions:", int(active_mask_b4.sum()))
print("Inactive positions:", int((~active_mask_b4).sum()))

# --------------------------------------------------
# 2. Measure effects per gene
# --------------------------------------------------

effect_rows_b4 = []
per_gene_changes_b4 = {}

for name, baseline in baseline_predictions_07.items():

    modified = (
        getattr(ablated_outputs_07[0], name)
        .detach()
        .cpu()
    )

    assert modified.shape == baseline.shape

    # Average absolute change across output channels.
    per_gene = (
        (modified - baseline)
        .abs()
        .reshape(6642, -1)
        .mean(dim=1)
        .numpy()
    )

    per_gene_changes_b4[name] = per_gene

    active_values = per_gene[active_mask_b4]
    inactive_values = per_gene[~active_mask_b4]

    total_change = float(per_gene.sum())

    effect_rows_b4.append({
        "output": name,
        "mean_abs_all": float(per_gene.mean()),
        "mean_abs_active": float(active_values.mean()),
        "mean_abs_inactive": float(inactive_values.mean()),
        "active_genes_changed": int(
            (active_values > 1e-6).sum()
        ),
        "inactive_genes_changed": int(
            (inactive_values > 1e-6).sum()
        ),
        "fraction_effect_on_active": (
            float(active_values.sum() / total_change)
            if total_change > 0 else np.nan
        ),
    })

B4_summary = pd.DataFrame(effect_rows_b4)

print("\n=== ACTIVE VS INACTIVE GENE EFFECTS ===")

print(
    B4_summary.round(8).to_string(index=False)
)

# --------------------------------------------------
# 3. Identify the largest DE-logit changes
# --------------------------------------------------

de_changes_b4 = per_gene_changes_b4["de_logits"]

top_positions_b4 = np.argsort(
    -de_changes_b4
)[:10]

top_genes_b4 = pd.DataFrame({
    "gene_position": top_positions_b4,
    "gene_id": current_gene_ids_07[top_positions_b4],
    "feature_activation": (
        feature_activation_b4[top_positions_b4]
    ),
    "active": active_mask_b4[top_positions_b4],
    "mean_abs_de_logit_change": (
        de_changes_b4[top_positions_b4]
    ),
})

print("\n=== TOP 10 CHANGED GENE POSITIONS ===")

print(
    top_genes_b4.round(6).to_string(index=False)
)

# --------------------------------------------------
# 4. Verify output localization
# --------------------------------------------------

print("\n=== LOCALIZATION VERIFICATION ===")

for name, per_gene in per_gene_changes_b4.items():

    max_inactive = float(
        per_gene[~active_mask_b4].max()
    )

    print(
        f"{name}: "
        f"maximum inactive-position change = "
        f"{max_inactive:.9e}"
    )

print("\nRead-only gene-level analysis completed.")

=== B.4. GENE-LEVEL EFFECT LOCALIZATION ===

=== FEATURE ACTIVITY ===
Total gene positions: 6642
Active positions: 68
Inactive positions: 6574

=== ACTIVE VS INACTIVE GENE EFFECTS ===
   output  mean_abs_all  mean_abs_active  mean_abs_inactive  active_genes_changed  inactive_genes_changed  fraction_effect_on_active
de_logits      0.000929         0.090766                0.0                    68                       0                        1.0
      lfc      0.000849         0.082908                0.0                    68                       0                        1.0
dp_logits      0.002426         0.236951                0.0                    68                       0                        1.0

=== TOP 10 CHANGED GENE POSITIONS ===
 gene_position  gene_id  feature_activation  active  mean_abs_de_logit_change
          2768     2768           15.881584    True                  0.279709
          3899     3899           11.778842    True                  0.247788
           

### B.5. Full-Validation Feature 189 Ablation

The initial feature 189 intervention produced nonzero
changes in PIE predictions, with effects confined to
the feature-active gene positions in the examined
validation condition.

A six-condition cross-context pilot suggested variation
in effect magnitude across experimental conditions.

This experiment extends the analysis to all 32 saved
validation conditions:

- HepG2: 10 conditions.
- Jurkat: 13 conditions.
- RPE1: 9 conditions.

For each condition, the original PIE baseline and
feature 189 ablation are evaluated using identical inputs.

The experiment measures changes in DE logits,
log-fold-change predictions, and delta-p logits.

Condition-level results are retained in memory for
subsequent matched-control and uncertainty analyses.

The evaluation uses a fixed pretrained PIE model and
a fixed trained SAE. Conditions are observational
evaluation units, not independent biological replicates.

No checkpoints, activation datasets, or previous
research results are modified.

In [13]:
import h5py
import numpy as np
import pandas as pd
import torch

print("=== B.5. FULL-VALIDATION FEATURE 189 ABLATION ===")

assert model_07.training is False
assert sae_07.training is False
assert FEATURE_07 == 189
assert identity_pass_07

# --------------------------------------------------
# 1. Recover all saved validation conditions
# --------------------------------------------------

with h5py.File(VAL_H5_07, "r") as h5:
    validation_indices_b5 = np.asarray(
        h5["condition_indices"][:],
        dtype=np.int64
    )

assert len(validation_indices_b5) == 32
assert len(set(validation_indices_b5.tolist())) == 32

# --------------------------------------------------
# 2. Evaluate unmodified and intervened PIE
# --------------------------------------------------

records_b5 = []

B5_OUTPUTS = ("de_logits", "lfc", "dp_logits")

for position, dataset_index in enumerate(validation_indices_b5):

    dataset_index = int(dataset_index)

    row = dm_07.val_dataset.rows[dataset_index]
    context = str(row.context).strip().lower()

    assert context in {"hepg2", "jurkat", "rpe1"}

    sample = dm_07.val_dataset[dataset_index]
    batch = collate_fn_07([sample])

    assert len(batch.groups) == 1

    batch_ready = _batch_to(
        batch,
        DEVICE_07,
        model_07.latents.dtype
    )

    # Unmodified reference prediction.
    with torch.inference_mode():
        baseline_output = model_07(batch_ready)[0]

    baseline = {
        name: getattr(
            baseline_output, name
        ).detach().cpu().clone()
        for name in B5_OUTPUTS
    }

    # Reuse the verified intervention implementation
    # from B.3, with the original feature 189 direction.
    modified, hook_state = run_b3_intervention(
        batch_ready,
        direction_189_07
    )

    active_count = int(
        hook_state["active_positions"]
    )

    assert 0 <= active_count <= 6642

    # Independently reproduce the original A.3
    # baseline when encountering the first condition.
    if position == 0:
        assert dataset_index == val_index_07

        for name in B5_OUTPUTS:
            assert torch.allclose(
                baseline[name],
                baseline_predictions_07[name],
                rtol=1e-6,
                atol=1e-6
            )

        print("Original baseline reproduction: PASS")

    for name in B5_OUTPUTS:

        original = baseline[name]
        intervened = modified[name]

        assert original.shape == intervened.shape
        assert original.shape[1] == 6642

        per_gene_change = (
            (intervened - original)
            .abs()
            .reshape(6642, -1)
            .mean(dim=1)
        )

        assert torch.isfinite(per_gene_change).all()

        total_change = float(
            per_gene_change.sum().item()
        )

        records_b5.append({
            "context": context,
            "dataset_index": dataset_index,
            "hdf5_position": position,
            "output": name,
            "active_positions": active_count,
            "mean_abs_change": float(
                per_gene_change.mean().item()
            ),
            "max_gene_mean_abs_change": float(
                per_gene_change.max().item()
            ),
            "changed_gene_positions": int(
                (per_gene_change > 1e-6).sum().item()
            ),
            "total_change_per_active_position": (
                total_change / active_count
                if active_count > 0
                else np.nan
            ),
        })

    if (position + 1) % 4 == 0:
        print(
            f"Completed {position + 1}/32 "
            "validation conditions"
        )

B5_results = pd.DataFrame(records_b5)

assert len(B5_results) == 32 * 3

assert B5_results.groupby("output")[
    "dataset_index"
].nunique().eq(32).all()

# --------------------------------------------------
# 3. Summarize by cellular context
# --------------------------------------------------

print("\n=== CONTEXT-LEVEL RESULTS ===")

B5_context_summary = (
    B5_results
    .groupby(["context", "output"])
    .agg(
        conditions=("dataset_index", "count"),
        mean_active_positions=("active_positions", "mean"),
        mean_abs_change=("mean_abs_change", "mean"),
        median_abs_change=("mean_abs_change", "median"),
        mean_change_per_active=(
            "total_change_per_active_position", "mean"
        ),
        conditions_with_effect=(
            "changed_gene_positions",
            lambda x: int((x > 0).sum())
        ),
    )
    .reset_index()
)

print(
    B5_context_summary
    .round(6)
    .to_string(index=False)
)

# --------------------------------------------------
# 4. Full-validation summary
# --------------------------------------------------

print("\n=== FULL-VALIDATION SUMMARY ===")

B5_summary = (
    B5_results
    .groupby("output")
    .agg(
        conditions=("dataset_index", "count"),
        mean_abs_change=("mean_abs_change", "mean"),
        median_abs_change=("mean_abs_change", "median"),
        min_abs_change=("mean_abs_change", "min"),
        max_abs_change=("mean_abs_change", "max"),
        mean_active_positions=("active_positions", "mean"),
        conditions_with_effect=(
            "changed_gene_positions",
            lambda x: int((x > 0).sum())
        ),
    )
    .reset_index()
)

print(
    B5_summary.round(6).to_string(index=False)
)

# --------------------------------------------------
# 5. Inspect the variation between conditions
# --------------------------------------------------

print("\n=== STRONGEST DE-LOGIT EFFECTS ===")

de_results_b5 = (
    B5_results[B5_results["output"] == "de_logits"]
    .sort_values("mean_abs_change", ascending=False)
)

print(
    de_results_b5[
        [
            "context",
            "dataset_index",
            "active_positions",
            "mean_abs_change",
            "max_gene_mean_abs_change",
        ]
    ].head(8).round(6).to_string(index=False)
)

print("\nRead-only full-validation analysis completed.")
print("All intervention hooks removed.")
print("Original PIE and SAE checkpoints unchanged.")

=== B.5. FULL-VALIDATION FEATURE 189 ABLATION ===
Original baseline reproduction: PASS
Completed 4/32 validation conditions
Completed 8/32 validation conditions
Completed 12/32 validation conditions
Completed 16/32 validation conditions
Completed 20/32 validation conditions
Completed 24/32 validation conditions
Completed 28/32 validation conditions
Completed 32/32 validation conditions

=== CONTEXT-LEVEL RESULTS ===
context    output  conditions  mean_active_positions  mean_abs_change  median_abs_change  mean_change_per_active  conditions_with_effect
  hepg2 de_logits          10              67.900000         0.000857           0.000841                0.083760                      10
  hepg2 dp_logits          10              67.900000         0.001917           0.001979                0.186377                      10
  hepg2       lfc          10              67.900000         0.001296           0.000893                0.127020                      10
 jurkat de_logits          13   

### B.6. Full-Validation Matched-Direction Controls

The feature 189 ablation produced nonzero prediction
changes across all 32 saved validation conditions.

This experiment evaluates whether those changes are
larger than effects produced by matched perturbations
in random hidden-space directions.

Five fixed random directions from the previous
cross-context pilot are reused across all conditions.

Each control direction is orthogonal to the feature 189
decoder direction and has the same Euclidean norm.

All interventions use the same feature 189 activation
gate, so intervention positions and displacement
magnitudes are matched within each condition.

The previously computed feature 189 effects from B.5
are reused without repeating target ablations.

For each PIE output, paired comparisons are made
at the experimental-condition level.

The five random directions constitute a limited
exploratory control set, not a sufficiently sampled
null distribution for confirmatory significance testing.

Condition-level effects are not independent biological
replicates, and comparisons across the three cell
contexts remain descriptive.

No model parameters or existing research files
are modified.

In [14]:
import h5py
import numpy as np
import pandas as pd
import torch

print("=== B.6. FULL-VALIDATION MATCHED-DIRECTION CONTROLS ===")

assert len(B5_results) == 32 * 3
assert len(validation_indices_b5) == 32
assert FEATURE_07 == 189
assert model_07.training is False
assert sae_07.training is False

B6_OUTPUTS = ("de_logits", "lfc", "dp_logits")

# --------------------------------------------------
# 1. Reuse the five fixed control directions
# --------------------------------------------------

B6_directions = {
    name: direction
    for name, direction in directions_b3.items()
    if name.startswith("random_")
}

assert len(B6_directions) == 5

target_direction_b6 = direction_189_07.detach().cpu()
target_norm_b6 = torch.linalg.vector_norm(
    target_direction_b6
)

for name, direction in B6_directions.items():

    assert direction.shape == (768,)

    assert torch.allclose(
        torch.linalg.vector_norm(direction),
        target_norm_b6,
        rtol=1e-5,
        atol=1e-6,
    )

    assert abs(
        torch.dot(direction, target_direction_b6).item()
    ) < 1e-5

print("Control directions:", len(B6_directions))
print("Direction norm checks: PASS")

# --------------------------------------------------
# 2. Evaluate random-direction controls
# --------------------------------------------------

records_b6 = []

for position, dataset_index in enumerate(validation_indices_b5):

    dataset_index = int(dataset_index)

    row = dm_07.val_dataset.rows[dataset_index]
    context = str(row.context).strip().lower()

    sample = dm_07.val_dataset[dataset_index]
    batch = collate_fn_07([sample])

    batch_ready = _batch_to(
        batch,
        DEVICE_07,
        model_07.latents.dtype
    )

    with torch.inference_mode():
        baseline_output = model_07(batch_ready)[0]

    baseline = {
        name: getattr(
            baseline_output, name
        ).detach().cpu().clone()
        for name in B6_OUTPUTS
    }

    expected_active = int(
        B5_results.loc[
            (B5_results["dataset_index"] == dataset_index)
            & (B5_results["output"] == "de_logits"),
            "active_positions"
        ].iloc[0]
    )

    for control_name, direction in B6_directions.items():

        modified, state = run_b3_intervention(
            batch_ready,
            direction
        )

        assert state["active_positions"] == expected_active

        for output_name in B6_OUTPUTS:

            original = baseline[output_name]
            intervened = modified[output_name]

            assert original.shape == intervened.shape
            assert torch.isfinite(intervened).all()

            per_gene_change = (
                (intervened - original)
                .abs()
                .reshape(6642, -1)
                .mean(dim=1)
            )

            records_b6.append({
                "context": context,
                "dataset_index": dataset_index,
                "hdf5_position": position,
                "intervention": control_name,
                "output": output_name,
                "active_positions": state["active_positions"],
                "mean_abs_change": float(
                    per_gene_change.mean().item()
                ),
            })

    if (position + 1) % 4 == 0:
        print(
            f"Completed {position + 1}/32 "
            "validation conditions"
        )

B6_controls = pd.DataFrame(records_b6)

assert len(B6_controls) == 32 * 5 * 3

# --------------------------------------------------
# 3. Check consistency with the earlier B.3 pilot
# --------------------------------------------------

previous_controls = B3_results[
    B3_results["intervention"].str.startswith("random_")
][
    ["dataset_index", "intervention", "output",
     "mean_abs_change"]
].copy()

comparison_check = previous_controls.merge(
    B6_controls[
        ["dataset_index", "intervention", "output",
         "mean_abs_change"]
    ],
    on=["dataset_index", "intervention", "output"],
    suffixes=("_b3", "_b6"),
    validate="one_to_one"
)

assert len(comparison_check) == 6 * 5 * 3

assert np.allclose(
    comparison_check["mean_abs_change_b3"],
    comparison_check["mean_abs_change_b6"],
    rtol=1e-5,
    atol=1e-7,
)

print("\nPrevious six-condition control reproduction: PASS")

# --------------------------------------------------
# 4. Construct paired condition-level comparisons
# --------------------------------------------------

control_summary_b6 = (
    B6_controls
    .groupby(
        ["context", "dataset_index", "output"],
        as_index=False
    )
    .agg(
        random_mean=("mean_abs_change", "mean"),
        random_min=("mean_abs_change", "min"),
        random_max=("mean_abs_change", "max"),
    )
)

target_b6 = B5_results[
    ["context", "dataset_index", "output",
     "active_positions", "mean_abs_change"]
].rename(
    columns={"mean_abs_change": "target_change"}
)

B6_comparison = target_b6.merge(
    control_summary_b6,
    on=["context", "dataset_index", "output"],
    validate="one_to_one"
)

assert len(B6_comparison) == 32 * 3
assert B6_comparison.notna().all().all()

B6_comparison["paired_difference"] = (
    B6_comparison["target_change"]
    - B6_comparison["random_mean"]
)

B6_comparison["target_to_random_ratio"] = (
    B6_comparison["target_change"]
    / B6_comparison["random_mean"].replace(0, np.nan)
)

B6_comparison["above_random_mean"] = (
    B6_comparison["target_change"]
    > B6_comparison["random_mean"]
)

B6_comparison["above_all_five_controls"] = (
    B6_comparison["target_change"]
    > B6_comparison["random_max"]
)

# --------------------------------------------------
# 5. Full-validation results by PIE output
# --------------------------------------------------

B6_summary = (
    B6_comparison
    .groupby("output")
    .agg(
        conditions=("dataset_index", "count"),
        mean_target=("target_change", "mean"),
        mean_random=("random_mean", "mean"),
        median_paired_difference=(
            "paired_difference", "median"
        ),
        above_random_mean=(
            "above_random_mean", "sum"
        ),
        above_all_controls=(
            "above_all_five_controls", "sum"
        ),
    )
)

B6_summary["ratio_of_means"] = (
    B6_summary["mean_target"]
    / B6_summary["mean_random"]
)

print("\n=== FULL-VALIDATION CONTROL SUMMARY ===")

print(
    B6_summary.round(6).to_string()
)

# --------------------------------------------------
# 6. Context-level results
# --------------------------------------------------

B6_context_summary = (
    B6_comparison
    .groupby(["context", "output"])
    .agg(
        conditions=("dataset_index", "count"),
        mean_target=("target_change", "mean"),
        mean_random=("random_mean", "mean"),
        mean_paired_difference=(
            "paired_difference", "mean"
        ),
        above_random_mean=(
            "above_random_mean", "sum"
        ),
    )
    .reset_index()
)

B6_context_summary["ratio_of_means"] = (
    B6_context_summary["mean_target"]
    / B6_context_summary["mean_random"]
)

print("\n=== CONTEXT-LEVEL CONTROL SUMMARY ===")

print(
    B6_context_summary.round(6).to_string(index=False)
)

# --------------------------------------------------
# 7. Individual control directions
# --------------------------------------------------

B6_direction_summary = (
    B6_controls
    .groupby(["output", "intervention"])
    ["mean_abs_change"]
    .mean()
    .unstack("intervention")
)

print("\n=== MEAN EFFECT BY RANDOM DIRECTION ===")

print(
    B6_direction_summary.round(6).to_string()
)

print("\n=== INTEGRITY ===")
print("Conditions:", B6_comparison["dataset_index"].nunique())
print("Control interventions:", len(B6_directions))
print("Control evaluations:", 32 * len(B6_directions))
print("Earlier pilot reproduction: PASS")
print("All intervention hooks removed.")
print("Original PIE and SAE checkpoints unchanged.")

=== B.6. FULL-VALIDATION MATCHED-DIRECTION CONTROLS ===
Control directions: 5
Direction norm checks: PASS
Completed 4/32 validation conditions
Completed 8/32 validation conditions
Completed 12/32 validation conditions
Completed 16/32 validation conditions
Completed 20/32 validation conditions
Completed 24/32 validation conditions
Completed 28/32 validation conditions
Completed 32/32 validation conditions

Previous six-condition control reproduction: PASS

=== FULL-VALIDATION CONTROL SUMMARY ===
           conditions  mean_target  mean_random  median_paired_difference  above_random_mean  above_all_controls  ratio_of_means
output                                                                                                                          
de_logits          32     0.000959     0.000795                  0.000038                 17                   9        1.206079
dp_logits          32     0.001793     0.001320                  0.000080                 18                   9 

### B.7. Condition-Level Uncertainty and Robustness

The full-validation analysis compared feature 189
ablation with five fixed norm-matched random directions
across 32 validation conditions.

The target direction produced larger mean absolute
prediction changes on average, but did not consistently
outperform the controls for every condition.

This analysis estimates the uncertainty of paired
condition-level effect differences using a stratified
bootstrap.

The resampling procedure:

- Preserves the numbers of HepG2, Jurkat, and RPE1
  conditions.
- Resamples conditions with replacement within contexts.
- Preserves the pairing between target and control
  effects for each sampled condition.
- Uses 5,000 bootstrap replicates with a fixed seed.

The analysis reports percentile intervals for the
mean paired difference and ratio of mean effects.

Leave-one-condition-out sensitivity is also examined.

Control-direction sensitivity is assessed separately
using the five previously selected random directions.

These intervals describe resampling variability
conditional on the observed validation conditions,
fixed SAE, and five selected control directions.

They are exploratory uncertainty diagnostics,
not confirmatory confidence intervals for a
population-level causal or biological effect.

No additional model inference is performed.

In [15]:
import numpy as np
import pandas as pd

print("=== B.7. CONDITION-LEVEL UNCERTAINTY ===")

B7_SEED = 20261117
B7_N_BOOT = 5000

rng_b7 = np.random.default_rng(B7_SEED)

assert len(B6_comparison) == 32 * 3
assert len(B6_controls) == 32 * 5 * 3

contexts_b7 = {
    "hepg2": 10,
    "jurkat": 13,
    "rpe1": 9,
}

outputs_b7 = ["de_logits", "lfc", "dp_logits"]

# --------------------------------------------------
# 1. Stratified paired bootstrap
# --------------------------------------------------

results_b7 = []

for output_name in outputs_b7:

    frame = (
        B6_comparison[
            B6_comparison["output"] == output_name
        ]
        .sort_values("dataset_index")
        .reset_index(drop=True)
    )

    assert len(frame) == 32
    assert frame["dataset_index"].nunique() == 32

    target = frame["target_change"].to_numpy(
        dtype=np.float64
    )

    control = frame["random_mean"].to_numpy(
        dtype=np.float64
    )

    differences = target - control

    assert np.isfinite(target).all()
    assert np.isfinite(control).all()
    assert np.all(control > 0)

    strata = []

    for context, expected_count in contexts_b7.items():

        indices = np.flatnonzero(
            frame["context"].to_numpy() == context
        )

        assert len(indices) == expected_count

        strata.append(indices)

    observed_difference = float(
        differences.mean()
    )

    observed_ratio = float(
        target.mean() / control.mean()
    )

    bootstrap_difference = np.empty(
        B7_N_BOOT, dtype=np.float64
    )

    bootstrap_ratio = np.empty(
        B7_N_BOOT, dtype=np.float64
    )

    for b in range(B7_N_BOOT):

        sampled_indices = np.concatenate([
            rng_b7.choice(
                indices,
                size=len(indices),
                replace=True
            )
            for indices in strata
        ])

        sampled_target = target[sampled_indices]
        sampled_control = control[sampled_indices]

        bootstrap_difference[b] = (
            sampled_target - sampled_control
        ).mean()

        bootstrap_ratio[b] = (
            sampled_target.mean()
            / sampled_control.mean()
        )

    # Leave one validation condition out at a time.
    loo_differences = np.asarray([
        np.delete(differences, i).mean()
        for i in range(len(differences))
    ])

    results_b7.append({
        "output": output_name,
        "observed_difference": observed_difference,
        "bootstrap_low": float(
            np.percentile(
                bootstrap_difference, 2.5
            )
        ),
        "bootstrap_high": float(
            np.percentile(
                bootstrap_difference, 97.5
            )
        ),
        "observed_ratio": observed_ratio,
        "ratio_low": float(
            np.percentile(
                bootstrap_ratio, 2.5
            )
        ),
        "ratio_high": float(
            np.percentile(
                bootstrap_ratio, 97.5
            )
        ),
        "loo_min_difference": float(
            loo_differences.min()
        ),
        "loo_max_difference": float(
            loo_differences.max()
        ),
        "conditions_above_control": int(
            (differences > 0).sum()
        ),
    })

B7_uncertainty = pd.DataFrame(results_b7)

print("\n=== STRATIFIED BOOTSTRAP ===")

print(
    B7_uncertainty[
        [
            "output",
            "observed_difference",
            "bootstrap_low",
            "bootstrap_high",
            "observed_ratio",
            "ratio_low",
            "ratio_high",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)

# --------------------------------------------------
# 2. Leave-one-condition-out sensitivity
# --------------------------------------------------

print("\n=== LEAVE-ONE-CONDITION-OUT ===")

print(
    B7_uncertainty[
        [
            "output",
            "loo_min_difference",
            "loo_max_difference",
            "conditions_above_control",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)

# --------------------------------------------------
# 3. Sensitivity to individual random directions
# --------------------------------------------------

direction_means_b7 = (
    B6_controls
    .groupby(["output", "intervention"])[
        "mean_abs_change"
    ]
    .mean()
    .reset_index()
)

direction_rows_b7 = []

for output_name in outputs_b7:

    target_mean = float(
        B6_comparison.loc[
            B6_comparison["output"] == output_name,
            "target_change"
        ].mean()
    )

    controls = (
        direction_means_b7.loc[
            direction_means_b7["output"] == output_name,
            "mean_abs_change"
        ].to_numpy()
    )

    assert len(controls) == 5

    direction_rows_b7.append({
        "output": output_name,
        "target_mean": target_mean,
        "min_control_mean": float(controls.min()),
        "max_control_mean": float(controls.max()),
        "target_exceeds_directions": int(
            (target_mean > controls).sum()
        ),
        "control_directions": len(controls),
    })

B7_direction_sensitivity = pd.DataFrame(
    direction_rows_b7
)

print("\n=== CONTROL-DIRECTION SENSITIVITY ===")

print(
    B7_direction_sensitivity.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)

# --------------------------------------------------
# 4. Final checks
# --------------------------------------------------

assert len(B7_uncertainty) == 3

assert np.isfinite(
    B7_uncertainty.select_dtypes(
        include=[np.number]
    ).to_numpy()
).all()

assert B7_uncertainty[
    "conditions_above_control"
].between(0, 32).all()

print("\nBootstrap replicates:", B7_N_BOOT)
print("Validation conditions:", 32)
print("Independent biological replication: NOT ESTABLISHED")
print("Original research artifacts unchanged.")
print("Read-only uncertainty analysis completed.")

=== B.7. CONDITION-LEVEL UNCERTAINTY ===

=== STRATIFIED BOOTSTRAP ===
   output  observed_difference  bootstrap_low  bootstrap_high  observed_ratio  ratio_low  ratio_high
de_logits           0.00016386     0.00000056      0.00034105      1.20607949 1.00066329  1.44258638
      lfc           0.00027225    -0.00004987      0.00060013      1.29983472 0.95233407  1.74627040
dp_logits           0.00047319     0.00016951      0.00081902      1.35849034 1.13081731  1.61132625

=== LEAVE-ONE-CONDITION-OUT ===
   output  loo_min_difference  loo_max_difference  conditions_above_control
de_logits          0.00011987          0.00018853                        17
      lfc          0.00018735          0.00035902                        19
dp_logits          0.00036211          0.00051179                        18

=== CONTROL-DIRECTION SENSITIVITY ===
   output  target_mean  min_control_mean  max_control_mean  target_exceeds_directions  control_directions
de_logits   0.00095899        0.00060822   

### B.8. Comparative Ablation of Three SAE Features

The full-validation analysis established the computational
effect of decoder.0 feature 189 and compared it against
five norm-matched random directions.

This section compares three previously selected SAE
features: 0, 189, and 2120.

Each feature is ablated using its own SAE activation
and its own decoder direction:

h_modified = h - z_j * w_dec_j

The analysis covers all 32 saved validation conditions.

Feature 189 results from Section B.5 are reused.
Only features 0 and 2120 require additional
intervention forward passes.

The comparison reports:

- Number of feature-active gene positions.
- Mean absolute change in PIE predictions.
- Prediction change per active gene position.
- Variation across cellular contexts.

The reported `mean_change_per_active` is the arithmetic mean
of condition-specific ratios calculated where the feature has
positive activity. Conditions with zero active positions yield
undefined ratios and are excluded by the group aggregation.
It is not the ratio of total effects to total active positions,
and comparisons among features must account for this selection.

Because the features differ in firing frequency,
activation magnitude, and decoder direction norm,
raw intervention magnitudes are not directly
comparable measures of biological importance.

This comparison is exploratory. It does not
establish feature specificity or biological causality.

All interventions are performed in memory.
Original model parameters and research files
remain unchanged.

In [17]:
import numpy as np
import pandas as pd
import torch

print("=== B.8. COMPARATIVE SAE FEATURE ABLATION ===")

B8_FEATURES = [0, 189, 2120]
B8_NEW_FEATURES = [0, 2120]
B8_OUTPUTS = ("de_logits", "lfc", "dp_logits")

assert model_07.training is False
assert sae_07.training is False
assert W_dec_07.shape == (3072, 768)
assert len(validation_indices_b5) == 32
assert len(B5_results) == 32 * 3

# --------------------------------------------------
# 1. Reusable single-feature intervention
# --------------------------------------------------

@torch.inference_mode()
def run_feature_ablation_b8(batch_ready, feature_id):

    assert 0 <= feature_id < 3072

    direction = W_dec_07[feature_id].detach()

    direction_norm = float(
        torch.linalg.vector_norm(direction).item()
    )

    state = {
        "calls": 0,
        "positions": 0,
        "active_positions": 0,
        "activation_sum": 0.0,
        "displacement_l2_sum": 0.0,
    }

    def hook(module, inputs, output):

        hidden = output.reshape(-1, 768)

        z, _ = sae_07.encode(
            hidden - mean_07
        )

        activation = z[:, feature_id]

        displacement = (
            activation[:, None]
            * direction[None, :]
        )

        state["calls"] += 1
        state["positions"] += hidden.shape[0]
        state["active_positions"] += int(
            (activation > 0).sum().item()
        )

        state["activation_sum"] += float(
            activation.sum().item()
        )

        state["displacement_l2_sum"] += float(
            torch.linalg.vector_norm(
                displacement, dim=1
            ).sum().item()
        )

        modified = hidden - displacement

        assert torch.isfinite(modified).all()

        return modified.reshape(output.shape)

    handle = model_07.decoder[0].register_forward_hook(
        hook
    )

    try:
        outputs = model_07(batch_ready)[0]
    finally:
        handle.remove()

    assert state["calls"] == expected_calls_07
    assert state["positions"] == 6642

    predictions = {
        name: getattr(outputs, name).detach().cpu().clone()
        for name in B8_OUTPUTS
    }

    state["direction_norm"] = direction_norm

    return predictions, state


# --------------------------------------------------
# 2. Evaluate features 0 and 2120
# --------------------------------------------------

records_b8 = []

for position, dataset_index in enumerate(
    validation_indices_b5
):

    dataset_index = int(dataset_index)

    row = dm_07.val_dataset.rows[dataset_index]
    context = str(row.context).strip().lower()

    sample = dm_07.val_dataset[dataset_index]
    batch = collate_fn_07([sample])

    batch_ready = _batch_to(
        batch,
        DEVICE_07,
        model_07.latents.dtype
    )

    with torch.inference_mode():
        baseline_output = model_07(batch_ready)[0]

    baseline = {
        name: getattr(
            baseline_output, name
        ).detach().cpu().clone()
        for name in B8_OUTPUTS
    }

    for feature_id in B8_NEW_FEATURES:

        modified, state = run_feature_ablation_b8(
            batch_ready,
            feature_id
        )

        n_active = state["active_positions"]

        for name in B8_OUTPUTS:

            assert modified[name].shape == baseline[name].shape

            per_gene = (
                (modified[name] - baseline[name])
                .abs()
                .reshape(6642, -1)
                .mean(dim=1)
            )

            total_effect = float(
                per_gene.sum().item()
            )

            records_b8.append({
                "feature": feature_id,
                "context": context,
                "dataset_index": dataset_index,
                "output": name,
                "active_positions": n_active,
                "mean_abs_change": float(
                    per_gene.mean().item()
                ),
                "max_gene_mean_abs_change": float(
                    per_gene.max().item()
                ),
                "changed_gene_positions": int(
                    (per_gene > 1e-6).sum().item()
                ),
                "total_change_per_active_position": (
                    total_effect / n_active
                    if n_active > 0 else np.nan
                ),
                "direction_norm": state["direction_norm"],
                "mean_displacement_l2_per_active": (
                    state["displacement_l2_sum"] / n_active
                    if n_active > 0 else np.nan
                ),
            })

    if (position + 1) % 4 == 0:
        print(
            f"Completed {position + 1}/32 "
            "validation conditions"
        )

B8_new_results = pd.DataFrame(records_b8)

assert len(B8_new_results) == 32 * 2 * 3

# --------------------------------------------------
# 3. Combine with previous feature 189 results
# --------------------------------------------------

feature189_b8 = B5_results.copy()
feature189_b8["feature"] = 189

B8_results = pd.concat(
    [feature189_b8, B8_new_results],
    ignore_index=True
)

assert len(B8_results) == 32 * 3 * 3

assert not B8_results.duplicated(
    ["feature", "dataset_index", "output"]
).any()

assert np.isfinite(
    B8_results["mean_abs_change"].to_numpy()
).all()

# --------------------------------------------------
# 4. Summarize the feature comparison
# --------------------------------------------------

B8_summary = (
    B8_results
    .groupby(["feature", "output"])
    .agg(
        conditions=("dataset_index", "count"),
        mean_active_positions=(
            "active_positions", "mean"
        ),
        mean_abs_change=(
            "mean_abs_change", "mean"
        ),
        median_abs_change=(
            "mean_abs_change", "median"
        ),
        mean_change_per_active=(
            "total_change_per_active_position",
            "mean"
        ),
        conditions_with_effect=(
            "changed_gene_positions",
            lambda x: int((x > 0).sum())
        ),
    )
    .reset_index()
)

print("\n=== THREE-FEATURE COMPARISON ===")

print(
    B8_summary.round(7).to_string(index=False)
)

# --------------------------------------------------
# 5. Compare DE-logit effects across contexts
# --------------------------------------------------

B8_de_context = (
    B8_results[
        B8_results["output"] == "de_logits"
    ]
    .groupby(["feature", "context"])
    .agg(
        conditions=("dataset_index", "count"),
        mean_active_positions=(
            "active_positions", "mean"
        ),
        mean_abs_change=(
            "mean_abs_change", "mean"
        ),
        mean_change_per_active=(
            "total_change_per_active_position",
            "mean"
        ),
    )
    .reset_index()
)

print("\n=== DE-LOGIT EFFECTS BY CONTEXT ===")

print(
    B8_de_context.round(7).to_string(index=False)
)

# --------------------------------------------------
# 6. Check the original feature 189 results
# --------------------------------------------------

original_189 = B8_results[
    B8_results["feature"] == 189
]

assert len(original_189) == 32 * 3

print("\n=== INTEGRITY ===")
print("Features evaluated:", B8_FEATURES)
print("Validation conditions:", 32)
print("New feature interventions:", 32 * 2)
print("Feature 189 results reused: PASS")
print("All intervention hooks removed.")
print("No checkpoints or research files modified.")

=== B.8. COMPARATIVE SAE FEATURE ABLATION ===
Completed 4/32 validation conditions
Completed 8/32 validation conditions
Completed 12/32 validation conditions
Completed 16/32 validation conditions
Completed 20/32 validation conditions
Completed 24/32 validation conditions
Completed 28/32 validation conditions
Completed 32/32 validation conditions

=== THREE-FEATURE COMPARISON ===
 feature    output  conditions  mean_active_positions  mean_abs_change  median_abs_change  mean_change_per_active  conditions_with_effect
       0 de_logits          32               42.68750         0.000784           0.000724                0.130158                      32
       0 dp_logits          32               42.68750         0.000963           0.000878                0.160041                      32
       0       lfc          32               42.68750         0.000652           0.000407                0.108209                      32
     189 de_logits          32               71.53125         0.00

### B.8.1. Condition-Level Heterogeneity of Feature Ablation

The full-validation comparison revealed substantial
differences among features 0, 189, and 2120.

Feature 2120 produced the largest average prediction
changes but showed detectable effects in only
19 of 32 validation conditions.

This analysis evaluates how intervention effects
are distributed across conditions and cell contexts.

The analysis reports:

- Mean and median DE-logit changes by cell context.
- Numbers of conditions with active SAE features.
- Numbers of conditions with detectable output effects.
- The contribution of the strongest three conditions
  to the total observed effect.
- Conditions producing the largest changes.

A detectable effect is defined using the existing
threshold of 1e-6 on per-gene mean absolute changes.

This descriptive analysis does not establish statistical
significance, feature specificity, or biological causality.

No additional model inference is performed.

In [18]:
import numpy as np
import pandas as pd

print("=== B.8.1. CONDITION-LEVEL HETEROGENEITY ===")

# --------------------------------------------------
# 1. Select DE-logit effects from the existing results
# --------------------------------------------------

de_b81 = B8_results.loc[
    B8_results["output"].eq("de_logits")
].copy()

assert len(de_b81) == 32 * 3
assert set(de_b81["feature"]) == {0, 189, 2120}

de_b81["has_activity"] = (
    de_b81["active_positions"] > 0
)

de_b81["detectable_effect"] = (
    de_b81["changed_gene_positions"] > 0
)

# --------------------------------------------------
# 2. Summarize effects by cell context
# --------------------------------------------------

context_b81 = (
    de_b81
    .groupby(["feature", "context"])
    .agg(
        conditions=("dataset_index", "count"),
        mean_active=("active_positions", "mean"),
        mean_effect=("mean_abs_change", "mean"),
        median_effect=("mean_abs_change", "median"),
        max_effect=("mean_abs_change", "max"),
        conditions_active=("has_activity", "sum"),
        conditions_with_effect=("detectable_effect", "sum"),
    )
    .reset_index()
)

print("\n=== EFFECTS BY CELL CONTEXT ===")

print(
    context_b81.to_string(
        index=False,
        float_format=lambda x: f"{x:.7f}"
    )
)

# --------------------------------------------------
# 3. Check concentration in the strongest conditions
# --------------------------------------------------

concentration_rows_b81 = []

for fid, frame in de_b81.groupby("feature"):

    effects = np.sort(
        frame["mean_abs_change"].to_numpy(
            dtype=np.float64
        )
    )[::-1]

    total = float(effects.sum())

    concentration_rows_b81.append({
        "feature": int(fid),
        "mean_effect": float(effects.mean()),
        "median_effect": float(np.median(effects)),
        "top3_effect_share": (
            float(effects[:3].sum() / total)
            if total > 0 else np.nan
        ),
        "conditions_active": int(
            frame["has_activity"].sum()
        ),
        "conditions_with_effect": int(
            frame["detectable_effect"].sum()
        ),
    })

B81_concentration = pd.DataFrame(
    concentration_rows_b81
)

print("\n=== EFFECT CONCENTRATION ===")

print(
    B81_concentration.to_string(
        index=False,
        float_format=lambda x: f"{x:.7f}"
    )
)

# --------------------------------------------------
# 4. Inspect the strongest conditions for each feature
# --------------------------------------------------

print("\n=== TOP THREE CONDITIONS PER FEATURE ===")

strongest_b81 = (
    de_b81
    .sort_values(
        ["feature", "mean_abs_change"],
        ascending=[True, False]
    )
    .groupby("feature")
    .head(3)
)

print(
    strongest_b81[
        [
            "feature",
            "context",
            "dataset_index",
            "active_positions",
            "mean_abs_change",
            "changed_gene_positions",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.7f}"
    )
)

# --------------------------------------------------
# 5. Investigate conditions without detectable effects
# --------------------------------------------------

print("\n=== FEATURE 2120: NO DETECTABLE EFFECT ===")

no_effect_b81 = de_b81.loc[
    de_b81["feature"].eq(2120)
    & ~de_b81["detectable_effect"]
]

print("Conditions:", len(no_effect_b81))

print(
    no_effect_b81[
        [
            "context",
            "dataset_index",
            "active_positions",
            "mean_abs_change",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.9f}"
    )
)

print("\nRead-only heterogeneity analysis completed.")

=== B.8.1. CONDITION-LEVEL HETEROGENEITY ===

=== EFFECTS BY CELL CONTEXT ===
 feature context  conditions  mean_active  mean_effect  median_effect  max_effect  conditions_active  conditions_with_effect
       0   hepg2          10   53.8000000    0.0008128      0.0007433   0.0011896                 10                      10
       0  jurkat          13   38.0769231    0.0008226      0.0007628   0.0012973                 13                      13
       0    rpe1           9   37.0000000    0.0006948      0.0006906   0.0008991                  9                       9
     189   hepg2          10   67.9000000    0.0008569      0.0008410   0.0017803                 10                      10
     189  jurkat          13   70.1538462    0.0010486      0.0008361   0.0021285                 13                      13
     189    rpe1           9   77.5555556    0.0009430      0.0008628   0.0014628                  9                       9
    2120   hepg2          10 1020.4000000    0.

### B.9. Learned SAE-Dictionary Direction Controls

The previous experiments established nonzero computational
effects for three decoder.0 SAE features.

Feature 189 showed consistent effects across validation
conditions, whereas feature 2120 exhibited strongly
concentrated, condition-dependent effects.

The current experiment tests whether the decoder direction
of feature 189 produces larger prediction changes than
directions learned by other SAE features.

Six validation conditions are reused, with two from each
cell context.

Sixteen control directions are sampled without replacement
from the same decoder.0 SAE dictionary, excluding the three
diagnostic features.

All control directions are rescaled to match the Euclidean
norm of the feature 189 decoder direction.

For every control intervention, the feature 189 activation
coefficient and firing positions are retained:

h_modified = h - z_189 * w_control

This isolates the effect of changing the intervention
direction while holding its activation gate and displacement
norms fixed.

The experiment is exploratory. The controls are sampled
from a fixed trained dictionary and do not constitute
independently trained or biologically validated features.

No intervention claims about the causal specificity of
feature 189 are made from this pilot alone.

All model parameters and existing research artifacts
remain unchanged.

In [19]:
import numpy as np
import pandas as pd
import torch

print("=== B.9. LEARNED SAE-DICTIONARY CONTROLS ===")

B9_SEED = 20261117
B9_N_CONTROLS = 16
B9_OUTPUTS = ("de_logits", "lfc", "dp_logits")

assert W_dec_07.shape == (3072, 768)
assert len(selected_conditions) == 6
assert len(B5_results) == 32 * 3
assert FEATURE_07 == 189
assert model_07.training is False
assert sae_07.training is False

# --------------------------------------------------
# 1. Select control features from the SAE dictionary
# --------------------------------------------------

rng_b9 = np.random.default_rng(B9_SEED)

excluded_b9 = {0, 189, 2120}

candidate_ids_b9 = np.array(
    [i for i in range(3072) if i not in excluded_b9],
    dtype=np.int32
)

control_ids_b9 = rng_b9.choice(
    candidate_ids_b9,
    size=B9_N_CONTROLS,
    replace=False
)

target_direction_b9 = (
    W_dec_07[189].detach().cpu().float()
)

target_norm_b9 = torch.linalg.vector_norm(
    target_direction_b9
)

control_directions_b9 = {}
direction_records_b9 = []

for feature_id in control_ids_b9:

    feature_id = int(feature_id)

    raw_direction = (
        W_dec_07[feature_id].detach().cpu().float()
    )

    raw_norm = torch.linalg.vector_norm(raw_direction)

    assert raw_norm > 0

    normalized_direction = (
        raw_direction / raw_norm * target_norm_b9
    )

    cosine = float(
        torch.dot(
            raw_direction / raw_norm,
            target_direction_b9 / target_norm_b9
        )
    )

    control_directions_b9[feature_id] = normalized_direction

    direction_records_b9.append({
        "feature": feature_id,
        "original_norm": float(raw_norm),
        "cosine_to_189": cosine,
    })

B9_directions = pd.DataFrame(direction_records_b9)

print("\n=== CONTROL DICTIONARY ===")

print("Control features:", len(control_directions_b9))
print("Target feature:", 189)
print("Target direction norm:", float(target_norm_b9))
print("Sampled IDs:", sorted(control_directions_b9))

print(
    "Cosine range:",
    f"[{B9_directions['cosine_to_189'].min():.4f}, "
    f"{B9_directions['cosine_to_189'].max():.4f}]"
)

# --------------------------------------------------
# 2. Evaluate the six previously selected conditions
# --------------------------------------------------

records_b9 = []

for context, h5_position, dataset_index in selected_conditions:

    sample = dm_07.val_dataset[int(dataset_index)]

    batch = collate_fn_07([sample])

    batch_ready = _batch_to(
        batch,
        DEVICE_07,
        model_07.latents.dtype
    )

    with torch.inference_mode():
        baseline_output = model_07(batch_ready)[0]

    baseline = {
        name: getattr(baseline_output, name)
        .detach().cpu().clone()
        for name in B9_OUTPUTS
    }

    expected_active = int(
        B5_results.loc[
            B5_results["dataset_index"].eq(dataset_index)
            & B5_results["output"].eq("de_logits"),
            "active_positions"
        ].iloc[0]
    )

    for control_id, direction in control_directions_b9.items():

        modified, state = run_b3_intervention(
            batch_ready,
            direction
        )

        assert state["calls"] == expected_calls_07
        assert state["active_positions"] == expected_active

        for name in B9_OUTPUTS:

            assert modified[name].shape == baseline[name].shape

            per_gene_change = (
                (modified[name] - baseline[name])
                .abs()
                .reshape(6642, -1)
                .mean(dim=1)
            )

            records_b9.append({
                "context": context,
                "dataset_index": int(dataset_index),
                "control_feature": int(control_id),
                "output": name,
                "active_positions": expected_active,
                "mean_abs_change": float(
                    per_gene_change.mean().item()
                ),
            })

    print(
        f"Completed: {context}, "
        f"dataset index {dataset_index}"
    )

B9_controls = pd.DataFrame(records_b9)

assert len(B9_controls) == 6 * B9_N_CONTROLS * 3

assert np.isfinite(
    B9_controls["mean_abs_change"].to_numpy()
).all()

# --------------------------------------------------
# 3. Compare against existing feature 189 effects
# --------------------------------------------------

selected_ids_b9 = [
    int(index)
    for _, _, index in selected_conditions
]

target_b9 = (
    B5_results.loc[
        B5_results["dataset_index"].isin(
            selected_ids_b9
        ),
        [
            "context",
            "dataset_index",
            "output",
            "mean_abs_change",
        ]
    ]
    .rename(columns={
        "mean_abs_change": "target_change"
    })
)

assert len(target_b9) == 6 * 3

control_summary_b9 = (
    B9_controls
    .groupby(
        ["context", "dataset_index", "output"],
        as_index=False
    )
    .agg(
        control_mean=("mean_abs_change", "mean"),
        control_median=("mean_abs_change", "median"),
        control_max=("mean_abs_change", "max"),
    )
)

B9_comparison = target_b9.merge(
    control_summary_b9,
    on=["context", "dataset_index", "output"],
    validate="one_to_one"
)

B9_comparison["target_to_control_ratio"] = (
    B9_comparison["target_change"]
    / B9_comparison["control_mean"].replace(0, np.nan)
)

B9_comparison["above_control_mean"] = (
    B9_comparison["target_change"]
    > B9_comparison["control_mean"]
)

B9_comparison["above_all_controls"] = (
    B9_comparison["target_change"]
    > B9_comparison["control_max"]
)

assert len(B9_comparison) == 18

print("\n=== PER-CONDITION COMPARISON ===")

print(
    B9_comparison[
        [
            "context",
            "dataset_index",
            "output",
            "target_change",
            "control_mean",
            "target_to_control_ratio",
            "above_control_mean",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# --------------------------------------------------
# 4. Summarize all six conditions
# --------------------------------------------------

B9_summary = (
    B9_comparison
    .groupby("output")
    .agg(
        conditions=("dataset_index", "count"),
        mean_target=("target_change", "mean"),
        mean_control=("control_mean", "mean"),
        above_control_mean=("above_control_mean", "sum"),
        above_all_controls=("above_all_controls", "sum"),
    )
)

B9_summary["ratio_of_means"] = (
    B9_summary["mean_target"]
    / B9_summary["mean_control"]
)

print("\n=== SAE-DICTIONARY CONTROL SUMMARY ===")

print(
    B9_summary.to_string(
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== INTEGRITY ===")
print("Conditions:", len(selected_conditions))
print("Control directions:", B9_N_CONTROLS)
print("Control forward passes:", 6 * B9_N_CONTROLS)
print("Target results reused from B.5: PASS")
print("Original checkpoints unchanged.")
print("All intervention hooks removed.")

=== B.9. LEARNED SAE-DICTIONARY CONTROLS ===

=== CONTROL DICTIONARY ===
Control features: 16
Target feature: 189
Target direction norm: 0.5036172270774841
Sampled IDs: [205, 525, 594, 722, 780, 1420, 1436, 1958, 2056, 2093, 2161, 2210, 2427, 2453, 2569, 2954]
Cosine range: [-0.0675, 0.0907]
Completed: hepg2, dataset index 1
Completed: hepg2, dataset index 5
Completed: jurkat, dataset index 103
Completed: jurkat, dataset index 104
Completed: rpe1, dataset index 204
Completed: rpe1, dataset index 221

=== PER-CONDITION COMPARISON ===
context  dataset_index    output  target_change  control_mean  target_to_control_ratio  above_control_mean
  hepg2              1 de_logits       0.000929      0.000825                 1.126752                True
  hepg2              1       lfc       0.000849      0.000520                 1.631185                True
  hepg2              1 dp_logits       0.002426      0.001494                 1.624080                True
  hepg2              5 de_logits 

In [20]:
print("=== B.9. FINAL DICTIONARY-CONTROL SUMMARY ===")

print(
    B9_summary.reset_index().to_string(
        index=False,
        float_format=lambda x: f"{x:.7f}"
    )
)

print("\n=== INTERPRETATION COUNTS ===")
for output_name, row in B9_summary.iterrows():
    print(
        f"{output_name}: "
        f"above control mean in {int(row['above_control_mean'])}/6; "
        f"above all controls in {int(row['above_all_controls'])}/6"
    )

=== B.9. FINAL DICTIONARY-CONTROL SUMMARY ===
   output  conditions  mean_target  mean_control  above_control_mean  above_all_controls  ratio_of_means
de_logits           6    0.0010041     0.0009394                   3                   1       1.0688600
dp_logits           6    0.0016224     0.0014453                   2                   2       1.1225260
      lfc           6    0.0013322     0.0010232                   2                   1       1.3019944

=== INTERPRETATION COUNTS ===
de_logits: above control mean in 3/6; above all controls in 1/6
dp_logits: above control mean in 2/6; above all controls in 2/6
lfc: above control mean in 2/6; above all controls in 1/6


## C. Feature Activation Patching

### C.1. Within-Context Feature 189 Patching

This experiment transfers the activation coefficient of
decoder.0 SAE feature 189 between two validation conditions
from the same cell context.

The donor and recipient are evaluated using the same
pretrained PIE model and identical output-gene ordering.

For each gene position, the feature-specific intervention is:

h_patched = h_recipient
          + (z_donor_189 - z_recipient_189) * w_dec_189

The original hidden representation is retained apart from
this feature-direction displacement.

The experiment verifies:

- Matching output-gene identifiers.
- Reproduction of the original recipient baseline.
- An identity patch using the recipient's own activations.
- The magnitude and localization of the actual patch.
- Changes in PIE predictions and distance to donor outputs.

A reduction in output distance to the donor is an
exploratory computational result, not proof of
biological transfer of a perturbation mechanism.

The intervention modifies only in-memory activations.
No pretrained weights or existing research files
are changed.

In [21]:
import numpy as np
import torch
import pandas as pd

print("=== C.1. WITHIN-CONTEXT ACTIVATION PATCHING ===")

C1_FEATURE = 189
C1_RECIPIENT = 1
C1_DONOR = 5
C1_OUTPUTS = ("de_logits", "lfc", "dp_logits")

assert model_07.training is False
assert sae_07.training is False
assert DEVICE_07.type == "cpu"

# --------------------------------------------------
# 1. Prepare the two validation inputs
# --------------------------------------------------

def prepare_condition_c1(dataset_index):
    sample = dm_07.val_dataset[dataset_index]
    batch = collate_fn_07([sample])

    assert len(batch.groups) == 1

    gene_ids = (
        batch.groups[0]
        .gene_ids.detach().cpu().numpy()
    )

    ready = _batch_to(
        batch,
        DEVICE_07,
        model_07.latents.dtype
    )

    return ready, gene_ids


recipient_batch_c1, recipient_genes_c1 = (
    prepare_condition_c1(C1_RECIPIENT)
)

donor_batch_c1, donor_genes_c1 = (
    prepare_condition_c1(C1_DONOR)
)

recipient_context_c1 = str(
    dm_07.val_dataset.rows[C1_RECIPIENT].context
).lower()

donor_context_c1 = str(
    dm_07.val_dataset.rows[C1_DONOR].context
).lower()

assert recipient_context_c1 == donor_context_c1
assert recipient_context_c1 == "hepg2"

assert np.array_equal(
    recipient_genes_c1,
    donor_genes_c1
)

assert np.array_equal(
    recipient_genes_c1,
    current_gene_ids_07
)

print("\n=== CONDITIONS ===")
print("Recipient:", C1_RECIPIENT)
print("Donor:", C1_DONOR)
print("Cell context:", recipient_context_c1)
print("Gene alignment: PASS")

# --------------------------------------------------
# 2. Capture baseline decoder representations
# --------------------------------------------------

@torch.inference_mode()
def capture_condition_c1(batch_ready):

    chunks = []

    def hook(module, inputs, output):
        assert isinstance(output, torch.Tensor)
        chunks.append(output.detach().cpu().clone())

    handle = model_07.decoder[0].register_forward_hook(
        hook
    )

    try:
        result = model_07(batch_ready)[0]
    finally:
        handle.remove()

    assert len(chunks) == expected_calls_07

    predictions = {
        name: getattr(result, name).detach().cpu().clone()
        for name in C1_OUTPUTS
    }

    return predictions, chunks


recipient_output_c1, recipient_chunks_c1 = (
    capture_condition_c1(recipient_batch_c1)
)

donor_output_c1, donor_chunks_c1 = (
    capture_condition_c1(donor_batch_c1)
)

for name in C1_OUTPUTS:
    assert torch.allclose(
        recipient_output_c1[name],
        baseline_predictions_07[name],
        rtol=1e-6,
        atol=1e-6
    )

print("Original recipient baseline reproduction: PASS")

# --------------------------------------------------
# 3. Extract feature-189 coefficient per gene
# --------------------------------------------------

@torch.inference_mode()
def feature_coefficients_c1(chunks):

    coefficients = []

    for chunk in chunks:
        hidden = chunk.reshape(-1, 768)

        z, _ = sae_07.encode(
            hidden - mean_07
        )

        coefficients.append(
            z[:, C1_FEATURE].detach().cpu().clone()
        )

    result = torch.cat(coefficients)

    assert result.shape == (6642,)
    assert torch.isfinite(result).all()

    return result


z_recipient_c1 = feature_coefficients_c1(
    recipient_chunks_c1
)

z_donor_c1 = feature_coefficients_c1(
    donor_chunks_c1
)

delta_z_c1 = z_donor_c1 - z_recipient_c1

print("\n=== FEATURE COEFFICIENTS ===")
print(
    "Active recipient positions:",
    int((z_recipient_c1 > 0).sum())
)
print(
    "Active donor positions:",
    int((z_donor_c1 > 0).sum())
)
print(
    "Positions with coefficient differences:",
    int((delta_z_c1.abs() > 1e-6).sum())
)
print(
    "Maximum coefficient difference:",
    float(delta_z_c1.abs().max())
)

# --------------------------------------------------
# 4. Reusable patch applied inside PIE forward
# --------------------------------------------------

@torch.inference_mode()
def run_patch_c1(source_coefficients):

    state = {"calls": 0, "offset": 0}

    direction = direction_189_07.detach()

    def patch_hook(module, inputs, output):

        call_index = state["calls"]

        # Check that the unmodified recipient hidden
        # representation is reproduced.
        assert torch.allclose(
            output,
            recipient_chunks_c1[call_index],
            rtol=1e-6,
            atol=1e-6
        )

        hidden = output.reshape(-1, 768)

        start = state["offset"]
        end = start + hidden.shape[0]

        replacement = source_coefficients[start:end]
        original = z_recipient_c1[start:end]

        displacement = (
            (replacement - original)[:, None]
            * direction[None, :]
        )

        state["offset"] = end
        state["calls"] += 1

        return (
            hidden + displacement
        ).reshape(output.shape)

    handle = model_07.decoder[0].register_forward_hook(
        patch_hook
    )

    try:
        result = model_07(recipient_batch_c1)[0]
    finally:
        handle.remove()

    assert state["calls"] == expected_calls_07
    assert state["offset"] == 6642

    return {
        name: getattr(result, name).detach().cpu().clone()
        for name in C1_OUTPUTS
    }


# --------------------------------------------------
# 5. Identity patch: recipient -> recipient
# --------------------------------------------------

identity_patch_c1 = run_patch_c1(z_recipient_c1)

for name in C1_OUTPUTS:
    assert torch.allclose(
        identity_patch_c1[name],
        recipient_output_c1[name],
        rtol=1e-6,
        atol=1e-6
    )

print("\nIdentity patch: PASS")

# --------------------------------------------------
# 6. Actual donor -> recipient patch
# --------------------------------------------------

patched_output_c1 = run_patch_c1(z_donor_c1)

rows_c1 = []

for name in C1_OUTPUTS:

    recipient = recipient_output_c1[name]
    donor = donor_output_c1[name]
    patched = patched_output_c1[name]

    assert recipient.shape == donor.shape == patched.shape

    intervention_change = (
        patched - recipient
    ).abs().mean().item()

    distance_before = (
        recipient - donor
    ).abs().mean().item()

    distance_after = (
        patched - donor
    ).abs().mean().item()

    rows_c1.append({
        "output": name,
        "mean_patch_effect": intervention_change,
        "donor_distance_before": distance_before,
        "donor_distance_after": distance_after,
        "distance_reduction_fraction": (
            (distance_before - distance_after)
            / distance_before
            if distance_before > 0
            else np.nan
        ),
    })

C1_results = pd.DataFrame(rows_c1)

print("\n=== PATCHING RESULTS ===")

print(
    C1_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)

print("\n=== INTEGRITY ===")
print("Baseline reproduction: PASS")
print("Identity patch: PASS")
print("Donor-to-recipient patch: completed")
print("All hooks removed.")
print("Original research artifacts unchanged.")

=== C.1. WITHIN-CONTEXT ACTIVATION PATCHING ===

=== CONDITIONS ===
Recipient: 1
Donor: 5
Cell context: hepg2
Gene alignment: PASS
Original recipient baseline reproduction: PASS

=== FEATURE COEFFICIENTS ===
Active recipient positions: 68
Active donor positions: 66
Positions with coefficient differences: 68
Maximum coefficient difference: 2.275125503540039

Identity patch: PASS

=== PATCHING RESULTS ===
   output  mean_patch_effect  donor_distance_before  donor_distance_after  distance_reduction_fraction
de_logits         0.00004678             0.74902576            0.74903351                  -0.00001034
      lfc         0.00003611             0.92231357            0.92231768                  -0.00000446
dp_logits         0.00013701             1.82383788            1.82385707                  -0.00001052

=== INTEGRITY ===
Baseline reproduction: PASS
Identity patch: PASS
Donor-to-recipient patch: completed
All hooks removed.
Original research artifacts unchanged.


### C.2. Localized Patching Effects and Donor Alignment

The initial within-context feature 189 patch changed
PIE predictions but did not reduce global output
distance to the donor condition.

However, the intervention affected only gene positions
where the donor and recipient SAE coefficients differed.

Global distance can therefore obscure local effects.

This analysis evaluates donor alignment specifically
on the positions modified by the activation patch.

For each PIE output, the analysis measures:

- Mean absolute patch effect on modified positions.
- Donor distance before and after patching.
- Fractional reduction in local donor distance.
- Cosine alignment between the patch-induced change
  and the recipient-to-donor prediction difference.
- Number of modified gene positions that became
  closer to the donor.

The analysis also verifies that positions with
unchanged SAE coefficients remain unaffected.

These results describe the effect of the specified
feature-direction patch. They do not establish
biological transfer, feature specificity, or
independent causal mechanisms.

No additional PIE inference is performed.

In [22]:
import numpy as np
import pandas as pd

print("=== C.2. LOCALIZED PATCHING ANALYSIS ===")

C2_OUTPUTS = ("de_logits", "lfc", "dp_logits")

# --------------------------------------------------
# 1. Identify positions modified by the patch
# --------------------------------------------------

patch_mask_c2 = (
    delta_z_c1.detach().cpu().abs().numpy() > 1e-6
)

assert patch_mask_c2.shape == (6642,)

n_patched_c2 = int(patch_mask_c2.sum())

assert n_patched_c2 > 0

print("\n=== PATCHED POSITIONS ===")
print("Modified gene positions:", n_patched_c2)
print("Unmodified gene positions:", 6642 - n_patched_c2)

# --------------------------------------------------
# 2. Compare local effects and donor alignment
# --------------------------------------------------

records_c2 = []
gene_details_c2 = {}

for name in C2_OUTPUTS:

    recipient = (
        recipient_output_c1[name]
        .detach().cpu()
        .reshape(6642, -1)
        .numpy()
    )

    donor = (
        donor_output_c1[name]
        .detach().cpu()
        .reshape(6642, -1)
        .numpy()
    )

    patched = (
        patched_output_c1[name]
        .detach().cpu()
        .reshape(6642, -1)
        .numpy()
    )

    assert recipient.shape == donor.shape == patched.shape

    patch_change = patched - recipient
    donor_difference = donor - recipient

    # Restrict to positions whose SAE coefficients changed.
    change_local = patch_change[patch_mask_c2]
    donor_local = donor_difference[patch_mask_c2]

    recipient_local = recipient[patch_mask_c2]
    patched_local = patched[patch_mask_c2]
    donor_output_local = donor[patch_mask_c2]

    distance_before = float(
        np.mean(np.abs(recipient_local - donor_output_local))
    )

    distance_after = float(
        np.mean(np.abs(patched_local - donor_output_local))
    )

    patch_magnitude = float(
        np.mean(np.abs(change_local))
    )

    change_vector = change_local.ravel().astype(np.float64)
    donor_vector = donor_local.ravel().astype(np.float64)

    change_norm = np.linalg.norm(change_vector)
    donor_norm = np.linalg.norm(donor_vector)

    if change_norm > 0 and donor_norm > 0:

        cosine_alignment = float(
            np.dot(change_vector, donor_vector)
            / (change_norm * donor_norm)
        )

    else:
        cosine_alignment = np.nan

    # Per-gene donor distance, averaged across output channels.
    gene_before = np.mean(
        np.abs(recipient - donor),
        axis=1
    )

    gene_after = np.mean(
        np.abs(patched - donor),
        axis=1
    )

    gene_effect = np.mean(
        np.abs(patched - recipient),
        axis=1
    )

    closer = (
        gene_after[patch_mask_c2]
        < gene_before[patch_mask_c2] - 1e-8
    )

    farther = (
        gene_after[patch_mask_c2]
        > gene_before[patch_mask_c2] + 1e-8
    )

    max_unmodified_change = float(
        np.abs(patch_change[~patch_mask_c2]).max()
    )

    records_c2.append({
        "output": name,
        "local_patch_effect": patch_magnitude,
        "donor_distance_before": distance_before,
        "donor_distance_after": distance_after,
        "local_distance_reduction": (
            1.0 - distance_after / distance_before
            if distance_before > 0 else np.nan
        ),
        "cosine_alignment": cosine_alignment,
        "genes_closer": int(closer.sum()),
        "genes_farther": int(farther.sum()),
        "max_unmodified_change": max_unmodified_change,
    })

    gene_details_c2[name] = pd.DataFrame({
        "gene_position": np.flatnonzero(patch_mask_c2),
        "gene_id": recipient_genes_c1[patch_mask_c2],
        "recipient_coefficient": (
            z_recipient_c1.numpy()[patch_mask_c2]
        ),
        "donor_coefficient": (
            z_donor_c1.numpy()[patch_mask_c2]
        ),
        "patch_effect": gene_effect[patch_mask_c2],
        "donor_distance_before": gene_before[patch_mask_c2],
        "donor_distance_after": gene_after[patch_mask_c2],
    })

C2_results = pd.DataFrame(records_c2)

print("\n=== LOCAL DONOR ALIGNMENT ===")

print(
    C2_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)

# --------------------------------------------------
# 3. Identify strongest local prediction changes
# --------------------------------------------------

print("\n=== TOP PATCHED GENES: DE LOGITS ===")

C2_top_genes = (
    gene_details_c2["de_logits"]
    .sort_values("patch_effect", ascending=False)
    .head(10)
)

print(
    C2_top_genes.to_string(
        index=False,
        float_format=lambda x: f"{x:.7f}"
    )
)

# --------------------------------------------------
# 4. Verify localization
# --------------------------------------------------

print("\n=== LOCALIZATION VERIFICATION ===")

for _, row in C2_results.iterrows():

    print(
        f"{row['output']}: "
        f"max unmodified-position change = "
        f"{row['max_unmodified_change']:.9e}"
    )

assert np.isfinite(
    C2_results["local_patch_effect"]
).all()

print("\nRead-only local patching analysis completed.")
print("No PIE inference or checkpoint modifications.")

=== C.2. LOCALIZED PATCHING ANALYSIS ===

=== PATCHED POSITIONS ===
Modified gene positions: 68
Unmodified gene positions: 6574

=== LOCAL DONOR ALIGNMENT ===
   output  local_patch_effect  donor_distance_before  donor_distance_after  local_distance_reduction  cosine_alignment  genes_closer  genes_farther  max_unmodified_change
de_logits          0.00456896             0.53903097            0.53977758               -0.00138509       -0.27861024            33             35             0.00000000
      lfc          0.00352723             1.25485361            1.25525904               -0.00032309        0.04101548            36             32             0.00000000
dp_logits          0.01338237             1.53906751            1.54094934               -0.00122271       -0.11146705            42             26             0.00000000

=== TOP PATCHED GENES: DE LOGITS ===
 gene_position  gene_id  recipient_coefficient  donor_coefficient  patch_effect  donor_distance_before  donor_distance_

### C.3. Controlled Within-Context Activation Patching

The first activation-patching experiment did not reduce
recipient-to-donor prediction distance, either globally
or on the modified gene positions.

This experiment extends patching to nine directed
recipient–donor pairs, with three pairs sampled from
each validation cell context.

Within each context, conditions are selected without
replacement using a fixed random seed.

For every pair, the feature 189 coefficient difference
is computed between donor and recipient representations.

The following interventions are compared:

1. Patching along the feature 189 decoder direction.
2. Patching along five alternative learned SAE decoder
   directions of matched Euclidean norm.

All interventions use identical coefficient differences
and gene positions.

The primary outcome is the change in recipient-to-donor
mean absolute prediction distance on the positions where
the SAE coefficients differ.

A positive distance improvement indicates that the
patched recipient became closer to the donor.

A negative improvement indicates increased distance.

These are exploratory computational comparisons,
not biological perturbation-transfer experiments.

The experiment does not establish independently
replicated or statistically significant causal mechanisms.

All interventions are performed in memory.

In [23]:
import numpy as np
import pandas as pd
import torch

print("=== C.3. CONTROLLED ACTIVATION PATCHING ===")

C3_SEED = 20261117
C3_PAIRS_PER_CONTEXT = 3
C3_OUTPUTS = ("de_logits", "lfc", "dp_logits")

assert model_07.training is False
assert sae_07.training is False
assert len(validation_indices_b5) == 32
assert len(control_directions_b9) >= 5

# --------------------------------------------------
# 1. Select nine directed pairs
# --------------------------------------------------

rng_c3 = np.random.default_rng(C3_SEED)

pairs_c3 = []

for context in ("hepg2", "jurkat", "rpe1"):

    candidates = [
        int(index)
        for index in validation_indices_b5
        if str(
            dm_07.val_dataset.rows[int(index)].context
        ).strip().lower() == context
    ]

    assert len(candidates) >= 6

    chosen = rng_c3.choice(
        candidates,
        size=6,
        replace=False
    )

    for k in range(C3_PAIRS_PER_CONTEXT):

        recipient_id = int(chosen[2 * k])
        donor_id = int(chosen[2 * k + 1])

        assert recipient_id != donor_id

        pairs_c3.append(
            (context, recipient_id, donor_id)
        )

assert len(pairs_c3) == 9

print("\n=== SELECTED PAIRS ===")

for context, recipient, donor in pairs_c3:
    print(
        f"{context}: recipient={recipient}, donor={donor}"
    )

# --------------------------------------------------
# 2. Prepare fixed control directions
# --------------------------------------------------

control_ids_c3 = sorted(
    control_directions_b9.keys()
)[:5]

directions_c3 = {
    "feature189": direction_189_07.detach().cpu()
}

for fid in control_ids_c3:
    directions_c3[f"sae_control_{fid}"] = (
        control_directions_b9[fid].detach().cpu()
    )

target_norm_c3 = torch.linalg.vector_norm(
    directions_c3["feature189"]
)

for direction in directions_c3.values():

    assert direction.shape == (768,)

    assert torch.allclose(
        torch.linalg.vector_norm(direction),
        target_norm_c3,
        rtol=1e-5,
        atol=1e-6
    )

print("\nControl SAE directions:", len(control_ids_c3))
print("Direction normalization: PASS")

# --------------------------------------------------
# 3. Reusable patching function
# --------------------------------------------------

@torch.inference_mode()
def run_patch_c3(
    recipient_batch,
    recipient_chunks,
    recipient_coefficients,
    donor_coefficients,
    direction
):

    state = {"calls": 0, "offset": 0}

    delta_z = (
        donor_coefficients
        - recipient_coefficients
    )

    def hook(module, inputs, output):

        index = state["calls"]

        assert index < len(recipient_chunks)

        assert torch.allclose(
            output,
            recipient_chunks[index],
            rtol=1e-6,
            atol=1e-6
        )

        hidden = output.reshape(-1, 768)

        start = state["offset"]
        end = start + hidden.shape[0]

        displacement = (
            delta_z[start:end, None]
            * direction[None, :]
        )

        state["offset"] = end
        state["calls"] += 1

        return (
            hidden + displacement
        ).reshape(output.shape)

    handle = model_07.decoder[0].register_forward_hook(
        hook
    )

    try:
        output = model_07(recipient_batch)[0]
    finally:
        handle.remove()

    assert state["calls"] == expected_calls_07
    assert state["offset"] == 6642

    return {
        name: getattr(output, name)
        .detach().cpu().clone()
        for name in C3_OUTPUTS
    }

# --------------------------------------------------
# 4. Run target and control patches
# --------------------------------------------------

records_c3 = []

for pair_index, (context, recipient_id, donor_id) in enumerate(
    pairs_c3
):

    recipient_batch, recipient_genes = (
        prepare_condition_c1(recipient_id)
    )

    donor_batch, donor_genes = (
        prepare_condition_c1(donor_id)
    )

    assert np.array_equal(
        recipient_genes,
        donor_genes
    )

    recipient_outputs, recipient_chunks = (
        capture_condition_c1(recipient_batch)
    )

    donor_outputs, donor_chunks = (
        capture_condition_c1(donor_batch)
    )

    z_recipient = feature_coefficients_c1(
        recipient_chunks
    )

    z_donor = feature_coefficients_c1(
        donor_chunks
    )

    modified_mask = (
        (z_donor - z_recipient)
        .abs()
        .numpy() > 1e-6
    )

    n_modified = int(modified_mask.sum())

    # Verify the intervention implementation
    # on the first pair.
    if pair_index == 0:

        identity = run_patch_c3(
            recipient_batch,
            recipient_chunks,
            z_recipient,
            z_recipient,
            directions_c3["feature189"]
        )

        for name in C3_OUTPUTS:
            assert torch.allclose(
                identity[name],
                recipient_outputs[name],
                rtol=1e-6,
                atol=1e-6
            )

        print("\nC.3 identity patch: PASS")

    for label, direction in directions_c3.items():

        patched = run_patch_c3(
            recipient_batch,
            recipient_chunks,
            z_recipient,
            z_donor,
            direction
        )

        for name in C3_OUTPUTS:

            recipient = (
                recipient_outputs[name]
                .reshape(6642, -1)
                .numpy()
            )

            donor = (
                donor_outputs[name]
                .reshape(6642, -1)
                .numpy()
            )

            modified = (
                patched[name]
                .reshape(6642, -1)
                .numpy()
            )

            if n_modified > 0:

                before = float(
                    np.abs(
                        recipient[modified_mask]
                        - donor[modified_mask]
                    ).mean()
                )

                after = float(
                    np.abs(
                        modified[modified_mask]
                        - donor[modified_mask]
                    ).mean()
                )

                local_effect = float(
                    np.abs(
                        modified[modified_mask]
                        - recipient[modified_mask]
                    ).mean()
                )

                improvement = before - after

            else:
                before = np.nan
                after = np.nan
                local_effect = 0.0
                improvement = np.nan

            records_c3.append({
                "pair": pair_index,
                "context": context,
                "recipient": recipient_id,
                "donor": donor_id,
                "intervention": label,
                "output": name,
                "modified_positions": n_modified,
                "local_patch_effect": local_effect,
                "distance_before": before,
                "distance_after": after,
                "local_improvement": improvement,
                "global_improvement": float(
                    np.abs(recipient - donor).mean()
                    - np.abs(modified - donor).mean()
                ),
            })

    print(
        f"Completed pair {pair_index + 1}/9: "
        f"{context}, {recipient_id} <- {donor_id}, "
        f"modified positions={n_modified}"
    )

C3_results = pd.DataFrame(records_c3)

assert len(C3_results) == 9 * 6 * 3

# --------------------------------------------------
# 5. Compare target against control directions
# --------------------------------------------------

target_c3 = C3_results[
    C3_results["intervention"] == "feature189"
].copy()

controls_c3 = (
    C3_results[
        C3_results["intervention"] != "feature189"
    ]
    .groupby(["pair", "output"], as_index=False)
    .agg(
        control_mean_improvement=(
            "local_improvement", "mean"
        ),
        control_best_improvement=(
            "local_improvement", "max"
        ),
    )
)

C3_comparison = target_c3.merge(
    controls_c3,
    on=["pair", "output"],
    validate="one_to_one"
)

C3_comparison["better_than_control_mean"] = (
    C3_comparison["local_improvement"]
    > C3_comparison["control_mean_improvement"]
)

C3_comparison["better_than_all_controls"] = (
    C3_comparison["local_improvement"]
    > C3_comparison["control_best_improvement"]
)

C3_comparison["reduced_donor_distance"] = (
    C3_comparison["local_improvement"] > 0
)

# --------------------------------------------------
# 6. Summary
# --------------------------------------------------

print("\n=== CONTROLLED PATCHING SUMMARY ===")

C3_summary = (
    C3_comparison
    .groupby("output")
    .agg(
        pairs=("pair", "count"),
        target_mean_improvement=(
            "local_improvement", "mean"
        ),
        control_mean_improvement=(
            "control_mean_improvement", "mean"
        ),
        target_reduced_distance=(
            "reduced_donor_distance", "sum"
        ),
        target_above_control_mean=(
            "better_than_control_mean", "sum"
        ),
        target_above_all_controls=(
            "better_than_all_controls", "sum"
        ),
    )
)

print(
    C3_summary.to_string(
        float_format=lambda x: f"{x:.8f}"
    )
)

print("\n=== DE-LOGIT RESULTS BY PAIR ===")

print(
    C3_comparison.loc[
        C3_comparison["output"] == "de_logits",
        [
            "context",
            "recipient",
            "donor",
            "modified_positions",
            "local_improvement",
            "control_mean_improvement",
            "better_than_control_mean",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)

print("\n=== INTEGRITY ===")
print("Pairs:", len(pairs_c3))
print("Control directions:", len(control_ids_c3))
print("Original checkpoints unchanged.")
print("All intervention hooks removed.")

=== C.3. CONTROLLED ACTIVATION PATCHING ===

=== SELECTED PAIRS ===
hepg2: recipient=41, donor=5
hepg2: recipient=72, donor=11
hepg2: recipient=73, donor=75
jurkat: recipient=124, donor=104
jurkat: recipient=151, donor=103
jurkat: recipient=125, donor=169
rpe1: recipient=256, donor=221
rpe1: recipient=204, donor=285
rpe1: recipient=229, donor=267

Control SAE directions: 5
Direction normalization: PASS

C.3 identity patch: PASS
Completed pair 1/9: hepg2, 41 <- 5, modified positions=66
Completed pair 2/9: hepg2, 72 <- 11, modified positions=71
Completed pair 3/9: hepg2, 73 <- 75, modified positions=67
Completed pair 4/9: jurkat, 124 <- 104, modified positions=73
Completed pair 5/9: jurkat, 151 <- 103, modified positions=72
Completed pair 6/9: jurkat, 125 <- 169, modified positions=67
Completed pair 7/9: rpe1, 256 <- 221, modified positions=109
Completed pair 8/9: rpe1, 204 <- 285, modified positions=76
Completed pair 9/9: rpe1, 229 <- 267, modified positions=86

=== CONTROLLED PATCHING 

In [24]:
print("=== C.3. FINAL PATCHING SUMMARY ===")

display(
    C3_summary.reset_index().round(8)
)

print("\n=== DE-LOGIT RESULTS BY PAIR ===")

display(
    C3_comparison.loc[
        C3_comparison["output"] == "de_logits",
        [
            "context",
            "recipient",
            "donor",
            "modified_positions",
            "local_improvement",
            "control_mean_improvement",
            "better_than_control_mean",
            "better_than_all_controls",
        ]
    ].round(8)
)

=== C.3. FINAL PATCHING SUMMARY ===


,output,pairs,target_mean_improvement,control_mean_improvement,target_reduced_distance,target_above_control_mean,target_above_all_controls
0,de_logits,9,-0.003013,-0.000074,3,2,1
1,dp_logits,9,-0.001328,0.000056,3,3,2
2,lfc,9,-0.002843,-0.000353,2,1,0



=== DE-LOGIT RESULTS BY PAIR ===


,context,recipient,donor,modified_positions,local_improvement,control_mean_improvement,better_than_control_mean,better_than_all_controls
0,hepg2,41,5,66,0.000555,-0.001411,True,True
3,hepg2,72,11,71,0.001039,0.000261,True,False
6,hepg2,73,75,67,0.001054,0.001107,False,False
9,jurkat,124,104,73,-0.004627,-0.000007,False,False
12,jurkat,151,103,72,-0.012296,0.003096,False,False
15,jurkat,125,169,67,-0.003449,-0.000858,False,False
18,rpe1,256,221,109,-0.001582,-0.000245,False,False
21,rpe1,204,285,76,-0.000208,0.000714,False,False
24,rpe1,229,267,86,-0.007603,-0.003327,False,False


### C.4. Donor-Specificity and Reverse-Patching Controls

The nine-pair experiment did not demonstrate consistent
transfer of donor predictions through feature 189.

This section tests whether the identity of the donor
condition matters for the observed patching effect.

For each existing recipient–donor pair, four
interventions are evaluated:

1. Transfer from the original donor.
2. Transfer from two alternative donors sampled
   from the same cell context.
3. Reverse the original donor coefficient difference.

All interventions use the same decoder.0 SAE feature
189 direction and the same recipient input.

Alternative donors retain their naturally observed
SAE coefficients. Their coefficient differences may
therefore differ in magnitude from the original donor.

The reverse control uses the same displacement magnitude
as the original patch, with the opposite sign.

Donor distance is evaluated against the original
donor prediction for every intervention.

For a fair within-pair descriptive comparison, local effects are
evaluated on a common union of positions affected
by the candidate coefficient transfers. This mask differs
from the original-donor-only mask used in Section C.3.
Consequently, C.3 and C.4 local improvement values for the same
recipient–donor pair need not be numerically identical; they answer
slightly different questions. The C.4 code explicitly checks
reproduction of the C.3 *global* improvement, not equality of
the two differently masked local metrics.

The experiment is exploratory. Nine condition pairs
and two decoy donors per pair are insufficient for
confirmatory inference about donor specificity.

No model weights or research artifacts are modified.

In [25]:
import numpy as np
import pandas as pd
import torch

print("=== C.4. DONOR-SPECIFICITY CONTROLS ===")

rng_c4 = np.random.default_rng(20261117)

assert len(pairs_c3) == 9
assert len(C3_results) == 9 * 6 * 3
assert model_07.training is False
assert sae_07.training is False

records_c4 = []

for pair_id, (context, recipient_id, donor_id) in enumerate(pairs_c3):

    # Select two alternative donors in the same context.
    candidates = [
        int(i)
        for i in validation_indices_b5
        if str(
            dm_07.val_dataset.rows[int(i)].context
        ).strip().lower() == context
        and int(i) not in {recipient_id, donor_id}
    ]

    assert len(candidates) >= 2

    decoy_ids = rng_c4.choice(
        candidates, size=2, replace=False
    )

    # Original recipient and donor.
    recipient_batch, recipient_genes = (
        prepare_condition_c1(recipient_id)
    )
    donor_batch, donor_genes = (
        prepare_condition_c1(donor_id)
    )

    assert np.array_equal(recipient_genes, donor_genes)

    recipient_outputs, recipient_chunks = (
        capture_condition_c1(recipient_batch)
    )
    donor_outputs, donor_chunks = (
        capture_condition_c1(donor_batch)
    )

    z_rec = feature_coefficients_c1(recipient_chunks)
    z_donor = feature_coefficients_c1(donor_chunks)

    true_delta = z_donor - z_rec
    true_norm = float(torch.linalg.vector_norm(true_delta))

    assert true_norm > 0

    # Prepare all coefficient sources.
    sources = {
        "true_donor": z_donor,
        "reverse_target": z_rec - true_delta,
    }

    for j, decoy_id in enumerate(decoy_ids, start=1):

        decoy_batch, decoy_genes = (
            prepare_condition_c1(int(decoy_id))
        )

        assert np.array_equal(recipient_genes, decoy_genes)

        _, decoy_chunks = capture_condition_c1(decoy_batch)

        sources[f"decoy_{j}"] = (
            feature_coefficients_c1(decoy_chunks)
        )

    # Common evaluation mask, independent of PIE outputs.
    union_mask = torch.zeros(6642, dtype=torch.bool)

    for coefficients in sources.values():
        union_mask |= (coefficients - z_rec).abs() > 1e-6

    mask = union_mask.numpy()
    n_union = int(mask.sum())

    assert n_union > 0

    # Reuse the previously verified patching function.
    for label, coefficients in sources.items():

        patched = run_patch_c3(
            recipient_batch,
            recipient_chunks,
            z_rec,
            coefficients,
            direction_189_07.detach().cpu()
        )

        coefficient_norm = float(
            torch.linalg.vector_norm(coefficients - z_rec)
        )

        for output_name in C3_OUTPUTS:

            recipient = (
                recipient_outputs[output_name]
                .reshape(6642, -1)
                .numpy()
            )

            donor = (
                donor_outputs[output_name]
                .reshape(6642, -1)
                .numpy()
            )

            modified = (
                patched[output_name]
                .reshape(6642, -1)
                .numpy()
            )

            before = float(
                np.abs(recipient[mask] - donor[mask]).mean()
            )
            after = float(
                np.abs(modified[mask] - donor[mask]).mean()
            )

            global_improvement = float(
                np.abs(recipient - donor).mean()
                - np.abs(modified - donor).mean()
            )

            # Reproduce the existing C.3 target experiment.
            if label == "true_donor":

                previous = C3_results.loc[
                    C3_results["pair"].eq(pair_id)
                    & C3_results["intervention"].eq("feature189")
                    & C3_results["output"].eq(output_name)
                ]

                assert len(previous) == 1

                assert np.isclose(
                    global_improvement,
                    float(previous.iloc[0]["global_improvement"]),
                    atol=1e-6,
                    rtol=1e-4
                )

            records_c4.append({
                "pair": pair_id,
                "context": context,
                "recipient": recipient_id,
                "donor": donor_id,
                "source": label,
                "output": output_name,
                "evaluated_positions": n_union,
                "coefficient_norm_ratio": (
                    coefficient_norm / true_norm
                ),
                "local_improvement": before - after,
                "global_improvement": global_improvement,
            })

    print(
        f"Completed {pair_id + 1}/9: {context}, "
        f"{recipient_id} <- {donor_id}"
    )

C4_results = pd.DataFrame(records_c4)

assert len(C4_results) == 9 * 4 * 3

# --------------------------------------------------
# Compare true donor with alternative donors.
# --------------------------------------------------

true_c4 = C4_results[
    C4_results["source"].eq("true_donor")
].copy()

decoys_c4 = (
    C4_results[
        C4_results["source"].str.startswith("decoy_")
    ]
    .groupby(["pair", "output"], as_index=False)
    .agg(
        decoy_mean=("local_improvement", "mean"),
        decoy_best=("local_improvement", "max"),
        mean_decoy_norm_ratio=(
            "coefficient_norm_ratio", "mean"
        ),
    )
)

reverse_c4 = C4_results[
    C4_results["source"].eq("reverse_target")
][["pair", "output", "local_improvement"]].rename(
    columns={"local_improvement": "reverse_improvement"}
)

C4_comparison = (
    true_c4.merge(
        decoys_c4,
        on=["pair", "output"],
        validate="one_to_one"
    )
    .merge(
        reverse_c4,
        on=["pair", "output"],
        validate="one_to_one"
    )
)

C4_comparison["better_than_decoy_mean"] = (
    C4_comparison["local_improvement"]
    > C4_comparison["decoy_mean"]
)

C4_comparison["better_than_both_decoys"] = (
    C4_comparison["local_improvement"]
    > C4_comparison["decoy_best"]
)

C4_comparison["better_than_reverse"] = (
    C4_comparison["local_improvement"]
    > C4_comparison["reverse_improvement"]
)

C4_comparison["reduced_donor_distance"] = (
    C4_comparison["local_improvement"] > 0
)

# --------------------------------------------------
# Compact summary.
# --------------------------------------------------

C4_summary = (
    C4_comparison
    .groupby("output")
    .agg(
        pairs=("pair", "count"),
        mean_true_improvement=("local_improvement", "mean"),
        mean_decoy_improvement=("decoy_mean", "mean"),
        mean_reverse_improvement=("reverse_improvement", "mean"),
        true_reduced_distance=("reduced_donor_distance", "sum"),
        above_decoy_mean=("better_than_decoy_mean", "sum"),
        above_both_decoys=("better_than_both_decoys", "sum"),
        above_reverse=("better_than_reverse", "sum"),
        mean_decoy_norm_ratio=("mean_decoy_norm_ratio", "mean"),
    )
)

print("\n=== DONOR-SPECIFICITY SUMMARY ===")
print(
    C4_summary.reset_index().to_string(
        index=False,
        float_format=lambda x: f"{x:.7f}"
    )
)

print("\n=== DE-LOGIT PAIR COMPARISON ===")
print(
    C4_comparison.loc[
        C4_comparison["output"].eq("de_logits"),
        [
            "context",
            "recipient",
            "donor",
            "local_improvement",
            "decoy_mean",
            "reverse_improvement",
            "better_than_both_decoys",
            "better_than_reverse",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.7f}"
    )
)

print("\n=== INTEGRITY ===")
print("Pairs:", len(pairs_c3))
print("Alternative donors per pair: 2")
print("Original C.3 reproduction: PASS")
print("All intervention hooks removed.")
print("No research artifacts modified.")

=== C.4. DONOR-SPECIFICITY CONTROLS ===
Completed 1/9: hepg2, 41 <- 5
Completed 2/9: hepg2, 72 <- 11
Completed 3/9: hepg2, 73 <- 75
Completed 4/9: jurkat, 124 <- 104
Completed 5/9: jurkat, 151 <- 103
Completed 6/9: jurkat, 125 <- 169
Completed 7/9: rpe1, 256 <- 221
Completed 8/9: rpe1, 204 <- 285
Completed 9/9: rpe1, 229 <- 267

=== DONOR-SPECIFICITY SUMMARY ===
   output  pairs  mean_true_improvement  mean_decoy_improvement  mean_reverse_improvement  true_reduced_distance  above_decoy_mean  above_both_decoys  above_reverse  mean_decoy_norm_ratio
de_logits      9             -0.0027240              -0.0006118                 0.0029618                      3                 3                  3              3              1.0938008
dp_logits      9             -0.0010946               0.0015429                 0.0011865                      3                 1                  0              3              1.0938008
      lfc      9             -0.0025214               0.0011101        

### C.5. Activation Patching — Findings and Limitations

Controlled activation patching was evaluated for
decoder.0 SAE feature 189 using the pretrained PIE model.

The experiments transferred feature-specific SAE
coefficients between validation conditions while
preserving the remaining recipient hidden representation.

The identity-patching control reproduced the original
PIE outputs.

The initial within-context experiment altered the
recipient predictions but did not reduce their distance
to the donor predictions.

The subsequent nine-pair experiment covered HepG2,
Jurkat, and RPE1 conditions and compared feature 189
patching with five alternative learned SAE directions.

Feature 189 reduced local donor distance in only
three of nine pairs for DE logits, two of nine for
log-fold-change predictions, and three of nine for
delta-p logits.

Its improvement over the alternative SAE directions
was inconsistent.

A donor-specificity experiment further compared the
original donor with two alternative same-context donors
and a reversed coefficient displacement.

The original donor did not consistently outperform
these controls.

Mean donor-distance improvement, evaluated on each C.4
pair-specific union mask, was negative for all three PIE
outputs, whereas reversing the coefficient displacement
produced positive mean improvements in this exploratory panel.
The underlying alternatives are not exchangeability-matched
experiments, and this finding is not evidence of a reverse
biological mechanism.

The alternative donor controls were not exactly
displacement-magnitude matched. Their interpretation
is therefore conditional on the observed differences
in donor coefficient magnitudes.

The experiments establish that the selected
feature-direction interventions can modify PIE
predictions, but do not support consistent transfer
of donor predictions through feature 189 alone.

These results do not demonstrate biological
causality, mechanistic uniqueness, or
feature-specific mediation of perturbation effects.

The activation-patching pilot is complete.

## D. Consolidated Computational Intervention Evidence

### D.1. Intervention and Control Audit

This section consolidates the completed SAE
intervention experiments without executing new
PIE forward passes.

The analysis distinguishes:

- Nonzero computational effects following feature ablation.
- Sensitivity to intervention strength.
- Comparisons against random and learned SAE directions.
- Condition-level uncertainty.
- Activation patching and donor-specificity controls.

Computational influence is distinguished from
feature specificity and donor-state transfer.

Results are exploratory and conditional on the
fixed pretrained PIE model, trained SAE dictionaries,
selected validation conditions, and intervention
operators.

No independently validated biological causal
mechanism is claimed.

In [26]:
import numpy as np
import pandas as pd

print("=== D.1. COMPUTATIONAL INTERVENTION EVIDENCE ===")

# --------------------------------------------------
# 1. Verify completed analyses
# --------------------------------------------------

checks_d1 = {
    "Full feature-189 validation": len(B5_results) == 96,
    "Random-direction controls": len(B6_controls) == 480,
    "Condition bootstrap": len(B7_uncertainty) == 3,
    "Three-feature comparison": len(B8_results) == 288,
    "Learned-direction controls": len(B9_controls) == 288,
    "Nine-pair activation patching": len(C3_results) == 162,
    "Donor-specificity controls": len(C4_results) == 108,
}

for name, passed in checks_d1.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

assert all(checks_d1.values())

# --------------------------------------------------
# 2. Summarize ablation and uncertainty
# --------------------------------------------------

print("\n=== ABLATION AND RANDOM-DIRECTION EVIDENCE ===")

ablation_d1 = B7_uncertainty[
    [
        "output",
        "observed_ratio",
        "ratio_low",
        "ratio_high",
        "conditions_above_control",
    ]
].copy()

print(
    ablation_d1.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# --------------------------------------------------
# 3. Learned SAE-direction controls
# --------------------------------------------------

print("\n=== LEARNED SAE-DIRECTION EVIDENCE ===")

learned_d1 = B9_summary.reset_index()[
    [
        "output",
        "ratio_of_means",
        "above_control_mean",
        "above_all_controls",
    ]
]

print(
    learned_d1.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# --------------------------------------------------
# 4. Donor transfer evidence
# --------------------------------------------------

print("\n=== DONOR TRANSFER EVIDENCE ===")

transfer_d1 = C4_summary.reset_index()[
    [
        "output",
        "mean_true_improvement",
        "mean_decoy_improvement",
        "mean_reverse_improvement",
        "true_reduced_distance",
        "above_both_decoys",
        "above_reverse",
    ]
]

print(
    transfer_d1.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# --------------------------------------------------
# 5. Final qualitative distinctions
# --------------------------------------------------

print("\n=== EVIDENCE BOUNDARIES ===")

print("PIE baseline reproduction: PASS")
print("Identity intervention: PASS")
print("Feature-189 computational influence: OBSERVED")
print("Unique feature-189 direction: NOT ESTABLISHED")
print("Consistent donor-state transfer: NOT ESTABLISHED")
print("Biological causal validation: NOT PERFORMED")

print("\nRead-only intervention synthesis completed.")

=== D.1. COMPUTATIONAL INTERVENTION EVIDENCE ===
Full feature-189 validation: PASS
Random-direction controls: PASS
Condition bootstrap: PASS
Three-feature comparison: PASS
Learned-direction controls: PASS
Nine-pair activation patching: PASS
Donor-specificity controls: PASS

=== ABLATION AND RANDOM-DIRECTION EVIDENCE ===
   output  observed_ratio  ratio_low  ratio_high  conditions_above_control
de_logits        1.206079   1.000663    1.442586                        17
      lfc        1.299835   0.952334    1.746270                        19
dp_logits        1.358490   1.130817    1.611326                        18

=== LEARNED SAE-DIRECTION EVIDENCE ===
   output  ratio_of_means  above_control_mean  above_all_controls
de_logits        1.068860                   3                   1
dp_logits        1.122526                   2                   2
      lfc        1.301994                   2                   1

=== DONOR TRANSFER EVIDENCE ===
   output  mean_true_improvement  mean_de

## E. Cross-Layer Computational Interventions

### E.1. Decoder.0-to-Decoder.1 Feature Response

This experiment tests whether ablation of decoder.0
SAE feature 189 changes the activation of its
geometrically matched SAE feature in decoder.1.

The matched feature is recovered from the saved
dictionary correspondence of notebook 05.

The experiment first determines whether the match
is a mutual nearest neighbor. No mutuality is assumed.

The pretrained decoder.1 SAE is loaded from its
existing checkpoint.

For the first validation condition, decoder.1 hidden
representations are captured during:

1. An unmodified PIE forward pass.
2. A forward pass with decoder.0 feature 189 ablated.

The baseline decoder.1 representations are validated
against the previously saved activation HDF5.

Decoder.1 SAE activations are then compared before
and after intervention.

The analysis evaluates the matched feature response
alongside the responses of other decoder.1 SAE features.

A downstream activation change would demonstrate
sensitivity to the specified upstream intervention.

It would not establish a unique causal channel,
direct mediation, or a biological mechanism.

No pretrained weights or research files are modified.

In [27]:
import json
import numpy as np
import h5py
import torch
import torch.nn.functional as F

print("=== E.1. CROSS-LAYER FEATURE RESPONSE ===")

assert model_07.training is False
assert sae_07.training is False
assert FEATURE_07 == 189
assert identity_pass_07
assert len(baseline_predictions_07) == 3

# --------------------------------------------------
# 1. Recover the original dictionary correspondence
# --------------------------------------------------

match_path_e1 = (
    ROOT
    / "results/cross_layer_independent"
    / "dictionary_matches"
    / "decoder_0_to_decoder_1.npz"
)

with np.load(match_path_e1, allow_pickle=False) as data:
    match_ab_e1 = data["match_ab"].copy()
    mutual_mask_e1 = data["mutual_mask"].copy()
    best_cos_e1 = data["best_cosine"].copy()

assert match_ab_e1.shape == (3072,)
assert mutual_mask_e1.shape == (3072,)

TARGET_E1 = 189
MATCHED_E1 = int(match_ab_e1[TARGET_E1])
IS_MUTUAL_E1 = bool(mutual_mask_e1[TARGET_E1])

print("\n=== SAVED DICTIONARY MATCH ===")
print("Source: decoder.0 /", TARGET_E1)
print("Target: decoder.1 /", MATCHED_E1)
print("Mutual nearest neighbor:", IS_MUTUAL_E1)
print("Saved cosine:", float(best_cos_e1[TARGET_E1]))

# --------------------------------------------------
# 2. Load the pretrained decoder.1 SAE
# --------------------------------------------------

layer_e1 = "decoder.1"

sae_path_e1 = (
    ROOT / registry_07[layer_e1]["checkpoint"]
)

checkpoint_e1 = torch.load(
    sae_path_e1,
    map_location="cpu",
    weights_only=True,
)

assert checkpoint_e1["config"]["layer"] == layer_e1
assert (
    checkpoint_e1["config"]["init_seed"]
    == registry_07[layer_e1]["init_seed"]
)

sae_next_e1 = TopKSAE(
    d_model=768,
    n_features=3072,
    k=32,
)

sae_next_e1.load_state_dict(
    checkpoint_e1["model_state_dict"],
    strict=True,
)

sae_next_e1.requires_grad_(False)
sae_next_e1.eval()

mean_next_e1 = torch.from_numpy(
    np.load(
        sae_path_e1.parent / "activation_mean.npy"
    ).astype(np.float32)
)

assert mean_next_e1.shape == (768,)

# Independently verify the saved geometric match.

W_next_e1 = (
    sae_next_e1.W_dec.weight
    .detach().T.contiguous()
)

assert W_next_e1.shape == (3072, 768)

cosines_e1 = (
    F.normalize(W_next_e1, dim=1)
    @ F.normalize(
        direction_189_07.detach(),
        dim=0
    )
)

assert int(cosines_e1.argmax()) == MATCHED_E1

assert np.isclose(
    float(cosines_e1[MATCHED_E1]),
    float(best_cos_e1[TARGET_E1]),
    rtol=1e-5,
    atol=1e-5,
)

print("\n=== DECODER.1 SAE ===")
print("Checkpoint:", sae_path_e1.name)
print("Matching reproduction: PASS")
print("SAE training mode:", sae_next_e1.training)

# --------------------------------------------------
# 3. Capture decoder.1 with and without ablation
# --------------------------------------------------

@torch.inference_mode()
def capture_downstream_e1(apply_ablation=False):

    captured = []
    state = {
        "source_calls": 0,
        "active_positions": 0,
    }

    def capture_hook(module, inputs, output):
        assert isinstance(output, torch.Tensor)
        captured.append(output.detach().cpu().clone())

    def ablation_hook(module, inputs, output):

        hidden = output.reshape(-1, 768)

        z, _ = sae_07.encode(
            hidden - mean_07
        )

        activation = z[:, TARGET_E1]

        state["source_calls"] += 1
        state["active_positions"] += int(
            (activation > 0).sum().item()
        )

        displacement = (
            activation[:, None]
            * direction_189_07[None, :]
        )

        return (
            hidden - displacement
        ).reshape(output.shape)

    handles = []

    try:
        if apply_ablation:
            handles.append(
                model_07.decoder[0].register_forward_hook(
                    ablation_hook
                )
            )

        handles.append(
            model_07.decoder[1].register_forward_hook(
                capture_hook
            )
        )

        output = model_07(batch_ready_07)[0]

    finally:
        for handle in handles:
            handle.remove()

    assert len(captured) == expected_calls_07

    if apply_ablation:
        assert state["source_calls"] == expected_calls_07
        assert state["active_positions"] == active_positions_07

    hidden = torch.cat(
        captured,
        dim=1
    ).squeeze(0)

    assert hidden.shape == (6642, 768)

    predictions = {
        name: getattr(output, name)
        .detach().cpu().clone()
        for name in baseline_predictions_07
    }

    return hidden, predictions, state


hidden_next_base_e1, outputs_base_e1, _ = (
    capture_downstream_e1(False)
)

hidden_next_abl_e1, outputs_abl_e1, state_e1 = (
    capture_downstream_e1(True)
)

# --------------------------------------------------
# 4. Verify original activations and predictions
# --------------------------------------------------

with h5py.File(VAL_H5_07, "r") as h5:

    stored_next_e1 = np.asarray(
        h5["decoder"]["1"][0],
        dtype=np.float32
    )

assert np.allclose(
    hidden_next_base_e1.numpy(),
    stored_next_e1,
    rtol=1e-4,
    atol=1e-4
)

for name, baseline in baseline_predictions_07.items():

    assert torch.allclose(
        outputs_base_e1[name],
        baseline,
        rtol=1e-6,
        atol=1e-6
    )

    original_ablation = (
        getattr(ablated_outputs_07[0], name)
        .detach().cpu()
    )

    assert torch.allclose(
        outputs_abl_e1[name],
        original_ablation,
        rtol=1e-6,
        atol=1e-6
    )

print("\n=== REPRODUCIBILITY ===")
print("Decoder.1 saved HDF5 agreement: PASS")
print("Original baseline reproduction: PASS")
print("Original B.1 ablation reproduction: PASS")
print("Decoder.0 active positions:", state_e1["active_positions"])

# --------------------------------------------------
# 5. Inspect downstream hidden-state changes
# --------------------------------------------------

hidden_delta_e1 = (
    hidden_next_abl_e1 - hidden_next_base_e1
)

per_gene_l2_e1 = torch.linalg.vector_norm(
    hidden_delta_e1,
    dim=1
)

mask_e1 = torch.from_numpy(
    active_mask_b4.astype(bool)
)

assert mask_e1.shape == (6642,)
assert int(mask_e1.sum()) == active_positions_07

print("\n=== DOWNSTREAM HIDDEN-STATE RESPONSE ===")

print(
    "Mean L2 change on source-active positions:",
    float(per_gene_l2_e1[mask_e1].mean())
)

print(
    "Maximum L2 change on source-inactive positions:",
    float(per_gene_l2_e1[~mask_e1].max())
)

# --------------------------------------------------
# 6. Re-encode downstream representations with SAE
# --------------------------------------------------

with torch.inference_mode():

    z_next_base_e1, _ = sae_next_e1.encode(
        hidden_next_base_e1[mask_e1]
        - mean_next_e1
    )

    z_next_abl_e1, _ = sae_next_e1.encode(
        hidden_next_abl_e1[mask_e1]
        - mean_next_e1
    )

delta_z_next_e1 = (
    z_next_abl_e1 - z_next_base_e1
)

assert delta_z_next_e1.shape == (
    active_positions_07,
    3072
)

mean_response_e1 = (
    delta_z_next_e1.abs().mean(dim=0)
)

matched_response_e1 = float(
    mean_response_e1[MATCHED_E1]
)

matched_baseline_firing_e1 = int(
    (z_next_base_e1[:, MATCHED_E1] > 0).sum()
)

matched_ablated_firing_e1 = int(
    (z_next_abl_e1[:, MATCHED_E1] > 0).sum()
)

print("\n=== GEOMETRICALLY MATCHED FEATURE RESPONSE ===")

print("Decoder.1 feature:", MATCHED_E1)
print("Mutual match:", IS_MUTUAL_E1)
print("Mean absolute activation change:", matched_response_e1)
print("Baseline firing positions:", matched_baseline_firing_e1)
print("After-ablation firing positions:", matched_ablated_firing_e1)

if matched_response_e1 > 1e-10:

    response_rank_e1 = (
        1 + int(
            (mean_response_e1 > mean_response_e1[MATCHED_E1])
            .sum().item()
        )
    )

    print(
        "Response rank among all 3072 SAE features:",
        response_rank_e1
    )

else:
    print("Matched-feature response is numerically negligible.")
    response_rank_e1 = None

# --------------------------------------------------
# 7. Inspect the strongest downstream SAE responses
# --------------------------------------------------

top_ids_e1 = (
    torch.argsort(
        mean_response_e1,
        descending=True
    )[:8]
    .tolist()
)

print("\n=== TOP EIGHT DECODER.1 FEATURE RESPONSES ===")

for fid in top_ids_e1:

    print(
        f"Feature {fid:4d}: "
        f"mean_abs_delta_z="
        f"{float(mean_response_e1[fid]):.7f}"
        + (
            " [GEOMETRIC MATCH]"
            if fid == MATCHED_E1 else ""
        )
    )

print("\n=== INTEGRITY ===")
print("Original PIE predictions reproduced: PASS")
print("Decoder.1 baseline activations reproduced: PASS")
print("No research files modified.")
print("All intervention hooks removed.")

=== E.1. CROSS-LAYER FEATURE RESPONSE ===

=== SAVED DICTIONARY MATCH ===
Source: decoder.0 / 189
Target: decoder.1 / 413
Mutual nearest neighbor: True
Saved cosine: 0.30228251218795776

=== DECODER.1 SAE ===
Checkpoint: sae_final.pt
Matching reproduction: PASS
SAE training mode: False

=== REPRODUCIBILITY ===
Decoder.1 saved HDF5 agreement: PASS
Original baseline reproduction: PASS
Original B.1 ablation reproduction: PASS
Decoder.0 active positions: 68

=== DOWNSTREAM HIDDEN-STATE RESPONSE ===
Mean L2 change on source-active positions: 3.886430263519287
Maximum L2 change on source-inactive positions: 0.0

=== GEOMETRICALLY MATCHED FEATURE RESPONSE ===
Decoder.1 feature: 413
Mutual match: True
Mean absolute activation change: 0.8573279976844788
Baseline firing positions: 14
After-ablation firing positions: 9
Response rank among all 3072 SAE features: 1

=== TOP EIGHT DECODER.1 FEATURE RESPONSES ===
Feature  413: mean_abs_delta_z=0.8573280 [GEOMETRIC MATCH]
Feature 3022: mean_abs_delta_

### E.1.1. Matched-Feature Response Summary

The upstream feature 189 ablation altered the decoder.1
hidden representation at the source-active gene positions.

This analysis summarizes the response of the geometrically
matched decoder.1 feature 413 and compares its activation
change with the remaining features in the same SAE dictionary.

The comparison is descriptive. A high response rank does
not establish unique causal mediation, and a low rank does
not exclude a computational relationship.

No additional PIE inference is performed.

In [28]:
import numpy as np
import torch

print("=== E.1.1. MATCHED-FEATURE RESPONSE SUMMARY ===")

assert MATCHED_E1 == 413
assert IS_MUTUAL_E1
assert delta_z_next_e1.shape == (68, 3072)

# --------------------------------------------------
# 1. Matched SAE feature response
# --------------------------------------------------

matched_before = z_next_base_e1[:, MATCHED_E1]
matched_after = z_next_abl_e1[:, MATCHED_E1]

matched_delta = matched_after - matched_before

print("\n=== DECODER.1 FEATURE 413 ===")
print("Baseline active positions:",
      int((matched_before > 0).sum()))

print("After ablation active positions:",
      int((matched_after > 0).sum()))

print("Mean absolute activation change:",
      f"{matched_delta.abs().mean().item():.8f}")

print("Maximum absolute activation change:",
      f"{matched_delta.abs().max().item():.8f}")

print("Mean signed activation change:",
      f"{matched_delta.mean().item():.8f}")

print("Positions with changed activation (>1e-6):",
      int((matched_delta.abs() > 1e-6).sum()))

# --------------------------------------------------
# 2. Compare with all downstream SAE features
# --------------------------------------------------

response = mean_response_e1.detach().cpu()

rank = 1 + int(
    (response > response[MATCHED_E1]).sum()
)

n_responding = int(
    (response > 1e-6).sum()
)

print("\n=== RELATIVE RESPONSE ===")

print("Rank among 3072 SAE features:", rank)
print("SAE features with response > 1e-6:", n_responding)

print("Matched-feature response:",
      f"{response[MATCHED_E1].item():.8f}")

print("Dictionary median response:",
      f"{response.median().item():.8f}")

print("Dictionary 95th percentile:",
      f"{torch.quantile(response, 0.95).item():.8f}")

# --------------------------------------------------
# 3. Strongest downstream responses
# --------------------------------------------------

print("\n=== TOP EIGHT DOWNSTREAM RESPONSES ===")

for fid in torch.argsort(response, descending=True)[:8].tolist():

    print(
        f"Feature {fid:4d}: "
        f"{response[fid].item():.8f}"
        + (" [MATCHED]" if fid == MATCHED_E1 else "")
    )

print("\nRead-only response summary completed.")

=== E.1.1. MATCHED-FEATURE RESPONSE SUMMARY ===

=== DECODER.1 FEATURE 413 ===
Baseline active positions: 14
After ablation active positions: 9
Mean absolute activation change: 0.85732806
Maximum absolute activation change: 6.79804230
Mean signed activation change: -0.85732806
Positions with changed activation (>1e-6): 14

=== RELATIVE RESPONSE ===
Rank among 3072 SAE features: 1
SAE features with response > 1e-6: 1180
Matched-feature response: 0.85732800
Dictionary median response: 0.00000000
Dictionary 95th percentile: 0.04339994

=== TOP EIGHT DOWNSTREAM RESPONSES ===
Feature  413: 0.85732800 [MATCHED]
Feature 3022: 0.39376903
Feature  871: 0.35827509
Feature  223: 0.33168116
Feature 1168: 0.32336393
Feature 1724: 0.31761220
Feature  658: 0.31131688
Feature 2081: 0.29041883

Read-only response summary completed.


### E.2. Matched Downstream Response Under Learned-Direction Controls

Ablation of decoder.0 feature 189 produced the largest
mean absolute decoder.1 SAE activation response in its
geometrically matched feature 413.

This experiment tests the specificity of that response
relative to sixteen alternative directions from the
same trained decoder.0 SAE dictionary.

All interventions use the original feature 189
activation coefficients and firing positions.

The control directions are rescaled to the norm of
the feature 189 decoder direction.

For each intervention, the decoder.1 hidden states
are captured and encoded using the pretrained
decoder.1 SAE.

The response of feature 413 is compared across
the target intervention and the sixteen controls.

This design isolates directional sensitivity under
a fixed activation gate. It does not compare natural
ablations of sixteen independently activated features.

A strong matched-feature response can result from
dictionary geometry, TopK feature competition, or
other properties of the learned representations.

The experiment is exploratory and is restricted to
one validation condition. It does not establish
unique causal mediation or biological function.

All operations are performed in memory.

In [29]:
import numpy as np
import pandas as pd
import torch

print("=== E.2. CONTROLLED CROSS-LAYER RESPONSE ===")

assert MATCHED_E1 == 413
assert IS_MUTUAL_E1
assert FEATURE_07 == 189
assert len(control_directions_b9) == 16
assert z_next_base_e1.shape == (68, 3072)

TARGET_DIRECTION_E2 = direction_189_07.detach().cpu()

directions_e2 = {
    "feature189": TARGET_DIRECTION_E2
}

for fid, direction in sorted(control_directions_b9.items()):
    directions_e2[f"control_{fid}"] = direction.detach().cpu()

assert len(directions_e2) == 17

target_norm_e2 = torch.linalg.vector_norm(
    TARGET_DIRECTION_E2
)

for direction in directions_e2.values():
    assert torch.allclose(
        torch.linalg.vector_norm(direction),
        target_norm_e2,
        rtol=1e-5,
        atol=1e-6
    )

# --------------------------------------------------
# 1. Capture downstream activations under intervention
# --------------------------------------------------

@torch.inference_mode()
def downstream_response_e2(direction):

    captured = []
    state = {"calls": 0, "active": 0}

    def source_hook(module, inputs, output):

        hidden = output.reshape(-1, 768)

        z, _ = sae_07.encode(hidden - mean_07)
        activation = z[:, FEATURE_07]

        state["calls"] += 1
        state["active"] += int((activation > 0).sum())

        displacement = (
            activation[:, None] * direction[None, :]
        )

        return (hidden - displacement).reshape(output.shape)

    def downstream_hook(module, inputs, output):
        captured.append(output.detach().cpu().clone())

    handles = []

    try:
        handles.append(
            model_07.decoder[0].register_forward_hook(
                source_hook
            )
        )
        handles.append(
            model_07.decoder[1].register_forward_hook(
                downstream_hook
            )
        )

        _ = model_07(batch_ready_07)

    finally:
        for handle in handles:
            handle.remove()

    assert state["calls"] == expected_calls_07
    assert state["active"] == active_positions_07
    assert len(captured) == expected_calls_07

    hidden_next = torch.cat(
        captured, dim=1
    ).squeeze(0)

    assert hidden_next.shape == (6642, 768)

    # The source intervention is zero outside
    # the original feature-active positions.
    assert torch.allclose(
        hidden_next[~mask_e1],
        hidden_next_base_e1[~mask_e1],
        rtol=1e-6,
        atol=1e-6
    )

    z_next, _ = sae_next_e1.encode(
        hidden_next[mask_e1] - mean_next_e1
    )

    assert z_next.shape == (68, 3072)

    return z_next


# --------------------------------------------------
# 2. Run target and learned-direction controls
# --------------------------------------------------

records_e2 = []

for label, direction in directions_e2.items():

    z_intervened = downstream_response_e2(direction)

    # Reproduce the previously observed target response.
    if label == "feature189":
        assert torch.allclose(
            z_intervened,
            z_next_abl_e1,
            rtol=1e-5,
            atol=1e-5
        )
        print("Original E.1 response reproduction: PASS")

    delta = z_intervened - z_next_base_e1

    feature413_delta = delta[:, MATCHED_E1]

    response_by_feature = delta.abs().mean(dim=0)

    rank_413 = 1 + int(
        (
            response_by_feature
            > response_by_feature[MATCHED_E1]
        ).sum()
    )

    records_e2.append({
        "intervention": label,
        "matched_mean_abs_response": float(
            feature413_delta.abs().mean()
        ),
        "matched_mean_signed_response": float(
            feature413_delta.mean()
        ),
        "matched_max_abs_response": float(
            feature413_delta.abs().max()
        ),
        "matched_response_rank": rank_413,
        "matched_active_after": int(
            (z_intervened[:, MATCHED_E1] > 0).sum()
        ),
    })

E2_results = pd.DataFrame(records_e2)

assert len(E2_results) == 17

# --------------------------------------------------
# 3. Compare the target with sixteen controls
# --------------------------------------------------

target_e2 = E2_results.loc[
    E2_results["intervention"] == "feature189"
].iloc[0]

controls_e2 = E2_results.loc[
    E2_results["intervention"] != "feature189"
]

target_response_e2 = float(
    target_e2["matched_mean_abs_response"]
)

control_responses_e2 = controls_e2[
    "matched_mean_abs_response"
].to_numpy(dtype=np.float64)

n_stronger_e2 = int(
    (control_responses_e2 >= target_response_e2).sum()
)

print("\n=== FEATURE 413 RESPONSE COMPARISON ===")

print("Target mean absolute response:",
      f"{target_response_e2:.8f}")

print("Control mean response:",
      f"{control_responses_e2.mean():.8f}")

print("Control median response:",
      f"{np.median(control_responses_e2):.8f}")

print("Maximum control response:",
      f"{control_responses_e2.max():.8f}")

print(
    "Target / control mean:",
    f"{target_response_e2 / control_responses_e2.mean():.4f}"
    if control_responses_e2.mean() > 0 else "undefined"
)

print(
    "Controls with response >= target:",
    n_stronger_e2,
    "/16"
)

print(
    "Target rank among 17 interventions:",
    1 + n_stronger_e2
)

print("\n=== ALL INTERVENTION RESPONSES ===")

print(
    E2_results.sort_values(
        "matched_mean_abs_response",
        ascending=False
    ).to_string(
        index=False,
        float_format=lambda x: f"{x:.7f}"
    )
)

print("\n=== INTEGRITY ===")
print("Original response reproduction: PASS")
print("All directions norm-matched: PASS")
print("Original checkpoints unchanged.")
print("All intervention hooks removed.")

=== E.2. CONTROLLED CROSS-LAYER RESPONSE ===
Original E.1 response reproduction: PASS

=== FEATURE 413 RESPONSE COMPARISON ===
Target mean absolute response: 0.85732806
Control mean response: 0.12222171
Control median response: 0.07068281
Maximum control response: 0.39063820
Target / control mean: 7.0145
Controls with response >= target: 0 /16
Target rank among 17 interventions: 1

=== ALL INTERVENTION RESPONSES ===
intervention  matched_mean_abs_response  matched_mean_signed_response  matched_max_abs_response  matched_response_rank  matched_active_after
  feature189                  0.8573281                    -0.8573281                 6.7980423                      1                     9
 control_205                  0.3906382                    -0.3906382                 4.4207649                      4                    12
control_2427                  0.2591722                     0.2591722                 2.2398434                      5                    14
 control_594    

### E.3. Cross-Context Consistency of the 189-to-413 Response

The first cross-layer intervention showed that ablating
decoder.0 feature 189 produced the largest decoder.1
SAE activation response in its geometrically matched
feature 413.

The matched-feature response also exceeded responses
induced by sixteen norm-matched learned SAE directions
on the same validation condition.

This experiment evaluates whether that result persists
across six previously selected validation conditions,
with two conditions from each of HepG2, Jurkat, and RPE1.

For every condition:

- The unmodified decoder.0 and decoder.1 representations
  are captured.
- The feature 189 activation gate is reconstructed.
- The same target and sixteen control directions are
  applied to the decoder.0 representation.
- Decoder.1 SAE activations are recomputed.
- Feature 413 is evaluated against the fixed controls
  and ranked among all decoder.1 SAE features.

The control directions are held fixed across conditions.

The primary outcomes are the mean absolute activation
change of feature 413 and its rank relative to the
control interventions.

The experiment tests within-model computational directional
sensitivity, not unique causal mediation. The six conditions
were selected in an earlier pilot, not drawn as independent
biological experiments or held out from model interpretation.

Six validation conditions and sixteen fixed control
directions do not provide independent biological
replication or confirmatory statistical inference.

All interventions are applied in memory.

In [30]:
import numpy as np
import pandas as pd
import torch
import h5py

print("=== E.3. CROSS-CONTEXT MATCHED-FEATURE RESPONSE ===")

assert MATCHED_E1 == 413
assert IS_MUTUAL_E1
assert len(selected_conditions) == 6
assert len(directions_e2) == 17
assert model_07.training is False
assert sae_07.training is False
assert sae_next_e1.training is False

E3_FEATURE = 189
E3_MATCHED = 413

E3_directions = {
    name: direction.detach().cpu()
    for name, direction in directions_e2.items()
}

assert "feature189" in E3_directions

# --------------------------------------------------
# 1. Capture unmodified representations
# --------------------------------------------------

@torch.inference_mode()
def capture_baseline_e3(batch_ready):

    source_chunks = []
    downstream_chunks = []

    def source_hook(module, inputs, output):
        source_chunks.append(
            output.detach().cpu().clone()
        )

    def downstream_hook(module, inputs, output):
        downstream_chunks.append(
            output.detach().cpu().clone()
        )

    handles = [
        model_07.decoder[0].register_forward_hook(
            source_hook
        ),
        model_07.decoder[1].register_forward_hook(
            downstream_hook
        ),
    ]

    try:
        _ = model_07(batch_ready)
    finally:
        for handle in handles:
            handle.remove()

    assert len(source_chunks) == expected_calls_07
    assert len(downstream_chunks) == expected_calls_07

    hidden_next = torch.cat(
        downstream_chunks, dim=1
    ).squeeze(0)

    assert hidden_next.shape == (6642, 768)

    # Encode the source feature in manageable chunks.
    source_coefficients = []

    for chunk in source_chunks:
        hidden = chunk.reshape(-1, 768)

        z, _ = sae_07.encode(hidden - mean_07)

        source_coefficients.append(
            z[:, E3_FEATURE].detach().cpu()
        )

    coefficients = torch.cat(source_coefficients)

    assert coefficients.shape == (6642,)

    return source_chunks, hidden_next, coefficients


# --------------------------------------------------
# 2. Controlled upstream intervention
# --------------------------------------------------

@torch.inference_mode()
def run_cross_layer_e3(
    batch_ready,
    source_chunks,
    source_coefficients,
    direction,
):

    downstream_chunks = []
    state = {"calls": 0, "offset": 0}

    def source_hook(module, inputs, output):

        index = state["calls"]

        assert torch.allclose(
            output,
            source_chunks[index],
            rtol=1e-6,
            atol=1e-6
        )

        hidden = output.reshape(-1, 768)

        start = state["offset"]
        end = start + hidden.shape[0]

        displacement = (
            source_coefficients[start:end, None]
            * direction[None, :]
        )

        state["calls"] += 1
        state["offset"] = end

        return (hidden - displacement).reshape(
            output.shape
        )

    def downstream_hook(module, inputs, output):
        downstream_chunks.append(
            output.detach().cpu().clone()
        )

    handles = [
        model_07.decoder[0].register_forward_hook(
            source_hook
        ),
        model_07.decoder[1].register_forward_hook(
            downstream_hook
        ),
    ]

    try:
        _ = model_07(batch_ready)
    finally:
        for handle in handles:
            handle.remove()

    assert state["calls"] == expected_calls_07
    assert state["offset"] == 6642
    assert len(downstream_chunks) == expected_calls_07

    return torch.cat(
        downstream_chunks, dim=1
    ).squeeze(0)


# --------------------------------------------------
# 3. Evaluate six validation conditions
# --------------------------------------------------

E3_records = []

for context, h5_position, dataset_index in selected_conditions:

    dataset_index = int(dataset_index)

    batch_ready, gene_ids = prepare_condition_c1(
        dataset_index
    )

    source_chunks, baseline_hidden, coefficients = (
        capture_baseline_e3(batch_ready)
    )

    # Verify the original decoder.1 extraction.
    with h5py.File(VAL_H5_07, "r") as h5:

        assert int(
            h5["condition_indices"][h5_position]
        ) == dataset_index

        stored_hidden = np.asarray(
            h5["decoder"]["1"][h5_position],
            dtype=np.float32
        )

    assert np.allclose(
        baseline_hidden.numpy(),
        stored_hidden,
        rtol=1e-4,
        atol=1e-4
    )

    active_mask = coefficients > 0
    n_active = int(active_mask.sum())

    expected_active = int(
        B5_results.loc[
            B5_results["dataset_index"].eq(dataset_index)
            & B5_results["output"].eq("de_logits"),
            "active_positions"
        ].iloc[0]
    )

    assert n_active == expected_active
    assert n_active > 0

    with torch.inference_mode():
        z_baseline, _ = sae_next_e1.encode(
            baseline_hidden[active_mask] - mean_next_e1
        )

    for label, direction in E3_directions.items():

        modified_hidden = run_cross_layer_e3(
            batch_ready,
            source_chunks,
            coefficients,
            direction
        )

        assert torch.allclose(
            modified_hidden[~active_mask],
            baseline_hidden[~active_mask],
            atol=1e-6,
            rtol=1e-6
        )

        with torch.inference_mode():
            z_modified, _ = sae_next_e1.encode(
                modified_hidden[active_mask] - mean_next_e1
            )

        delta = z_modified - z_baseline
        responses = delta.abs().mean(dim=0)

        matched_response = float(
            responses[E3_MATCHED]
        )

        matched_rank = 1 + int(
            (responses > responses[E3_MATCHED]).sum()
        )

        # Verify reproducibility against E.1 on
        # the original validation condition.
        if (
            dataset_index == val_index_07
            and label == "feature189"
        ):
            assert torch.allclose(
                z_modified,
                z_next_abl_e1,
                rtol=1e-5,
                atol=1e-5
            )
            print("Original E.1 response: PASS")

        E3_records.append({
            "context": context,
            "dataset_index": dataset_index,
            "intervention": label,
            "source_active_positions": n_active,
            "matched_response": matched_response,
            "matched_signed_response": float(
                delta[:, E3_MATCHED].mean()
            ),
            "matched_response_rank": matched_rank,
            "matched_firing_before": int(
                (z_baseline[:, E3_MATCHED] > 0).sum()
            ),
            "matched_firing_after": int(
                (z_modified[:, E3_MATCHED] > 0).sum()
            ),
        })

    print(
        f"Completed: {context}, "
        f"dataset index {dataset_index}"
    )

E3_results = pd.DataFrame(E3_records)

assert len(E3_results) == 6 * 17

# --------------------------------------------------
# 4. Compare target with controls condition by condition
# --------------------------------------------------

comparison_rows = []

for (context, dataset_index), frame in E3_results.groupby(
    ["context", "dataset_index"]
):

    target = frame.loc[
        frame["intervention"] == "feature189"
    ].iloc[0]

    controls = frame.loc[
        frame["intervention"] != "feature189",
        "matched_response"
    ].to_numpy(dtype=np.float64)

    assert len(controls) == 16

    target_response = float(target["matched_response"])
    control_mean = float(controls.mean())

    n_controls_ge = int(
        (controls >= target_response).sum()
    )

    comparison_rows.append({
        "context": context,
        "dataset_index": dataset_index,
        "active_positions": int(
            target["source_active_positions"]
        ),
        "target_response": target_response,
        "control_mean": control_mean,
        "control_max": float(controls.max()),
        "target_to_control_ratio": (
            target_response / control_mean
            if control_mean > 0 else np.nan
        ),
        "target_rank_among_17": 1 + n_controls_ge,
        "matched_rank_among_3072": int(
            target["matched_response_rank"]
        ),
        "matched_firing_before": int(
            target["matched_firing_before"]
        ),
        "matched_firing_after": int(
            target["matched_firing_after"]
        ),
    })

E3_comparison = pd.DataFrame(comparison_rows)

assert len(E3_comparison) == 6

print("\n=== CROSS-CONTEXT FEATURE 413 RESPONSE ===")

print(
    E3_comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# --------------------------------------------------
# 5. Summary by cellular context
# --------------------------------------------------

E3_context_summary = (
    E3_comparison
    .groupby("context")
    .agg(
        conditions=("dataset_index", "count"),
        mean_target=("target_response", "mean"),
        mean_control=("control_mean", "mean"),
        target_best_among_17=(
            "target_rank_among_17",
            lambda s: int((s == 1).sum())
        ),
        median_matched_rank=(
            "matched_rank_among_3072", "median"
        ),
    )
)

E3_context_summary["ratio_of_means"] = (
    E3_context_summary["mean_target"]
    / E3_context_summary["mean_control"]
)

print("\n=== CONTEXT SUMMARY ===")

print(
    E3_context_summary.to_string(
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n=== INTEGRITY ===")
print("Validation conditions:", len(E3_comparison))
print("Directions per condition:", len(E3_directions))
print("Original checkpoints unchanged.")
print("All intervention hooks removed.")

=== E.3. CROSS-CONTEXT MATCHED-FEATURE RESPONSE ===
Original E.1 response: PASS
Completed: hepg2, dataset index 1
Completed: hepg2, dataset index 5
Completed: jurkat, dataset index 103
Completed: jurkat, dataset index 104
Completed: rpe1, dataset index 204
Completed: rpe1, dataset index 221

=== CROSS-CONTEXT FEATURE 413 RESPONSE ===
context  dataset_index  active_positions  target_response  control_mean  control_max  target_to_control_ratio  target_rank_among_17  matched_rank_among_3072  matched_firing_before  matched_firing_after
  hepg2              1                68         0.857328      0.122222     0.390638                 7.014531                     1                        1                     14                     9
  hepg2              5                66         0.862508      0.142312     0.367429                 6.060698                     1                        1                     13                     8
 jurkat            103                67         0.891481 

### E.4. Geometric Alignment and Cross-Layer Response

The upstream feature 189 intervention produced the
largest response of the geometrically matched
decoder.1 feature 413 in all six evaluated conditions.

It also exceeded the responses induced by sixteen
alternative learned SAE directions.

However, feature 189 and feature 413 were originally
matched using cosine similarity between their
decoder dictionary directions.

This creates a potential geometric explanation
for the observed intervention sensitivity.

The present analysis examines the relationship
between source-to-target dictionary alignment
and downstream activation response.

For each of the seventeen intervention directions,
we compute:

- Signed cosine similarity to the decoder.1
  feature 413 dictionary direction.
- Absolute cosine similarity.
- Mean feature 413 response across the six
  validation conditions.
- Rank by geometric alignment and response magnitude.

Descriptive correlations are calculated across the
fixed intervention panel.

A correlation between geometric alignment and
downstream response would indicate a possible
geometric contribution, but would not establish
that geometry fully explains the effect.

The analysis cannot identify a unique causal
mediation pathway or separate TopK competition
from other representation effects.

No additional model inference is performed.

In [31]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

print("=== E.4. GEOMETRY AND DOWNSTREAM RESPONSE AUDIT ===")

assert MATCHED_E1 == 413
assert FEATURE_07 == 189
assert IS_MUTUAL_E1

assert len(E3_results) == 6 * 17
assert len(E3_comparison) == 6
assert len(E3_directions) == 17

# --------------------------------------------------
# 1. Recover the decoder.1 matched-feature direction
# --------------------------------------------------

target_next_e4 = (
    W_next_e1[MATCHED_E1]
    .detach()
    .cpu()
    .float()
)

assert target_next_e4.shape == (768,)

target_next_e4 = F.normalize(
    target_next_e4, dim=0
)

# --------------------------------------------------
# 2. Calculate geometric alignment for each direction
# --------------------------------------------------

geometry_rows_e4 = []

for label, direction in E3_directions.items():

    direction = (
        direction.detach().cpu().float()
    )

    assert direction.shape == (768,)

    normalized = F.normalize(
        direction, dim=0
    )

    signed_cosine = float(
        torch.dot(normalized, target_next_e4)
    )

    geometry_rows_e4.append({
        "intervention": label,
        "signed_cosine": signed_cosine,
        "abs_cosine": abs(signed_cosine),
    })

E4_geometry = pd.DataFrame(geometry_rows_e4)

assert len(E4_geometry) == 17

# --------------------------------------------------
# 3. Aggregate downstream responses across conditions
# --------------------------------------------------

E4_responses = (
    E3_results
    .groupby("intervention", as_index=False)
    .agg(
        mean_response=("matched_response", "mean"),
        median_response=("matched_response", "median"),
        mean_signed_response=(
            "matched_signed_response", "mean"
        ),
    )
)

E4_results = E4_geometry.merge(
    E4_responses,
    on="intervention",
    validate="one_to_one"
)

assert len(E4_results) == 17

# Rank from highest to lowest.
E4_results["response_rank"] = (
    E4_results["mean_response"]
    .rank(method="min", ascending=False)
    .astype(int)
)

E4_results["geometric_rank"] = (
    E4_results["abs_cosine"]
    .rank(method="min", ascending=False)
    .astype(int)
)

# --------------------------------------------------
# 4. Print the geometric comparison
# --------------------------------------------------

print("\n=== GEOMETRY AND RESPONSE COMPARISON ===")

print(
    E4_results.sort_values(
        "mean_response",
        ascending=False
    ).to_string(
        index=False,
        float_format=lambda x: f"{x:.7f}"
    )
)

# --------------------------------------------------
# 5. Compare feature 189 with learned controls
# --------------------------------------------------

target_e4 = E4_results.loc[
    E4_results["intervention"].eq("feature189")
].iloc[0]

controls_e4 = E4_results.loc[
    ~E4_results["intervention"].eq("feature189")
].copy()

print("\n=== TARGET VS CONTROL GEOMETRY ===")

print(
    "Feature 189 cosine to feature 413:",
    f"{target_e4['signed_cosine']:.7f}"
)

print(
    "Maximum control absolute cosine:",
    f"{controls_e4['abs_cosine'].max():.7f}"
)

print(
    "Mean control absolute cosine:",
    f"{controls_e4['abs_cosine'].mean():.7f}"
)

print(
    "Feature 189 geometric rank:",
    int(target_e4["geometric_rank"]),
    "/17"
)

print(
    "Feature 189 response rank:",
    int(target_e4["response_rank"]),
    "/17"
)

# --------------------------------------------------
# 6. Descriptive geometry-response correlations
# --------------------------------------------------

pearson_all_e4 = E4_results[
    ["abs_cosine", "mean_response"]
].corr(method="pearson").iloc[0, 1]

spearman_all_e4 = E4_results[
    ["abs_cosine", "mean_response"]
].corr(method="spearman").iloc[0, 1]

pearson_controls_e4 = controls_e4[
    ["abs_cosine", "mean_response"]
].corr(method="pearson").iloc[0, 1]

spearman_controls_e4 = controls_e4[
    ["abs_cosine", "mean_response"]
].corr(method="spearman").iloc[0, 1]

print("\n=== GEOMETRY-RESPONSE ASSOCIATION ===")

print(
    "Pearson, all 17 directions:",
    f"{pearson_all_e4:.6f}"
)

print(
    "Spearman, all 17 directions:",
    f"{spearman_all_e4:.6f}"
)

print(
    "Pearson, 16 controls only:",
    f"{pearson_controls_e4:.6f}"
)

print(
    "Spearman, 16 controls only:",
    f"{spearman_controls_e4:.6f}"
)

# --------------------------------------------------
# 7. Integrity checks
# --------------------------------------------------

assert np.isfinite(
    E4_results[
        ["signed_cosine", "abs_cosine", "mean_response"]
    ].to_numpy()
).all()

assert np.isclose(
    float(target_e4["signed_cosine"]),
    float(best_cos_e1[TARGET_E1]),
    rtol=1e-5,
    atol=1e-5,
)

assert int(target_e4["response_rank"]) == 1

print("\n=== INTEGRITY ===")
print("Intervention directions:", len(E4_results))
print("Validation conditions:", 6)
print("Original cross-layer cosine reproduced: PASS")
print("No additional PIE inference performed.")
print("No checkpoints or research files modified.")

=== E.4. GEOMETRY AND DOWNSTREAM RESPONSE AUDIT ===

=== GEOMETRY AND RESPONSE COMPARISON ===
intervention  signed_cosine  abs_cosine  mean_response  median_response  mean_signed_response  response_rank  geometric_rank
  feature189      0.3022826   0.3022826      0.8440792        0.8599181            -0.8440792              1               1
 control_205      0.0615810   0.0615810      0.3762724        0.3790338            -0.3762724              2               3
control_2427     -0.0052855   0.0052855      0.2668683        0.2631697             0.2668683              3              15
control_2210     -0.0701765   0.0701765      0.2628709        0.2580772             0.2628709              4               2
 control_594      0.0376652   0.0376652      0.2542733        0.2584556            -0.2542734              5               9
control_1420      0.0433820   0.0433820      0.1765064        0.1931549            -0.1765064              6               8
 control_525     -0.0515811   0

### E.5. Cross-Layer Intervention — Findings and Limitations

The intervention experiments examined the computational
relationship between decoder.0 SAE feature 189 and its
geometrically matched decoder.1 SAE feature 413.

The pair was identified as a mutual nearest-neighbor
match in the previously saved dictionary correspondence.

The signed cosine similarity between their decoder
directions was approximately 0.3023.

Ablation of feature 189 altered the decoder.1 hidden
representation and reduced the activation of feature 413
in the initial validation condition.

Across six validation conditions spanning HepG2, Jurkat,
and RPE1, feature 413 ranked first among all 3,072
decoder.1 SAE features by mean absolute activation
response to feature 189 ablation. This rank was computed
**only on source-feature-active gene positions** in each
condition, not across all 6,642 output genes.

The target intervention also exceeded sixteen fixed,
norm-matched learned-direction controls in all six
evaluated conditions.

However, feature 189 had substantially greater
geometric alignment with feature 413 than any of
the selected controls. The controls were matched for vector
norm and source-feature gating, not for cosine similarity
to feature 413. TopK competition may also alter the measured
SAE response. Therefore, response superiority among these
controls cannot identify a geometry-independent or uniquely
mediated effect.

The strongest observed downstream response is
therefore consistent with a geometric explanation
and does not establish a distinct causal pathway.

The experiments demonstrate computational sensitivity
of the downstream representation to the specified
upstream intervention.

They do not establish unique causal mediation,
feature-specific biological function, or an
independently validated biological mechanism.

Separating geometric alignment from additional
computational effects would require a stronger
control design, including geometry-matched
interventions and potentially direct downstream
mediation experiments.

The cross-layer intervention pilot is complete.

## F. Final Research Synthesis

### F.1. Computational Intervention Findings

This notebook investigated the computational influence
of selected SAE features in the pretrained PIE model.
The complete experiment was evaluated with fixed pretrained
weights, three preselected decoder.0 SAE features, and
previously selected validation conditions.

**Evidence summary (descriptive, not confirmatory):**

| Question | Observed result | Main qualification |
|---|---|---|
| Does feature 189 ablation alter PIE predictions? | Detectable changes for all 32 selected conditions | Changes are expected under a nonzero hidden-state intervention and do not by themselves imply feature specificity |
| Is feature 189 more influential than controls? | Mean effects 1.21–1.36 times the means of five random directions | Only 17–19 of 32 conditions exceeded the mean control; learned-direction evidence was weaker |
| Are feature effects uniform? | Feature 2120 had effects in 19/32 conditions; the three largest accounted for about 72% of total DE-logit effect | Effects and activation frequency differ greatly between features |
| Does feature 189 patch donor state? | In nine directed pairs, donor distance decreased for only 2–3 pairs per output | Donor and direction controls did not support reliable transfer |
| Is feature 413 downstream-responsive? | Matched feature ranked first among 3,072 in all six selected conditions | Pair was preselected geometrically; controls were not geometry-matched |


#### Baseline and intervention integrity

The original PIE model was reconstructed from its
pretrained checkpoint.

Decoder.0 activations were reproduced against the
existing validation HDF5 without numerical differences
in the initial condition.

Identity interventions reproduced the original PIE
predictions exactly.

All feature interventions were applied during PIE
forward execution without updating model parameters.

#### Feature ablation

Decoder.0 SAE features 0, 189, and 2120 were evaluated
on all 32 saved validation conditions.

Feature 189 produced detectable changes in PIE outputs
across all conditions and three cell contexts.

Compared with five fixed random directions, feature 189
produced moderately larger mean prediction changes.

However, the advantage was not consistent across
individual conditions or learned SAE-direction controls.

Feature 2120 produced larger mean output changes in
some conditions, but its effects were strongly
concentrated and absent in other conditions.

These findings distinguish computational influence
from stability and biological interpretability. The
operator subtracts a decoded feature contribution from
the original PIE hidden state; it does not enforce an
exact zero of the feature upon subsequent SAE re-encoding.
Raw DE logits, LFC, and delta-p logits have different
scales, so absolute effects are interpreted within
each output head rather than compared across heads.

#### Activation patching

Feature 189 coefficients were transferred between
validation conditions using controlled interventions
on decoder.0 representations.

Patching changed PIE predictions but did not
consistently reduce their distance to donor predictions.

Nine-pair comparisons with alternative SAE directions,
alternative donors, and reversed interventions did not
support reliable donor-state transfer.

These negative results are retained as part of the
scientific findings.

#### Cross-layer response

Feature 189 was geometrically matched to decoder.1
SAE feature 413.

Across six validation conditions, upstream feature 189
ablation produced the largest activation response in
feature 413 among all 3,072 decoder.1 SAE features.

The response also exceeded the responses induced by
sixteen selected learned-direction controls.

Nevertheless, the target direction was much more
geometrically aligned with feature 413 than the
controls.

The experiments therefore establish a consistent
computational response across the six selected conditions
under the tested intervention, but not unique causal
mediation or independent biological replication.

#### Overall conclusion

The pretrained PIE system is sensitive to targeted
interventions along selected SAE decoder directions.

These interventions provide evidence about how
individual representation components affect PIE
predictions and downstream hidden states.

The strongest positive finding is the reproducible
cross-layer response of the geometrically matched
feature pair 189–413.

The activation-patching experiments did not support
consistent transfer of donor predictions through
feature 189 alone.

The results remain conditional on the pretrained PIE
model, fixed SAE dictionaries, selected conditions,
and specified intervention operators.

They do not establish biological causal mechanisms,
independent biological replication, or unique
mechanistic mediation. The bootstrap intervals are
conditional on the selected 32 conditions and five fixed
control directions, not uncertainty bounds over new
control directions, model checkpoints, or experimental
replicates. The cross-layer analysis lacks geometry-matched
interventions and formal downstream mediation tests.

In [32]:
import numpy as np
import pandas as pd

print("=== F.2. FINAL INTERVENTION CONSISTENCY AUDIT ===")

checks_f2 = {
    "Full feature-189 evaluation":
        len(B5_results) == 32 * 3,

    "Random-direction controls":
        len(B6_controls) == 32 * 5 * 3,

    "Condition-level bootstrap":
        len(B7_uncertainty) == 3,

    "Three-feature comparison":
        len(B8_results) == 32 * 3 * 3,

    "Learned SAE-direction controls":
        len(B9_controls) == 6 * 16 * 3,

    "Nine-pair activation patching":
        len(C3_results) == 9 * 6 * 3,

    "Donor-specificity controls":
        len(C4_results) == 9 * 4 * 3,

    "Cross-context downstream interventions":
        len(E3_results) == 6 * 17,

    "Geometric response audit":
        len(E4_results) == 17,

    "Expected cross-layer pair":
        TARGET_E1 == 189 and MATCHED_E1 == 413,

    "Mutual dictionary match":
        bool(IS_MUTUAL_E1),

    "Matched feature ranked first in six conditions":
        E3_comparison[
            "matched_rank_among_3072"
        ].eq(1).all(),

    "Target direction ranked first in six conditions":
        E3_comparison[
            "target_rank_among_17"
        ].eq(1).all(),

    "Geometry and response rank agree":
        int(target_e4["geometric_rank"]) == 1
        and int(target_e4["response_rank"]) == 1,

    "All response values finite":
        np.isfinite(
            E4_results[
                ["abs_cosine", "mean_response"]
            ].to_numpy()
        ).all(),
}

print("\n=== CONSISTENCY CHECKS ===")

for name, passed in checks_f2.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

failed_f2 = [
    name
    for name, passed in checks_f2.items()
    if not passed
]

print("\n=== GEOMETRY-RESPONSE CORRELATIONS ===")

print(
    f"Pearson, all 17 directions: "
    f"{pearson_all_e4:.6f}"
)
print(
    f"Spearman, all 17 directions: "
    f"{spearman_all_e4:.6f}"
)
print(
    f"Pearson, controls only: "
    f"{pearson_controls_e4:.6f}"
)
print(
    f"Spearman, controls only: "
    f"{spearman_controls_e4:.6f}"
)

print("\n=== FINAL STATUS ===")

print("Checks performed:", len(checks_f2))
print("Checks passed:", len(checks_f2) - len(failed_f2))
print("Checks failed:", len(failed_f2))

if failed_f2:
    raise AssertionError(
        "Consistency audit failed: "
        + ", ".join(failed_f2)
    )

print("\nInternal numerical consistency: PASS")
print("Independent methodological audit: NOT YET PERFORMED")
print("Unique causal mediation: NOT ESTABLISHED")
print("Biological causal validation: NOT PERFORMED")
print("No additional model inference performed.")

=== F.2. FINAL INTERVENTION CONSISTENCY AUDIT ===

=== CONSISTENCY CHECKS ===
Full feature-189 evaluation: PASS
Random-direction controls: PASS
Condition-level bootstrap: PASS
Three-feature comparison: PASS
Learned SAE-direction controls: PASS
Nine-pair activation patching: PASS
Donor-specificity controls: PASS
Cross-context downstream interventions: PASS
Geometric response audit: PASS
Expected cross-layer pair: PASS
Mutual dictionary match: PASS
Matched feature ranked first in six conditions: PASS
Target direction ranked first in six conditions: PASS
Geometry and response rank agree: PASS
All response values finite: PASS

=== GEOMETRY-RESPONSE CORRELATIONS ===
Pearson, all 17 directions: 0.879634
Spearman, all 17 directions: 0.517157
Pearson, controls only: 0.454645
Spearman, controls only: 0.420588

=== FINAL STATUS ===
Checks performed: 15
Checks passed: 15
Checks failed: 0

Internal numerical consistency: PASS
Independent methodological audit: NOT YET PERFORMED
Unique causal mediat